# 10 · Vincular exportadores con Orbis y FDI Markets

> Record linkage por **nombre + país** (y web, cuando existe) · motor `vinculador_empresas 1.0.0` sobre **rues-linker 0.22.4** · Google Colab

**Qué hace.** Toma una base **ancla** (la segmentación de exportadores) y la cruza con una o más bases **destino** (Orbis y FDI Markets). Para cada exportador dice si está en cada base, **a qué nivel** y **con qué confianza**, y trae los datos del registro destino (ingresos, empleados, proyectos, capex, web, matriz, contactos).

**Qué entrega.** Un Excel con una hoja principal, `EXPORTADORES` (**una fila por empresa, todas**, vinculadas o no), más `ENLACES` (cada vínculo con su porqué), `REVISION` (lo que decide un humano), `DECISIONES` (historial) y hojas de detalle, validación y control.

**Cómo decide** (todo lo que compara nombres viene de rues-linker: saneamiento, formas legales, catálogo de países, IDF y `SimilitudNombre`):

| nivel | regla | banda |
|---|---|---|
| **EMPRESA** | todas las palabras del nombre se corresponden (salvo forma legal, tildes, una errata o el truncamiento de aduana) y el país coincide | **A** si es idéntico · **B** si hubo errata, truncamiento, alias o nombre de una sola palabra |
| **GRUPO** | comparten la marca y lo que sobra es genérico (país, sector, *holding*); o el mismo nombre en otro país | **B** |
| **candidato** | comparte una palabra distintiva pero difiere en otras, posible sigla, matriz reportada o mismo sitio web | **C**: no se vincula sin decisión humana |

El **país es evidencia, no bloqueo**: el mismo nombre en otro país es GRUPO (filial o matriz), nunca EMPRESA. Un registro con el **mismo sitio web** que un vínculo EMPRESA se propone como vínculo (así aparece *Ritter Sport* para *Alfred Ritter GmbH & Co. KG*).

**Revisión.** Las decisiones humanas viven en archivos (CSV o el propio Excel de resultados) y se aplican en orden: la última manda. Cada vínculo dice su `ORIGEN` (automático, confirmado, agregado o descartado por revisión).

**Orden:** 1 → 8. **Solo se edita la celda 3.**

## 1 · Entorno
Monta Drive e instala rues-linker desde el tag publicado en GitHub. Si en el kernel había otra versión importada, pide reiniciar.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  1 · ENTORNO: Drive y rues-linker                                      ║
# ╚══════════════════════════════════════════════════════════════════════╝
import importlib
import subprocess
import sys

#  De dónde instalar rues-linker. Por defecto, el tag publicado en GitHub.
#  Alternativa: la ruta a una copia en Drive, p. ej. "/content/drive/MyDrive/ProColombia/rues-linker_v2-main".
ORIGEN_LIBRERIA = "git+https://github.com/EnriqueForero/rues-linker_v2@v0.22.4"  # @param {type:"string"}
VERSION_MINIMA = (0, 22, 4)

try:  # Colab: montar Drive
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except ImportError:
    print("ℹ️  fuera de Colab: no se monta Drive")


def _version_instalada():
    from importlib.metadata import PackageNotFoundError, version
    try:
        return tuple(int(x) for x in version("rues-linker").split(".")[:3])
    except (PackageNotFoundError, ValueError):
        return None


if (_version_instalada() or (0,)) < VERSION_MINIMA:
    if "record_linkage" in sys.modules:
        raise RuntimeError("Qué pasó: hay una versión vieja de record_linkage ya importada en este kernel.\n"
                           "Qué hacer: Entorno de ejecución → Reiniciar sesión, y vuelva a correr desde esta celda.")
    print(f"📦 instalando rues-linker desde {ORIGEN_LIBRERIA} …")
    destino = f"rues-linker @ {ORIGEN_LIBRERIA}" if ORIGEN_LIBRERIA.startswith("git+") else ORIGEN_LIBRERIA
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", destino], check=True)
    importlib.invalidate_caches()

import record_linkage  # noqa: E402

v = _version_instalada()
if v is None or v < VERSION_MINIMA:
    raise RuntimeError(f"rues-linker {v} < {VERSION_MINIMA}: la instalación no quedó bien.")
for m in ("matching.nombre_idf", "matching.normalizadores", "paises.catalogo", "processing.saneamiento"):
    importlib.import_module(f"record_linkage.{m}")
print(f"✅ rues-linker {record_linkage.__version__} · {record_linkage.__file__}")

## 2 · Motor de vinculación (no editar)
Escribe `vinculador_empresas.py` en la sesión y lo importa. Es el mismo archivo que acompaña la skill; si la librería lo incorpora más adelante, esta celda se reemplaza por un `import`.

In [ ]:
%%writefile vinculador_empresas.py
"""vinculador_empresas — record linkage de empresas por nombre (+ país y otras variables).

Motor de vinculación construido SOBRE rues-linker (>= 0.22.4). Toma una base
ANCLA (p. ej. exportadores) y la cruza con N bases DESTINO (p. ej. Orbis, FDI
Markets) y entrega, por cada empresa ancla, con qué registros destino es la
MISMA EMPRESA o el MISMO GRUPO empresarial, con qué confianza y por qué.

Qué viene de rues-linker (no se reimplementa):
  * saneamiento de texto (entidades HTML, controles), ASCII/mayúsculas,
    reunión de iniciales y retiro de formas legales (``normalizar_nombre``);
  * canonización de país contra el catálogo ISO 3166 (``canonizar_pais``);
  * IDF del corpus y comparador ``SimilitudNombre`` (Jaccard-IDF + Jaro-Winkler
    con las tres puertas de contención) → ``SCORE_RUES_LINKER``;
  * listas declaradas de genéricos y sufijos legales.

Qué agrega este motor (lo que la librería no trae para un CRUCE entre bases):
  1. Comparación EXHAUSTIVA ancla × destino (recall del bloqueo = 1 por
     construcción; las bases de este tipo son de miles, no de millones).
  2. Alineación palabra a palabra: EMPRESA exige que TODAS las palabras no
     legales se correspondan (se admite una errata por palabra); GRUPO exige una
     palabra distintiva compartida y que todo lo que sobra sea genérico.
  3. El país es EVIDENCIA, no bloqueo: mismo nombre en otro país = GRUPO
     (filial/matriz), nunca EMPRESA.
  4. Superficies alternas: alias entre paréntesis y matriz reportada
     (Parent company / GUO).
  5. Propagación por sitio web: un registro destino con el mismo dominio que
     un vínculo EMPRESA se propone como vínculo (Ritter Sport = Alfred Ritter).
  6. Decisiones humanas en un CSV/Excel que se aplican en la siguiente corrida
     (confirmar, cambiar nivel, agregar, rechazar) y quedan trazadas.
  7. Deduplicación interna de cada base, consolidación por empresa ancla,
     validación contra una referencia, invariantes y Excel de entrega.

Bandas: A = automático, alta confianza · B = automático, revisión rápida
recomendada · C = candidato, NO se vincula sin decisión humana.

Autor: Claude para Enrique Forero · 2026-09-29 · versión 1.0.0
"""

from __future__ import annotations

import hashlib
import json
import math
import platform
import re
import time
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz import process as rf_process
from rapidfuzz.distance import JaroWinkler, Levenshtein

import record_linkage
from record_linkage.matching.genericos import (
    GENERICOS_ESTRUCTURALES,
    GENERICOS_GEOGRAFIA,
    GENERICOS_SECTOR,
    SUFIJOS_INTERNACIONALES,
)
from record_linkage.matching.idf import construir_idf
from record_linkage.matching.nombre_idf import SimilitudNombre, neutralizar_genericos
from record_linkage.matching.normalizadores import LOCALES, normalizar_nombre
from record_linkage.paises import CATALOGO_PAISES, canonizar_pais, sugerir_alias_pais
from record_linkage.processing.saneamiento import (
    PRELIMPIEZA_COMERCIO_EXTERIOR,
    ascii_mayusculas,
    sanear_texto,
    unir_iniciales,
)

VERSION_MOTOR = "1.0.0"

__all__ = [
    "Fuente",
    "ConfigVinculacion",
    "Resultado",
    "ejecutar",
    "exportar_excel",
    "leer_decisiones",
    "plantilla_decisiones",
]


#: Nombres de pila frecuentes (ES/IT/DE/FR/EN): compartir SOLO uno de ellos no es evidencia de
#: identidad ('ALFRED RITTER' ≠ 'Alfred C Toepfer'; 'PAUL REBER' ≠ 'Paul'). No afecta a EMPRESA,
#: que exige que TODAS las palabras se correspondan.
NOMBRES_PILA_BASE = frozenset("""
JOSE JOSEP JOSEF JOSEPH GIUSEPPE ANTONIO ANTONI ANTOINE ANTON JUAN JEAN JOHN JOHANN JOHANNES GIOVANNI PAUL PABLO
PAOLO PAULO PEDRO PIETRO PIERRE PETER MANUEL LUIS LUIGI LOUIS LUDWIG LUCA LUCAS CARLOS CARLO CARL KARL CHARLES
FRANCISCO FRANCESCO FRANZ FRANCOIS FRANK MIGUEL MICHEL MICHAEL MICHELE ANDRES ANDREA ANDREAS ANDRE AUGUST AUGUSTO
ALFRED ALFREDO DAVID DAVIDE HENRI HENRY HEINRICH HEINZ ENRIQUE ENRICO HANS GUIDO MARIO MARIA MARIE ANNA ANA ROSA
CRISTINA CHRISTINA KRISTINA ELLA JULIE MONICA ROBERT ROBERTO RICHARD RICARDO WILHELM WILLIAM GUILLERMO FRIEDRICH
FEDERICO FERNANDO RAFAEL RAFFAELE ALFONSO ALESSANDRO ALEXANDER ALEJANDRO GEORG GEORGES GEORGE JORGE GIORGIO THOMAS
TOMAS MARTIN MARTINO OTTO ERICH ERIK EMIL EMILIO FELIX JAKOB JACOB JACQUES IGNACIO VICENTE VINCENZO SALVATORE
DOMENICO GABRIEL GABRIELE NICOLAS NICOLA SEBASTIAN SIMON VICTOR VITTORIO HERMANN OSCAR GUSTAV GUSTAVO PHILIPP
PHILIPPE FELIPE ALBERT ALBERTO ALBERTINE BERNARD BERNARDO ERNST WALTER WERNER KURT HELMUT JUERGEN JURGEN RAUL
""".split())

# ════════════════════════════════════════════════════════════════════════════
# 1 · CONTRATOS DE ENTRADA
# ════════════════════════════════════════════════════════════════════════════
@dataclass
class Fuente:
    """Una base de entrada y el papel de cada columna.

    Solo ``col_nombre`` es obligatoria. Cada variable adicional que se declare
    en la ancla Y en un destino se usa como evidencia (ver ``evidencia_extra``).

    Args:
        clave: nombre corto y único (``EXPORTADORES``, ``ORBIS``, ``FDI``).
        archivo: ruta del Excel/CSV/Parquet.
        hoja: hoja del Excel (nombre o posición).
        col_nombre: razón social.
        col_pais: país del registro.
        cols_pais_extra: otras columnas de país que también cuentan como
            "mismo país" (FDI: ``HQ country`` además de ``País de consulta``).
        col_id: identificador nativo único (BvD ID). Si falta, se usa la fila.
        prefijo_id: prefijo del ID por fila cuando no hay ``col_id``.
        col_grupo: nombre de la matriz / grupo (Parent company, GUO - Name).
        col_web, col_email, col_telefono, col_ciudad, col_direccion: contacto.
        col_id_comparable: identificador comparable ENTRE bases (NIF/VAT/NIT).
        cols_metricas: numéricas que se suman por empresa (solo ancla).
        cols_contexto: columnas que viajan a las salidas, en ese orden.
        nombres_contexto: renombre legible de columnas de contexto/métricas.
        largo_truncamiento: ancho del campo de nombre en el sistema de origen
            (aduana: 35). El último fragmento puede estar cortado.
        detectar_intermediarios: marcar agentes de carga (solo aduana).
        transformaciones: (columna, "hs6") para formatear códigos.
        etiqueta: nombre para mostrar (``FDI MARKETS``).
    """

    clave: str
    archivo: str
    col_nombre: str
    hoja: str | int = 0
    col_pais: str | None = None
    cols_pais_extra: tuple[str, ...] = ()
    col_id: str | None = None
    prefijo_id: str = ""
    col_grupo: str | None = None
    col_web: str | None = None
    col_email: str | None = None
    col_telefono: str | None = None
    col_ciudad: str | None = None
    col_direccion: str | None = None
    col_id_comparable: str | None = None
    #: Columna numérica de tamaño (ingresos, proyectos) para desempatar el vínculo principal:
    #: entre dos sociedades del mismo grupo igual de parecidas, se muestra la más grande.
    col_tamano: str | None = None
    cols_metricas: tuple[str, ...] = ()
    cols_contexto: tuple[str, ...] = ()
    nombres_contexto: dict = field(default_factory=dict)
    largo_truncamiento: int | None = None
    detectar_intermediarios: bool = False
    transformaciones: tuple[tuple[str, str], ...] = ()
    #: Métricas que el sistema de origen REPITE en varias filas (valor de un nivel de
    #: agregación superior). Se informa también su suma sin repetidos por
    #: (empresa, *claves_repeticion, valor). Medido en el piloto: Kilos se repite
    #: idéntico en filas de la misma empresa y partida con operaciones distintas.
    metricas_repetidas: tuple[str, ...] = ()
    claves_repeticion: tuple[str, ...] = ()
    etiqueta: str = ""

    def __post_init__(self) -> None:
        if not re.fullmatch(r"[A-Z][A-Z0-9_]{1,30}", self.clave):
            raise ValueError(f"clave={self.clave!r}: use MAYÚSCULAS, dígitos o '_' (p. ej. 'ORBIS').")
        self.prefijo_id = self.prefijo_id or self.clave[:3]
        self.etiqueta = self.etiqueta or self.clave
        self.cols_pais_extra = tuple(self.cols_pais_extra)
        self.cols_metricas = tuple(self.cols_metricas)
        self.cols_contexto = tuple(self.cols_contexto)

    def columnas_declaradas(self) -> list[str]:
        cols = [self.col_nombre, self.col_pais, *self.cols_pais_extra, self.col_id, self.col_grupo,
                self.col_web, self.col_email, self.col_telefono, self.col_ciudad, self.col_direccion,
                self.col_id_comparable, self.col_tamano, *self.cols_metricas, *self.cols_contexto]
        vistos, out = set(), []
        for c in cols:
            if c and c not in vistos:
                vistos.add(c)
                out.append(c)
        return out


@dataclass
class ConfigVinculacion:
    """Parámetros del cruce. Los defectos están medidos en el piloto agroalimentos 2026-09.

    Args:
        ancla: la base cuyas empresas se quieren enriquecer (una fila por empresa en la salida).
        destinos: bases contra las que se cruza (0..N). Sin destinos = solo deduplicación.
        alias_paises: ISO3 → grafías que el catálogo de la librería no trae (FRANCE, ITALY…).
        formas_legales_extra: formas legales a retirar además de las de la librería.
        genericos_extra: palabras de sector/geografía/forma de grupo que NUNCA identifican.
        patrones_intermediario: regex de agentes de carga (en la ancla).
        dominios_genericos: dominios web/correo que no identifican a nadie.
        umbral_piso_empresa: SCORE_RUES_LINKER mínimo para EMPRESA con palabras alineadas.
        umbral_dedup: SCORE_RUES_LINKER mínimo para unir dos grafías de la ancla sin revisión.
        umbral_parecido: SCORE mínimo para proponer a revisión un par sin palabra compartida.
        max_ediciones_palabra: errata admitida por palabra (Levenshtein).
        archivos_decisiones: CSV/XLSX con decisiones de revisión (se aplican en orden).
        archivo_referencia: decisiones de referencia SOLO para medir al motor (no se aplican).
        propagar_por_web: activar la propagación por dominio web.
        carpeta_salida, prefijo_salida, versionar_salida: dónde y cómo se guarda el Excel.
    """

    ancla: Fuente
    destinos: list[Fuente] = field(default_factory=list)
    alias_paises: dict = field(default_factory=dict)
    formas_legales_extra: frozenset = frozenset()
    genericos_extra: frozenset = frozenset()
    patrones_intermediario: tuple[str, ...] = ()
    dominios_genericos: frozenset = frozenset()
    nombres_pila: frozenset = NOMBRES_PILA_BASE
    umbral_piso_empresa: float = 0.70
    umbral_dedup: float = 0.84
    umbral_parecido: float = 0.88
    max_ediciones_palabra: int = 1
    archivos_decisiones: tuple[str, ...] = ()
    #: Decisiones cuyo AUTOR contiene esta marca son de revisión ASISTIDA (no validada por una persona):
    #: se aplican, pero el vínculo queda con confianza MEDIA y en REVISION como "validar".
    marca_sin_validar: str = "(verificar)"
    archivo_referencia: str | tuple | list | None = None
    propagar_por_web: bool = True
    carpeta_salida: str = "."
    prefijo_salida: str = "RESULTADO_VINCULACION"
    versionar_salida: bool = True
    titulo: str = "Vinculación de empresas"

    def __post_init__(self) -> None:
        claves = [self.ancla.clave, *[d.clave for d in self.destinos]]
        if len(set(claves)) != len(claves):
            raise ValueError(f"Qué pasó: claves de fuente repetidas {claves}.\nQué hacer: use una clave única por base.")
        for nombre, v in (("umbral_piso_empresa", self.umbral_piso_empresa), ("umbral_dedup", self.umbral_dedup),
                          ("umbral_parecido", self.umbral_parecido)):
            if not 0.5 <= v < 1.0:
                raise ValueError(f"{nombre}={v}: debe estar en [0,5; 1,0).")
        if self.max_ediciones_palabra not in (0, 1, 2):
            raise ValueError("max_ediciones_palabra debe ser 0, 1 o 2.")
        self.archivos_decisiones = tuple(a for a in self.archivos_decisiones if a)
        self.formas_legales_extra = frozenset(str(x).strip().upper() for x in self.formas_legales_extra)
        self.genericos_extra = frozenset(str(x).strip().upper() for x in self.genericos_extra)
        self.dominios_genericos = frozenset(str(x).strip().lower() for x in self.dominios_genericos)
        self.nombres_pila = frozenset(str(x).strip().upper() for x in self.nombres_pila)


# ════════════════════════════════════════════════════════════════════════════
# 2 · LECTURA
# ════════════════════════════════════════════════════════════════════════════
def sha256_archivo(ruta: str | Path) -> str:
    h = hashlib.sha256()
    with open(ruta, "rb") as fh:
        for bloque in iter(lambda: fh.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def _leer_tabla(ruta: str, hoja: str | int) -> pd.DataFrame:
    p = Path(ruta)
    if not p.exists():
        raise FileNotFoundError(f"Qué pasó: no existe {p}.\nQué hacer: corrija la ruta en la configuración.")
    suf = p.suffix.lower()
    if suf in (".xlsx", ".xlsm", ".xls"):
        return pd.read_excel(p, sheet_name=hoja, dtype=object)
    if suf == ".parquet":
        return pd.read_parquet(p)
    if suf in (".csv", ".txt", ".tsv"):
        texto = p.read_bytes()[:4096].decode("utf-8", errors="ignore")
        sep = "\t" if texto.count("\t") > texto.count(";") and texto.count("\t") > texto.count(",") else (
            ";" if texto.count(";") > texto.count(",") else ",")
        try:
            return pd.read_csv(p, sep=sep, dtype=object, encoding="utf-8-sig")
        except UnicodeDecodeError:
            return pd.read_csv(p, sep=sep, dtype=object, encoding="cp1252")
    raise ValueError(f"Formato no soportado: {p.suffix}")


def _hs6(x) -> str:
    """1905.9 → '190590'; 901.11 → '090111' (Excel guarda la partida como número)."""
    if x is None or (isinstance(x, float) and math.isnan(x)) or str(x).strip() == "":
        return ""
    s = str(x).strip()
    if re.fullmatch(r"\d{6}", s):
        return s
    try:
        v = float(s.replace(",", "."))
    except ValueError:
        return s
    if v >= 10000:  # ya viene como entero de 5-6 dígitos
        return f"{int(round(v)):06d}"
    return f"{v:.2f}".replace(".", "").zfill(6)


_TRANSFORMACIONES = {"hs6": _hs6}


def leer_fuente(f: Fuente) -> pd.DataFrame:
    """Lee la hoja, verifica el contrato y asigna FILA_EXCEL e ID_REGISTRO."""
    df = _leer_tabla(f.archivo, f.hoja).copy()
    df.columns = [str(c).strip() for c in df.columns]
    faltan = [c for c in f.columnas_declaradas() if c not in df.columns]
    if faltan:
        raise KeyError(
            f"Qué pasó: a la fuente {f.clave} (hoja {f.hoja!r}) le faltan las columnas {faltan}.\n"
            f"Por qué importa: son las que declara su Fuente.\n"
            f"Qué hacer: corrija la configuración. Columnas disponibles: {list(df.columns)}"
        )
    df = df.reset_index(drop=True).copy()
    df.insert(0, "FILA_EXCEL", np.arange(2, len(df) + 2))
    vacias = df[f.columnas_declaradas()].isna().all(axis=1)
    df = df[~vacias].reset_index(drop=True)
    for col, nombre in f.transformaciones:
        if nombre not in _TRANSFORMACIONES:
            raise KeyError(f"Transformación desconocida {nombre!r}. Disponibles: {sorted(_TRANSFORMACIONES)}")
        df[col] = df[col].map(_TRANSFORMACIONES[nombre])
    if f.col_id:
        ids = df[f.col_id].astype("string").fillna("").str.strip()
        if (ids == "").any():
            filas = df.loc[ids == "", "FILA_EXCEL"].tolist()[:15]
            raise ValueError(f"Qué pasó: {int((ids == '').sum())} filas de {f.clave} sin '{f.col_id}' (filas {filas}).\n"
                             "Qué hacer: suelen ser filas de continuación de campos multivalor; exporte una fila por empresa.")
        if ids.duplicated().any():
            raise ValueError(f"Qué pasó: '{f.col_id}' no es único en {f.clave} (p. ej. {ids[ids.duplicated()].head(3).tolist()}).\n"
                             "Qué hacer: declare col_id=None para usar la fila como identificador.")
        df.insert(1, "ID_REGISTRO", ids.to_numpy())
    else:
        df.insert(1, "ID_REGISTRO", [f"{f.prefijo_id}-F{n:04d}" for n in df["FILA_EXCEL"]])  # F = fila de Excel
    df.insert(2, "FUENTE", f.clave)
    return df


# ════════════════════════════════════════════════════════════════════════════
# 3 · NORMALIZACIÓN (rues-linker)
# ════════════════════════════════════════════════════════════════════════════
#: Palabras de enlace: no identifican ni cuentan como diferencia.
CONECTORES = frozenset({
    "Y", "E", "UND", "AND", "ET", "DE", "DEL", "DI", "DU", "VON", "V", "LA", "EL", "LOS", "LAS", "THE", "OF",
    "DA", "DO", "DER", "DEN", "VAN", "LE", "LES", "DES", "AL", "A", "EN", "CON", "FOR", "PER", "ZU", "ZUR",
    "IM", "AM", "GES", "D", "L", "I",
})

#: Formas legales como PALABRAS sueltas (además de las frases que retira la librería).
_LEGALES_BASE = frozenset({
    "GMBH", "KG", "SA", "SL", "SLU", "SAU", "SRL", "SRLS", "SPA", "SAS", "SASU", "SARL", "BV", "NV", "AG", "SE",
    "LTD", "LIMITED", "INC", "LLC", "CO", "SOCIEDAD", "LIMITADA", "ANONIMA", "UNIPERSONAL", "SOCIETA",
    "RESPONSABILITA", "LIMITATA", "AZIONI", "SOCIETE", "ANONYME", "GESELLSCHAFT", "HAFTUNG", "BESCHRANKTER",
    "BESCHRAENKTER", "AKTIENGESELLSCHAFT", "STIFTUNG", "KGAA", "MBH", "LTDA", "STI", "COOP", "COOPERATIVA",
    "COOPERATIVE", "SCOOP", "OHG", "UG", "EG", "EV", "EURL", "SNC", "SCA", "SCARL", "SOCIO", "UNICO",
    "SIMPLIFIEE", "ACTIONS", "PLC", "CORP", "CIA", "SC", "SPZOO", "ZOO", "SP", "AS", "AB", "OY", "APS",
    "SUCURSAL", "BRANCH", "FILIALE", "CV", "RL", "DE CV",
})


def _registrable(dominio: str) -> str:
    """'shop.lidl.co.uk' → 'lidl.co.uk'; 'www.gullon.es' → 'gullon.es'."""
    partes = [p for p in dominio.split(".") if p]
    if len(partes) <= 2:
        return ".".join(partes)
    if partes[-2] in {"co", "com", "org", "net", "gov", "ac", "edu"} and len(partes[-1]) == 2:
        return ".".join(partes[-3:])
    return ".".join(partes[-2:])


def _num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(pd.Series(s, dtype="object").replace({"n.a.": None, "n.s.": None, "-": None}), errors="coerce")


def dominio_web(x) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    s = str(x).strip().lower()
    if not s or s in {"nan", "none", "n.a.", "-"}:
        return ""
    s = re.split(r"[\s;,|]+", s)[0]
    s = re.sub(r"^[a-z]+://", "", s)
    s = s.split("/")[0].split("?")[0].split(":")[0]
    s = re.sub(r"^www\d?\.", "", s)
    return _registrable(s) if "." in s else ""


def dominio_email(x) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    m = re.search(r"@([A-Za-z0-9.\-]+\.[A-Za-z]{2,})", str(x))
    return _registrable(m.group(1).lower()) if m else ""


def digitos_telefono(x) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    d = re.sub(r"\D", "", str(x))
    return d[-8:] if len(d) >= 8 else ""


def id_comparable(x) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    return re.sub(r"[^A-Z0-9]", "", str(x).upper())


class Normalizador:
    """Normalización de nombres y países con la librería, configurada una sola vez."""

    LOCALE = "VINCULADOR_EMPRESAS"

    def __init__(self, cfg: ConfigVinculacion) -> None:
        self.cfg = cfg
        sufijos: set[str] = set()
        for base in ("ES", "EN"):
            sufijos |= set(LOCALES[base]["sufijos"])
        sufijos |= set(SUFIJOS_INTERNACIONALES) | set(cfg.formas_legales_extra)
        sufijos |= {t for t in _LEGALES_BASE if " " not in t and len(t) >= 2 and t not in {"CO", "SE", "AS", "AB", "SC", "SP", "RL"}}
        self.genericos = frozenset(set(GENERICOS_ESTRUCTURALES) | set(GENERICOS_SECTOR) | set(GENERICOS_GEOGRAFIA)
                                   | set(cfg.genericos_extra))
        LOCALES[self.LOCALE] = {"sufijos": frozenset(sufijos), "genericos": self.genericos}
        # Palabras legales: las de una sola palabra de la librería + las declaradas. De las FRASES
        # de la librería NO se toman sus palabras sueltas: "COMERCIALIZADORA INTERNACIONAL" es una
        # forma legal colombiana, pero "INTERNACIONAL" suelta es parte del nombre (medido: sin esta
        # regla EUROPASTRY INTERNACIONAL quedaba igual a EUROPASTRY S.A.).
        legales = set(_LEGALES_BASE) | {t for t in sufijos if " " not in t}
        for s in cfg.formas_legales_extra:
            legales |= set(re.sub(r"[^A-Z0-9 ]", " ", s.upper()).split())
        self.legales = frozenset(re.sub(r"[^A-Z0-9]", "", t) for t in legales if len(re.sub(r"[^A-Z0-9]", "", t)) >= 2)
        self.catalogo = self._catalogo(cfg.alias_paises)
        self.patron_inter = re.compile("|".join(cfg.patrones_intermediario), re.I) if cfg.patrones_intermediario else None

    @staticmethod
    def _catalogo(alias: dict) -> tuple:
        iso_validos = {iso for iso, _, _ in CATALOGO_PAISES}
        malos = sorted(set(alias) - iso_validos)
        if malos:
            raise KeyError(f"ALIAS_PAISES usa ISO3 inexistentes en el catálogo: {malos}")
        return tuple((iso, nombre, tuple(graf) + tuple(str(a).strip().upper() for a in alias.get(iso, ())))
                     for iso, nombre, graf in CATALOGO_PAISES)

    # ── nombres ──────────────────────────────────────────────────────────────
    _PAREN = re.compile(r"\(([^()]*)\)")
    _PAREN_ABIERTO = re.compile(r"\([^()]*$")
    _APOSTROFE_S = re.compile(r"(\w)\s?['`´’]\s?(S)\b", re.I)
    _APOSTROFE = re.compile(r"['`´’]")

    def separar_alias(self, nombre) -> tuple[str, str]:
        """'Vicky Foods (Dulcesol)' → ('Vicky Foods', 'Dulcesol'). Un paréntesis sin cerrar se descarta."""
        s = "" if nombre is None or (isinstance(nombre, float) and math.isnan(nombre)) else str(nombre)
        s = s.replace("&AMP;", "&").replace("&amp;", "&")
        if "(" not in s:
            return s, ""
        alias = " ".join(m.strip() for m in self._PAREN.findall(s) if m.strip())
        principal = self._PAREN_ABIERTO.sub(" ", self._PAREN.sub(" ", s))
        return re.sub(r"\s+", " ", principal).strip(), alias

    def normalizar(self, serie: pd.Series) -> pd.Series:
        """Cadena de la librería: sanear → ASCII/mayúsculas → iniciales → retirar formas legales."""
        s = pd.Series(serie, dtype="object").fillna("").astype(str)
        s = s.str.replace(self._APOSTROFE_S, r"\1\2", regex=True).str.replace(self._APOSTROFE, " ", regex=True)
        limpio = unir_iniciales(ascii_mayusculas(sanear_texto(s.astype("string"), prelimpieza=PRELIMPIEZA_COMERCIO_EXTERIOR)))
        out = normalizar_nombre(limpio, locale=self.LOCALE, quitar_sufijos=True, quitar_genericos=False)
        return out.astype(object).fillna("").astype(str).str.replace(r"[()]", " ", regex=True).str.replace(
            r"\s+", " ", regex=True).str.strip()

    def utiles(self, nombre: str) -> list[str]:
        return [t for t in nombre.split() if len(t) > 1 and t not in self.legales and t not in CONECTORES
                and not t.isdigit()]

    def distintivas(self, nombre: str) -> list[str]:
        return [t for t in self.utiles(nombre) if t not in self.genericos and len(t) >= 3]

    def marcas(self, nombre: str, truncado: bool = False) -> tuple[str, ...]:
        """Iniciales sueltas y números: 'J MARTINEZ' ≠ 'A MARTINEZ'; 'BIMBO 2' ≠ 'BIMBO 3'."""
        t = [x for x in nombre.split() if x not in self.legales and x not in CONECTORES]
        if truncado and t and len(t[-1]) == 1:
            t = t[:-1]
        return tuple(sorted(x for x in t if (len(x) == 1 and x.isalpha()) or x.isdigit()))

    def compacto(self, nombre: str) -> str:
        return "".join(self.utiles(nombre))

    # ── países ───────────────────────────────────────────────────────────────
    def paises(self, serie: pd.Series, fuente: str, columna: str) -> pd.DataFrame:
        """ISO3 + nombre canónico. Detiene la corrida si hay grafías fuera del catálogo."""
        s = pd.Series(serie, dtype="object").fillna("").astype(str).str.strip()
        vacio = s == ""
        res = canonizar_pais(s.where(~vacio, "SIN PAIS").astype("string"), catalogo=self.catalogo)
        tabla = res.tabla[["PAIS_ISO3", "PAIS_FINAL"]].copy()
        sin = res.tabla["PAIS_METODO"].eq("sin_clasificar") & ~vacio.to_numpy()
        if sin.any():
            sug = sugerir_alias_pais(res.sin_clasificar, catalogo=self.catalogo, maximo=50)
            ejemplos = ", ".join(f"'ISO3': ({str(v).upper()!r},)" for v in sug["valor"].head(5))
            raise ValueError(
                f"Qué pasó: {fuente}.{columna} trae grafías de país que el catálogo no reconoce:\n"
                f"{sug.to_string(index=False)}\n"
                f"Por qué importa: el país es evidencia del cruce; sin catalogar, 'France' y 'Francia' no coinciden.\n"
                f"Qué hacer: agréguelas en ALIAS_PAISES con su ISO3, p. ej. {ejemplos}. La sugerencia es una ayuda, no una regla."
            )
        tabla.loc[vacio.to_numpy(), ["PAIS_ISO3", "PAIS_FINAL"]] = ["", ""]
        return tabla.reset_index(drop=True)


def preparar_fuente(df: pd.DataFrame, f: Fuente, nz: Normalizador) -> pd.DataFrame:
    """Columnas estándar del motor: nombre normalizado, alias, grupo, país(es), contacto, banderas."""
    out = pd.DataFrame({"FUENTE": f.clave, "ID_REGISTRO": df["ID_REGISTRO"].to_numpy(),
                        "FILA_EXCEL": df["FILA_EXCEL"].to_numpy()})
    crudo = df[f.col_nombre].astype(object).where(df[f.col_nombre].notna(), "").astype(str).str.strip()
    out["NOMBRE_ORIGINAL"] = crudo.to_numpy()
    partes = crudo.map(nz.separar_alias)
    out["NOMBRE_NORM"] = nz.normalizar(partes.str[0]).to_numpy()
    alias = nz.normalizar(partes.str[1]).to_numpy()
    util = np.array([bool(nz.distintivas(a)) for a in alias])
    out["ALIAS_NORM"] = np.where(util & (alias != out["NOMBRE_NORM"].to_numpy()), alias, "")
    # Sin nombre principal utilizable pero con alias: el alias pasa a principal.
    vacio = (out["NOMBRE_NORM"] == "") & (out["ALIAS_NORM"] != "")
    out.loc[vacio, "NOMBRE_NORM"] = out.loc[vacio, "ALIAS_NORM"]
    out.loc[vacio, "ALIAS_NORM"] = ""
    if f.col_grupo:
        g = df[f.col_grupo].astype(object).where(df[f.col_grupo].notna(), "").astype(str).str.strip()
        out["GRUPO_ORIGINAL"] = g.to_numpy()
        gp = g.map(nz.separar_alias)
        gn = nz.normalizar(gp.str[0]).to_numpy()
        out["GRUPO_NORM"] = np.where(gn == out["NOMBRE_NORM"].to_numpy(), "", gn)
    else:
        out["GRUPO_ORIGINAL"], out["GRUPO_NORM"] = "", ""
    # País principal + adicionales (cualquiera cuenta como "mismo país")
    if f.col_pais:
        p = nz.paises(df[f.col_pais], f.clave, f.col_pais)
        out["PAIS_ORIGINAL"] = df[f.col_pais].astype(object).where(df[f.col_pais].notna(), "").astype(str).to_numpy()
        out["PAIS_ISO3"], out["PAIS_CANONICO"] = p["PAIS_ISO3"].to_numpy(), p["PAIS_FINAL"].to_numpy()
    else:
        out["PAIS_ORIGINAL"], out["PAIS_ISO3"], out["PAIS_CANONICO"] = "", "", ""
    isos = [set([x]) - {""} for x in out["PAIS_ISO3"]]
    for c in f.cols_pais_extra:
        p = nz.paises(df[c], f.clave, c)
        for k, x in enumerate(p["PAIS_ISO3"]):
            if x:
                isos[k].add(x)
    out["PAISES_EVIDENCIA"] = ["|".join(sorted(s)) for s in isos]
    # Contacto y variables adicionales
    def col(c):
        return df[c].to_numpy() if c else np.full(len(df), None, dtype=object)
    out["WEB"] = pd.Series(col(f.col_web), dtype=object).fillna("").astype(str).to_numpy()
    out["DOMINIO_WEB"] = [dominio_web(x) for x in col(f.col_web)]
    out["EMAIL"] = pd.Series(col(f.col_email), dtype=object).fillna("").astype(str).to_numpy()
    out["DOMINIO_EMAIL"] = [dominio_email(x) for x in col(f.col_email)]
    out["TELEFONO"] = pd.Series(col(f.col_telefono), dtype=object).fillna("").astype(str).str.lstrip("'").str.strip().to_numpy()
    out["TEL_DIGITOS"] = [digitos_telefono(x) for x in col(f.col_telefono)]
    out["CIUDAD"] = pd.Series(col(f.col_ciudad), dtype=object).fillna("").astype(str).str.strip().to_numpy()
    out["CIUDAD_NORM"] = ascii_mayusculas(pd.Series(out["CIUDAD"], dtype="string")).astype(object).to_numpy()
    out["DIRECCION"] = pd.Series(col(f.col_direccion), dtype=object).fillna("").astype(str).to_numpy()
    out["ID_COMPARABLE"] = [id_comparable(x) for x in col(f.col_id_comparable)]
    out["TAMANO"] = _num(pd.Series(col(f.col_tamano), dtype=object)).fillna(-1).to_numpy() if f.col_tamano else -1.0
    gen = nz.cfg.dominios_genericos
    for c in ("DOMINIO_WEB", "DOMINIO_EMAIL"):
        out[c] = np.where(pd.Series(out[c]).isin(gen), "", out[c])
    out["TRUNCADO"] = (crudo.str.len() >= f.largo_truncamiento).to_numpy() if f.largo_truncamiento else False
    out["INTERMEDIARIO"] = (crudo.str.contains(nz.patron_inter, na=False).to_numpy()
                            if (f.detectar_intermediarios and nz.patron_inter is not None) else False)
    out["SIN_NOMBRE_UTIL"] = [not nz.utiles(n) for n in out["NOMBRE_NORM"]]
    return out


# ════════════════════════════════════════════════════════════════════════════
# 4 · COMPARACIÓN DE NOMBRES
# ════════════════════════════════════════════════════════════════════════════
def misma_palabra(t: str, u: str, max_ed: int) -> bool:
    """Errata admitida: ≤ max_ed ediciones; una SUSTITUCIÓN solo en palabras de ≥ 8 letras.

    Medido en el piloto: Jaro-Winkler ≥ 0,92 unía MUSETTI con MUTTI; una sustitución
    corta unía ARBAL con ARGAL. Inserción/supresión sí: SCHAER/SCHAR, AUDEN/AUDENS.
    """
    if t == u:
        return True
    if max_ed <= 0 or min(len(t), len(u)) < 4 or Levenshtein.distance(t, u) > max_ed:
        return False
    return len(t) != len(u) or min(len(t), len(u)) >= 8


def alinear(ta: list[str], tb: list[str], max_ed: int, trunc_a: bool = False,
            trunc_b: bool = False) -> tuple[list[str], list[str], list[tuple[str, str, str]]]:
    """Empareja palabra a palabra dos listas de palabras útiles.

    Returns:
        (sobra_a, sobra_b, pares) con pares = [(palabra_a, palabra_b, tipo)] y tipo en
        {"exacta", "errata", "truncada"}. Un nombre truncado admite que su ÚLTIMA palabra
        sea prefijo de una palabra del otro, y que al otro le sobren palabras DESPUÉS
        de la última emparejada (se perdieron en el corte).
    """
    libres = list(tb)
    sobra_a: list[str] = []
    pares: list[tuple[str, str, str]] = []
    for k, t in enumerate(ta):
        hit, tipo = (t, "exacta") if t in libres else (None, "")
        if hit is None and len(t) >= 4 and max_ed > 0:
            cand = sorted((Levenshtein.distance(t, u), u) for u in libres if misma_palabra(t, u, max_ed))
            if cand:
                hit, tipo = cand[0][1], "errata"
        if hit is None and trunc_a and k == len(ta) - 1 and len(t) >= 2:
            cand = [u for u in libres if u.startswith(t) and u != t]
            if cand:
                hit, tipo = cand[0], "truncada"
        if hit is None and trunc_b and len(t) >= 3:
            cand = [u for u in libres if t.startswith(u) and u == tb[-1] and len(u) >= 2]
            if cand:
                hit, tipo = cand[0], "truncada"
        if hit is None:
            sobra_a.append(t)
        else:
            libres.remove(hit)
            pares.append((t, hit, tipo))
    sobra_b = libres
    # Pérdida por truncamiento: si la ÚLTIMA palabra del nombre truncado quedó emparejada, lo
    # que al otro le sobra DESPUÉS de esa posición se perdió en el corte (no es diferencia).
    if trunc_a and sobra_b and ta and pares and pares[-1][0] == ta[-1]:
        corte = tb.index(pares[-1][1])
        sobra_b = [u for u in sobra_b if tb.index(u) < corte]
    if trunc_b and sobra_a and tb and pares:
        ult_b = [p for p in pares if p[1] == tb[-1]]
        if ult_b:
            corte = ta.index(ult_b[0][0])
            sobra_a = [u for u in sobra_a if ta.index(u) < corte]
    return sobra_a, sobra_b, pares


def es_subsecuencia(corta: str, larga: str) -> bool:
    """'CTRA' ⊂ 'CUETARA', 'SERDIS' ⊂ 'SEURODIS': abreviaturas de aduana que quitan vocales."""
    if len(corta) < 3 or len(corta) >= len(larga) or corta[0] != larga[0]:
        return False
    it = iter(larga)
    return all(c in it for c in corta)


def es_abreviatura(a: list[str], b: list[str], genericos: frozenset, subsecuencia: bool = False) -> bool:
    """¿Las palabras de ``a`` son una abreviatura de ``b``? Solo alimenta la revisión.

    Cada palabra de ``a``, en orden, es prefijo de una palabra de ``b``, o la sigla de
    varias palabras consecutivas de ``b``, o una subsecuencia de consonantes; ancla en la
    primera palabra de ``b`` y al menos una abreviatura cubre una palabra distintiva.
    'GR EMP P ALIMENTACION' → GRUPO EMPRESARIAL PALACIOS ALIMENTACION; 'GEPSAU' → G E P (SAU).
    """
    if not a or not b or a[0][0] != b[0][0]:
        return False

    def cubre(i: int, j: int, abrevio: bool, distintiva: bool) -> bool:
        if i == len(a):
            return abrevio and distintiva
        if j >= len(b):
            return False
        t = a[i]
        if b[j].startswith(t) and cubre(i + 1, j + 1, abrevio or len(t) < len(b[j]),
                                        distintiva or (len(t) < len(b[j]) and b[j] not in genericos)):
            return True
        if subsecuencia and es_subsecuencia(t, b[j]) and cubre(i + 1, j + 1, True, distintiva or b[j] not in genericos):
            return True
        n = len(t)
        if 2 <= n <= len(b) - j and all(b[j + k][0] == t[k] for k in range(n)):
            if cubre(i + 1, j + n, True, distintiva or any(x not in genericos for x in b[j:j + n])):
                return True
        return i > 0 and cubre(i, j + 1, abrevio, distintiva)

    return (len(a) >= 2 or len(a[0]) >= 3) and cubre(0, 0, False, False)


def fichas_abreviatura(nombre: str, nz: "Normalizador") -> list[list[str]]:
    """Variantes de un nombre para buscar siglas: sin formas legales ni conectores, sin letra final
    suelta (fragmento de S.A./S.L. truncado) y, si la última palabra trae la forma legal pegada
    ('GEPSAU' = G·E·P + SAU), también sin ella."""
    t = [x for x in nombre.split() if x not in nz.legales and x not in CONECTORES]
    while len(t) > 1 and len(t[-1]) == 1:
        t = t[:-1]
    if not t:
        return []
    variantes = [t]
    ult = t[-1]
    for leg in sorted(nz.legales, key=lambda x: (-len(x), x)):
        if len(leg) >= 2 and ult.endswith(leg) and len(ult) - len(leg) >= 2:
            variantes.append(t[:-1] + [ult[: -len(leg)]])
            break
    return variantes


class Comparador:
    """Rasgos de un par de nombres: alineación, palabras compartidas, puntaje de la librería."""

    def __init__(self, nz: Normalizador, corpus: list[str], cfg: ConfigVinculacion) -> None:
        self.nz, self.cfg = nz, cfg
        vocab = pd.Series(sorted(set(corpus) - {""}))
        pesos = construir_idf(vocab.reset_index(drop=True))
        pesos = neutralizar_genericos(pesos, nz.genericos, peso=1.0)
        genericos_ruido = frozenset(set(GENERICOS_ESTRUCTURALES) | set(GENERICOS_GEOGRAFIA) | set(cfg.genericos_extra))
        # Comparador de la librería con sus defectos (los de deduplicar_importadores).
        self.lib = SimilitudNombre(pesos, vocab.to_numpy(), genericos_estructurales=genericos_ruido)
        self.vocab = set(vocab)
        # Frecuencia documental de cada palabra (para decidir si una palabra sola es "común").
        df_tok: dict[str, int] = defaultdict(int)
        for n in vocab:
            for t in set(n.split()):
                df_tok[t] += 1
        self.df_tok = dict(df_tok)
        self._cache: dict = {}

    def puntajes(self, a: np.ndarray, b: np.ndarray) -> pd.DataFrame:
        """SCORE_RUES_LINKER (0–1) y su desglose, vectorizado sobre pares."""
        if len(a) == 0:
            return pd.DataFrame(columns=["SCORE_RUES_LINKER", "SIM_TOKENS_IDF", "JARO_WINKLER", "DIF_INFORMATIVA",
                                         "USO_CONTENCION"])
        d = self.lib.partes(np.asarray(a, dtype=object), np.asarray(b, dtype=object))
        score = (self.lib.compare(np.asarray(a, dtype=object), np.asarray(b, dtype=object)) + 1.0) / 2.0
        return pd.DataFrame({"SCORE_RUES_LINKER": np.round(score, 4), "SIM_TOKENS_IDF": np.round(d["sim_tokens"].to_numpy(), 4),
                             "JARO_WINKLER": np.round(d["jaro_winkler"].to_numpy(), 4),
                             "DIF_INFORMATIVA": d["dif_informativa"].to_numpy(),
                             "USO_CONTENCION": d["uso_contencion"].to_numpy()})

    def rasgos(self, a: str, b: str, trunc_a: bool, trunc_b: bool = False, subs: bool = False) -> dict:
        clave = (a, b, trunc_a, trunc_b, subs)
        if clave in self._cache:
            return self._cache[clave]
        nz = self.nz
        ua, ub = nz.utiles(a), nz.utiles(b)
        sa, sb, pares = alinear(ua, ub, self.cfg.max_ediciones_palabra, trunc_a, trunc_b)
        comp = [(x, y) for x, y, _ in pares if x not in nz.genericos and y not in nz.genericos and len(x) >= 3 and len(y) >= 3]
        comp_trunc = [(x, y) for x, y, tp in pares if tp == "truncada" and y not in nz.genericos]
        fuertes = [(x, y) for x, y, tp in pares if tp == "exacta" and x not in nz.genericos and len(x) >= 3
                   and x not in self.cfg.nombres_pila]
        tipos = {tp for _, _, tp in pares}
        r = {
            "UTILES_A": len(ua), "UTILES_B": len(ub),
            "SOBRA_A": " ".join(sa), "SOBRA_B": " ".join(sb),
            "COMPARTIDAS": " ".join(sorted({y for _, y in comp})),
            "N_COMPARTIDAS": len(comp) + (len(comp_trunc) if not comp else 0),
            "N_FUERTES": len(fuertes),
            "SOLO_NOMBRE_PILA": bool(comp) and all(x in self.cfg.nombres_pila for x, _ in comp),
            "SOBRAS_GENERICAS": all(t in nz.genericos for t in sa + sb),
            "SIN_SOBRAS": not sa and not sb,
            "ERRATA": "errata" in tipos, "TRUNCADA": "truncada" in tipos,
            "MARCAS_OK": nz.marcas(a, trunc_a) == nz.marcas(b, trunc_b),
            "COMPACTO_IGUAL": bool(nz.compacto(a)) and nz.compacto(a) == nz.compacto(b),
            "ABREVIATURA": False,
        }
        if not r["N_COMPARTIDAS"] and not r["COMPACTO_IGUAL"]:
            va, vb = fichas_abreviatura(a, nz), fichas_abreviatura(b, nz)
            r["ABREVIATURA"] = any(es_abreviatura(x, y, nz.genericos, subs) or es_abreviatura(y, x, nz.genericos, subs)
                                   for x in va for y in vb)
        self._cache[clave] = r
        return r

    def palabra_comun(self, nombre: str) -> bool:
        """Nombre de UNA sola palabra útil que además aparece en varios nombres del corpus."""
        u = self.nz.utiles(nombre)
        return len(u) == 1 and self.df_tok.get(u[0], 0) >= 3


# ════════════════════════════════════════════════════════════════════════════
# 5 · UNION-FIND
# ════════════════════════════════════════════════════════════════════════════
def componentes(n: int, a, b) -> np.ndarray:
    """Componentes conexas numeradas 0..k-1 en orden de primera aparición (determinista)."""
    padre = list(range(n))

    def raiz(x: int) -> int:
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x
    for x, y in zip(a, b):
        rx, ry = raiz(int(x)), raiz(int(y))
        if rx != ry:
            padre[max(rx, ry)] = min(rx, ry)
    return pd.factorize(np.array([raiz(i) for i in range(n)]))[0]


# ════════════════════════════════════════════════════════════════════════════
# 6 · FORMA LEGAL (familias) — dos formas de familias distintas = personas jurídicas distintas
# ════════════════════════════════════════════════════════════════════════════
_FAMILIAS_LEGALES = [
    ("STIFTUNG_KG", r"STIFTUNG (?:&|\+|UND)? ?CO KG"), ("SE_KG", r"\bSE (?:&|\+|UND)? ?CO KG"),
    ("GMBH_KG", r"(?:GMBH|AG) (?:&|\+|UND)? ?CO (?:KG|OHG)"), ("KGAA", r"\bKGAA\b"), ("OHG", r"\bOHG\b"),
    ("KG", r"\bKG\b"), ("GMBH", r"\bGMBH\b|BESCHRA?E?NKTER HAFTUNG"), ("AG", r"\bAG\b|AKTIENGESELLSCHAFT"),
    ("SE", r"\bSE\b"), ("COOP", r"\bS ?COOP\b|COOPERATIV|\bEG\b|GENOSSENSCHAFT|\bSAT\b"), ("SAS", r"\bSASU?\b"),
    ("SL", r"\bSLU?\b|\bSRLS?\b|\bSARL\b|\bEURL\b|\bKFT\b|\bSP ZOO\b|LIMITADA|RESPONSABILITA LIMITATA"),
    ("SCA", r"\bSCA\b"), ("SA", r"\bSAU?\b|\bSPA\b|ANONIMA|PER AZIONI"), ("BV", r"\bBV\b"), ("NV", r"\bNV\b"),
    ("LTD", r"\bLTD\b|LIMITED\b"), ("INC", r"\bINC\b|\bCORP\b"), ("LLC", r"\bLLC\b"),
]
_FAMILIAS_RE = [(k, re.compile(p)) for k, p in _FAMILIAS_LEGALES]


def familias_legales(nombres: pd.Series) -> np.ndarray:
    x = ascii_mayusculas(sanear_texto(pd.Series(nombres, dtype="object").fillna("").astype(str).astype("string")))
    x = x.str.replace(r"\([^()]*\)?", " ", regex=True).str.replace(r"[.,;:/\-]", " ", regex=True)
    x = x.str.replace(r"\s+", " ", regex=True).str.replace(r"\b([A-Z]) (?=[A-Z]\b)", r"\1", regex=True).str.strip()

    def familia(v: str) -> str:
        for k, rx in _FAMILIAS_RE:
            if rx.search(v):
                return k
        return ""
    return np.array([familia(v) for v in x.astype(object).fillna("")], dtype=object)


def _familias_compatibles(fa: str, fb: str, estricto: bool = False) -> bool:
    if estricto:
        return fa == fb
    return fa == fb or not fa or not fb


# ════════════════════════════════════════════════════════════════════════════
# 7 · DECISIONES DE REVISIÓN (entrada humana)
# ════════════════════════════════════════════════════════════════════════════
COLUMNAS_DECISION = ["TIPO", "FUENTE", "CLAVE_A", "NOMBRE_A", "CLAVE_B", "NOMBRE_B", "DECISION", "AUTOR", "FECHA", "RAZON"]
DECISIONES_VINCULO = {"EMPRESA", "GRUPO", "NO_VINCULAR", "RELACION_PROPIEDAD"}
DECISIONES_DUPLICADO = {"MISMA_ENTIDAD", "DISTINTA"}
_SINONIMOS_DECISION = {"MISMA_EMPRESA": "EMPRESA", "MISMO_GRUPO": "GRUPO", "DISTINTA_EMPRESA": "NO_VINCULAR",
                       "NO": "NO_VINCULAR", "NO_ES": "NO_VINCULAR", "RECHAZAR": "NO_VINCULAR", "SI": "EMPRESA",
                       "PROPIEDAD": "RELACION_PROPIEDAD", "MISMA": "MISMA_ENTIDAD", "DUPLICADO": "MISMA_ENTIDAD"}


def _clave(x) -> pd.Series:
    """Clave natural sin espacios de más: sobrevive a un CSV abierto y guardado en Excel."""
    return pd.Series(x, dtype="object").fillna("").astype(str).str.strip().str.replace(r"\s+", " ", regex=True)


def clave_ancla(pais, nombre) -> pd.Series:
    return _clave(pais).str.cat(_clave(nombre), sep="|")


def leer_decisiones(rutas) -> pd.DataFrame:
    """Lee 0..N archivos de decisiones (CSV o XLSX con hoja REVISION/DECISIONES). El orden importa: el último manda."""
    piezas = []
    for orden, ruta in enumerate(rutas or ()):
        p = Path(ruta)
        if not p.exists():
            print(f"  ⚠️  archivo de decisiones no encontrado, se omite: {p}")
            continue
        if p.suffix.lower() in (".xlsx", ".xlsm"):
            # Excel de resultados: DECISIONES (historial aplicado) + REVISION (lo que se llenó después).
            hojas = pd.ExcelFile(p).sheet_names
            leidas = [h for h in ("DECISIONES", "REVISION") if h in hojas]
            if not leidas:
                raise ValueError(f"{p.name}: no tiene hoja DECISIONES ni REVISION.")
            partes = []
            for h in leidas:
                x = pd.read_excel(p, sheet_name=h, dtype=str).fillna("")
                x.columns = [c.strip().upper() for c in x.columns]
                if "DECISION" in x.columns:
                    partes.append(x[x["DECISION"].str.strip() != ""])
            r = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(columns=COLUMNAS_DECISION)
            clave = [c for c in ("TIPO", "FUENTE", "CLAVE_A", "CLAVE_B", "DECISION") if c in r.columns]
            r = r.drop_duplicates(clave, keep="last")
        else:
            texto = p.read_bytes()[:4096].decode("utf-8", errors="ignore")
            sep = ";" if texto.splitlines()[0].count(";") > texto.splitlines()[0].count(",") else ","
            try:
                r = pd.read_csv(p, dtype=str, keep_default_na=False, sep=sep, encoding="utf-8-sig")
            except UnicodeDecodeError:
                r = pd.read_csv(p, dtype=str, keep_default_na=False, sep=sep, encoding="cp1252")
        r.columns = [c.strip().upper() for c in r.columns]
        if "DECISION" not in r.columns and "DECISION_HUMANA" in r.columns:
            r = r.rename(columns={"DECISION_HUMANA": "DECISION"})
        faltan = [c for c in ("TIPO", "FUENTE", "CLAVE_A", "CLAVE_B", "DECISION") if c not in r.columns]
        if faltan:
            raise KeyError(f"Qué pasó: {p.name} no trae las columnas {faltan}.\nQué hacer: use la plantilla "
                           f"(columnas {COLUMNAS_DECISION}) o la hoja REVISION del Excel de resultados.")
        for c in COLUMNAS_DECISION:
            if c not in r.columns:
                r[c] = ""
        r = r[COLUMNAS_DECISION].apply(lambda c: c.astype(str).str.strip())
        r = r[r["DECISION"] != ""].copy()
        r["TIPO"] = r["TIPO"].str.upper().replace({"CRUCE": "VINCULO", "VÍNCULO": "VINCULO"})
        r["DECISION"] = r["DECISION"].str.upper().str.replace(" ", "_").replace(_SINONIMOS_DECISION)
        r["FUENTE"] = r["FUENTE"].str.upper()
        r["CLAVE_A"], r["CLAVE_B"] = _clave(r["CLAVE_A"]).to_numpy(), _clave(r["CLAVE_B"]).to_numpy()
        r["ARCHIVO"] = p.name
        r["ORDEN"] = orden
        malas = r[((r.TIPO == "VINCULO") & ~r.DECISION.isin(DECISIONES_VINCULO))
                  | ((r.TIPO == "DUPLICADO") & ~r.DECISION.isin(DECISIONES_DUPLICADO))
                  | ~r.TIPO.isin({"VINCULO", "DUPLICADO"})]
        if len(malas):
            raise ValueError(f"Qué pasó: {len(malas)} filas de {p.name} con TIPO o DECISION no válidos.\n"
                             f"Qué hacer: TIPO ∈ {{VINCULO, DUPLICADO}}; DECISION ∈ {sorted(DECISIONES_VINCULO)} "
                             f"o {sorted(DECISIONES_DUPLICADO)}.\n{malas.head().to_string()}")
        piezas.append(r)
    if not piezas:
        return pd.DataFrame(columns=COLUMNAS_DECISION + ["ARCHIVO", "ORDEN"])
    return pd.concat(piezas, ignore_index=True)


def plantilla_decisiones(ruta: str | Path) -> Path:
    """Escribe un CSV vacío con las columnas esperadas y dos filas de ejemplo comentadas."""
    ej = pd.DataFrame([
        ["VINCULO", "ORBIS", "España|GALLETAS GULLON, S.A.,", "GALLETAS GULLON, S.A.,", "ESA34002501",
         "GALLETAS GULLON, SA", "EMPRESA", "iniciales", "2026-09-29", "Mismo nombre y web gullon.es"],
        ["DUPLICADO", "EXPORTADORES", "España|CTRA SLU", "CTRA SLU", "España|CUETARA SLU", "CUETARA SLU",
         "MISMA_ENTIDAD", "iniciales", "2026-09-29", "Abreviatura de aduana de CUETARA"],
    ], columns=COLUMNAS_DECISION)
    ruta = Path(ruta)
    ej.to_csv(ruta, index=False, encoding="utf-8-sig")
    return ruta


# ════════════════════════════════════════════════════════════════════════════
# 8 · DEDUPLICACIÓN INTERNA
# ════════════════════════════════════════════════════════════════════════════
def _raiz_id(x: str) -> str:
    """Raíz de un BvD ID: sin sufijo de sucursal (DE…-1003) y, en Francia, el SIREN de un SIRET."""
    x = str(x).split("-")[0].strip()
    m = re.fullmatch(r"FR(\d{9})\d{5}", x)
    return f"FR{m.group(1)}" if m else x


def deduplicar_ancla(prep: pd.DataFrame, nz: Normalizador, comp: Comparador, cfg: ConfigVinculacion,
                     dec: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Agrupa las filas de la ancla en empresas. Devuelve (grafias, uniones, pendientes).

    Unidad de trabajo: la GRAFÍA (país + nombre tal como viene). Reglas automáticas,
    todas con la forma legal compatible (SL ≠ S. COOP.):
      1. mismo nombre normalizado en el mismo país;
      2. mismas letras con otros espacios (GB FOODS = GBFOODS);
      3. mismas palabras con una errata o un truncamiento y SCORE ≥ umbral_dedup.
    Lo que casi cumple (contención genérica, sigla, subsecuencia, palabra distintiva
    compartida) va a pendientes: lo decide un humano con TIPO=DUPLICADO.
    """
    g = prep.assign(CLAVE=clave_ancla(prep["PAIS_ORIGINAL"], prep["NOMBRE_ORIGINAL"]).to_numpy())
    graf = (g.groupby("CLAVE", sort=True)
             .agg(PAIS_ORIGINAL=("PAIS_ORIGINAL", "first"), NOMBRE_ORIGINAL=("NOMBRE_ORIGINAL", "first"),
                  PAIS_ISO3=("PAIS_ISO3", "first"), NOMBRE_NORM=("NOMBRE_NORM", "first"),
                  ALIAS_NORM=("ALIAS_NORM", "first"), TRUNCADO=("TRUNCADO", "max"),
                  INTERMEDIARIO=("INTERMEDIARIO", "max"), N_FILAS=("ID_REGISTRO", "size"))
             .reset_index())
    graf["FORMA_LEGAL"] = familias_legales(graf["NOMBRE_ORIGINAL"])
    n = len(graf)
    uniones: list[tuple[int, int, str]] = []
    pend: list[tuple[int, int, str]] = []
    iso = graf["PAIS_ISO3"].to_numpy()
    norm = graf["NOMBRE_NORM"].to_numpy()
    alias = graf["ALIAS_NORM"].to_numpy()
    forma = graf["FORMA_LEGAL"].to_numpy()
    trunc = graf["TRUNCADO"].to_numpy()
    util = np.array([bool(nz.utiles(x)) for x in norm])
    for pais in pd.unique(iso):
        idx = np.flatnonzero(iso == pais)
        if len(idx) < 2:
            continue
        sa = [(i, norm[i], "N") for i in idx if norm[i]] + [(i, alias[i], "A") for i in idx if alias[i]]
        a_i = np.array([x[0] for x in sa])
        a_s = np.array([x[1] for x in sa], dtype=object)
        a_t = np.array([x[2] for x in sa])
        ii, jj = np.triu_indices(len(sa), k=1)
        keep = a_i[ii] != a_i[jj]
        ii, jj = ii[keep], jj[keep]
        if not len(ii):
            continue
        punt = comp.puntajes(a_s[ii], a_s[jj])
        vistos: dict[tuple[int, int], tuple[int, str, str]] = {}
        for k in range(len(ii)):
            p, q = int(a_i[ii[k]]), int(a_i[jj[k]])
            x, y = a_s[ii[k]], a_s[jj[k]]
            por_alias = a_t[ii[k]] == "A" or a_t[jj[k]] == "A"
            sc = float(punt["SCORE_RUES_LINKER"].iat[k])
            if x == y:
                r = {"SIN_SOBRAS": True, "N_COMPARTIDAS": 1, "MARCAS_OK": True, "COMPACTO_IGUAL": True,
                     "SOBRAS_GENERICAS": True, "ABREVIATURA": False, "ERRATA": False, "TRUNCADA": False}
            else:
                r = comp.rasgos(x, y, bool(trunc[p]), bool(trunc[q]), subs=True)
            compatibles = _familias_compatibles(forma[p], forma[q])
            if not (util[p] and util[q]) and x != y:
                continue
            motivo, auto = "", False
            if x == y and not por_alias:
                auto, motivo = compatibles, ("mismo nombre normalizado" if compatibles else
                                             f"mismo nombre, forma legal distinta ({forma[p] or '—'} ≠ {forma[q] or '—'})")
            elif por_alias and (r["SIN_SOBRAS"] or r["COMPACTO_IGUAL"]):
                motivo = "coincide con el nombre entre paréntesis (¿marca?)"
            elif r["COMPACTO_IGUAL"]:
                auto, motivo = compatibles, "mismas letras, otros espacios" if compatibles else "mismas letras, forma legal distinta"
            elif r["SIN_SOBRAS"] and r["N_COMPARTIDAS"] and r["MARCAS_OK"] and (
                    sc >= cfg.umbral_dedup or (r["TRUNCADA"] and not r["ERRATA"] and sc >= 0.75)):
                auto = compatibles
                tipo = "truncamiento" if r["TRUNCADA"] else "errata"
                motivo = f"mismas palabras con {tipo}" if compatibles else f"mismas palabras, forma legal distinta ({forma[p] or '—'} ≠ {forma[q] or '—'})"
            elif r["SIN_SOBRAS"] and r["N_COMPARTIDAS"]:
                motivo = f"mismas palabras (score {sc:.2f})"
            elif r["N_COMPARTIDAS"] and r["SOBRAS_GENERICAS"]:
                motivo = "un nombre contiene al otro; lo que sobra es genérico"
            elif r["ABREVIATURA"]:
                motivo = "posible sigla o abreviatura"
            elif r["N_COMPARTIDAS"] and sc >= 0.75:
                motivo = f"comparten palabra distintiva ({r.get('COMPARTIDAS', '')})"
            elif sc >= cfg.umbral_parecido:
                motivo = f"nombres muy parecidos (score {sc:.2f})"
            if not motivo:
                continue
            clave = (min(p, q), max(p, q))
            rango = 0 if auto else 1
            if clave not in vistos or rango < vistos[clave][0]:
                vistos[clave] = (rango, motivo, "auto" if auto else "pend")
        for (p, q), (_, motivo, tipo) in vistos.items():
            (uniones if tipo == "auto" else pend).append((p, q, motivo))
    # Decisiones humanas TIPO=DUPLICADO sobre la ancla
    pos = {k: i for i, k in enumerate(graf["CLAVE"])}
    vetos: set[tuple[int, int]] = set()
    d = dec[(dec.TIPO == "DUPLICADO") & (dec.FUENTE == cfg.ancla.clave)] if len(dec) else dec
    desconocidas = []
    for r in d.sort_values("ORDEN", kind="stable").itertuples():
        if r.CLAVE_A not in pos or r.CLAVE_B not in pos:
            desconocidas.append((r.CLAVE_A, r.CLAVE_B))
            continue
        p, q = sorted((pos[r.CLAVE_A], pos[r.CLAVE_B]))
        if r.DECISION == "MISMA_ENTIDAD":
            uniones.append((p, q, f"decisión: {r.AUTOR or r.ARCHIVO}"))
            vetos.discard((p, q))
        else:
            vetos.add((p, q))
    uniones = [(p, q, m) for p, q, m in uniones if (min(p, q), max(p, q)) not in vetos]
    etiquetas = componentes(n, [u[0] for u in uniones], [u[1] for u in uniones])
    graf["_COMP"] = etiquetas
    un = pd.DataFrame(uniones, columns=["pos_a", "pos_b", "REGLA"])
    pe = pd.DataFrame(pend, columns=["pos_a", "pos_b", "MOTIVO"])
    if len(pe):
        pe = pe[etiquetas[pe.pos_a.to_numpy()] != etiquetas[pe.pos_b.to_numpy()]]
        pe = pe[[(min(a, b), max(a, b)) not in vetos for a, b in zip(pe.pos_a, pe.pos_b)]].reset_index(drop=True)
    graf.attrs["vetos"] = vetos
    graf.attrs["desconocidas"] = desconocidas
    return graf, un, pe


def deduplicar_destino(prep: pd.DataFrame, f: Fuente, nz: Normalizador, dec: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Entidades dentro de una base destino. Con identificador: SOLO por raíz del ID.

    Sin identificador: mismo nombre normalizado, mismos países de evidencia y misma forma
    legal (en registros de grupos como fDi Markets, 'Aldi' y 'Aldi Inc' son matriz y filial).
    """
    n = len(prep)
    uniones: list[tuple[int, int, str]] = []
    forma = familias_legales(prep["NOMBRE_ORIGINAL"])
    if f.col_id:
        raiz = prep["ID_REGISTRO"].map(_raiz_id).to_numpy()
        for _, idx in pd.Series(np.arange(n)).groupby(raiz, sort=True):
            idx = list(idx)
            uniones += [(idx[0], j, "mismo identificador raíz") for j in idx[1:]]
    else:
        # Sin identificador, en registros de grupos (fDi) dos nombres PARECIDOS suelen ser matriz y filial
        # ('Aldi' / 'Aldi Inc'; 'Compagnie Fruitiere' / 'Compagnie Fruitiere (UK)'): solo se une la MISMA
        # cadena (salvo mayúsculas y espacios) con los mismos países.
        crudo = _clave(prep["NOMBRE_ORIGINAL"]).str.casefold()
        llave = pd.Series(list(zip(crudo, prep["PAISES_EVIDENCIA"])))
        for (nom, _), idx in pd.Series(np.arange(n)).groupby(llave, sort=True):
            if not nom or not nz.utiles(prep["NOMBRE_NORM"].iat[int(idx.iloc[0])]):
                continue
            idx = list(idx)
            uniones += [(idx[0], j, "mismo nombre exacto y mismo país") for j in idx[1:]]
    pos = {k: i for i, k in enumerate(prep["ID_REGISTRO"])}
    d = dec[(dec.TIPO == "DUPLICADO") & (dec.FUENTE == f.clave)] if len(dec) else dec
    vetos = set()
    for r in d.sort_values("ORDEN", kind="stable").itertuples():
        if r.CLAVE_A in pos and r.CLAVE_B in pos:
            p, q = sorted((pos[r.CLAVE_A], pos[r.CLAVE_B]))
            if r.DECISION == "MISMA_ENTIDAD":
                uniones.append((p, q, f"decisión: {r.AUTOR or r.ARCHIVO}"))
            else:
                vetos.add((p, q))
    uniones = [(p, q, m) for p, q, m in uniones if (min(p, q), max(p, q)) not in vetos]
    et = componentes(n, [u[0] for u in uniones], [u[1] for u in uniones])
    out = prep.copy()
    out["ID_ENTIDAD"] = [f"{f.prefijo_id}-E{e + 1:04d}" for e in et]
    out["N_EN_ENTIDAD"] = out.groupby("ID_ENTIDAD")["ID_ENTIDAD"].transform("size").to_numpy()
    out["FORMA_LEGAL"] = forma
    un = pd.DataFrame(uniones, columns=["pos_a", "pos_b", "REGLA"])
    return out, un


# ════════════════════════════════════════════════════════════════════════════
# 9 · CANDIDATOS Y CLASIFICACIÓN ANCLA × DESTINO
# ════════════════════════════════════════════════════════════════════════════
ORDEN_NIVEL = {"EMPRESA": 0, "GRUPO": 1, "RELACION_PROPIEDAD": 2, "RELACION_MATRIZ": 3, "REVISAR": 4, "": 9}
ORDEN_BANDA = {"A": 0, "B": 1, "C": 2, "": 9}


def _claves_borrosas(nombre: str, nz: Normalizador) -> set[str]:
    """Palabras distintivas + sus variantes con una letra borrada (SymSpell): capturan erratas de una letra."""
    out = set()
    for t in nz.utiles(nombre):
        if t in nz.genericos or len(t) < 3:
            continue
        out.add(t)
        if len(t) >= 4:
            out.update(t[:i] + t[i + 1:] for i in range(len(t)))
    return out


def _superficies_destino(prep: pd.DataFrame) -> pd.DataFrame:
    piezas = [pd.DataFrame({"pos": np.arange(len(prep)), "SUP_B": prep[c].to_numpy(), "TIPO_SUP_B": t})
              for c, t in (("NOMBRE_NORM", "NOMBRE"), ("ALIAS_NORM", "ALIAS"), ("GRUPO_NORM", "MATRIZ"))]
    s = pd.concat(piezas, ignore_index=True)
    return s[s["SUP_B"] != ""].reset_index(drop=True)


def generar_candidatos(graf: pd.DataFrame, prep_b: pd.DataFrame, nz: Normalizador, comp: Comparador,
                       cfg: ConfigVinculacion) -> pd.DataFrame:
    """Pares (grafía ancla × registro destino) que merecen clasificarse.

    Exhaustivo en la práctica: se consideran TODOS los pares y se descartan solo los que no
    comparten ninguna palabra distintiva (ni con una errata), no se parecen como cadena
    (Jaro-Winkler < 0,85) y no son una sigla del otro.
    """
    sa = pd.concat([
        pd.DataFrame({"g": np.arange(len(graf)), "SUP_A": graf["NOMBRE_NORM"].to_numpy(), "TIPO_SUP_A": "NOMBRE"}),
        pd.DataFrame({"g": np.arange(len(graf)), "SUP_A": graf["ALIAS_NORM"].to_numpy(), "TIPO_SUP_A": "ALIAS"}),
    ], ignore_index=True)
    sa = sa[sa["SUP_A"] != ""].reset_index(drop=True)
    sb = _superficies_destino(prep_b)
    ua = pd.unique(sa["SUP_A"])
    ub = pd.unique(sb["SUP_B"])
    # (1) palabras compartidas con tolerancia de una letra (índice invertido)
    inv: dict[str, set[int]] = defaultdict(set)
    for j, s in enumerate(ub):
        for k in _claves_borrosas(s, nz):
            inv[k].add(j)
    pares: set[tuple[int, int]] = set()
    for i, s in enumerate(ua):
        js: set[int] = set()
        for k in _claves_borrosas(s, nz):
            js |= inv.get(k, set())
        pares.update((i, j) for j in js)
    # (2) cadenas parecidas (todas contra todas, en C)
    if len(ua) and len(ub):
        m = rf_process.cdist(list(ua), list(ub), scorer=JaroWinkler.normalized_similarity, score_cutoff=0.85,
                             dtype=np.float32, workers=-1)
        ii, jj = np.nonzero(m >= 0.85)
        pares.update(zip(ii.tolist(), jj.tolist()))
    # (3) siglas y abreviaturas (primera letra igual y nombre corto o con partes abreviadas)
    primeras_b = defaultdict(list)
    fichas_b = []
    for j, s in enumerate(ub):
        fb = [t for t in s.split() if t not in nz.legales and t not in CONECTORES]
        fichas_b.append(fb)
        if fb:
            primeras_b[fb[0][0]].append(j)
    for i, s in enumerate(ua):
        for fa in fichas_abreviatura(s, nz):
            if not fa or len(fa) > 5 or not any(len(t) <= 4 for t in fa):
                continue
            for j in primeras_b.get(fa[0][0], []):
                if (i, j) not in pares and es_abreviatura(fa, fichas_b[j], nz.genericos):
                    pares.add((i, j))
    if not pares:
        return pd.DataFrame()
    pi = np.array(sorted(pares))
    u = pd.DataFrame({"SUP_A": ua[pi[:, 0]], "SUP_B": ub[pi[:, 1]]})
    u = pd.concat([u, comp.puntajes(u["SUP_A"].to_numpy(), u["SUP_B"].to_numpy())], axis=1)
    # Expandir a grafías y registros
    x = sa.merge(u, on="SUP_A").merge(sb, on="SUP_B")
    x = x.rename(columns={"pos": "pos_b"})
    return x.reset_index(drop=True)


def clasificar(x: pd.DataFrame, graf: pd.DataFrame, prep_b: pd.DataFrame, nz: Normalizador, comp: Comparador,
               cfg: ConfigVinculacion) -> pd.DataFrame:
    """NIVEL (EMPRESA/GRUPO/RELACION_MATRIZ/REVISAR), BANDA (A/B/C) y MOTIVO legible por par."""
    if x.empty:
        return x
    trunc_a = graf["TRUNCADO"].to_numpy()[x["g"].to_numpy()]
    inter_a = graf["INTERMEDIARIO"].to_numpy()[x["g"].to_numpy()]
    iso_a = graf["PAIS_ISO3"].to_numpy()[x["g"].to_numpy()]
    evid_b = prep_b["PAISES_EVIDENCIA"].to_numpy()[x["pos_b"].to_numpy()]
    grupo_b = prep_b["GRUPO_ORIGINAL"].to_numpy()[x["pos_b"].to_numpy()]
    niveles, bandas, motivos, rasgos_l, mismo = [], [], [], [], []
    for k, row in enumerate(x.itertuples(index=False)):
        r = comp.rasgos(row.SUP_A, row.SUP_B, bool(trunc_a[k]))
        rasgos_l.append(r)
        if not iso_a[k] or not evid_b[k]:
            mp = None
        else:
            mp = iso_a[k] in set(evid_b[k].split("|"))
        mismo.append("SÍ" if mp else ("NO" if mp is False else "SIN DATO"))
        sc = row.SCORE_RUES_LINKER
        comparte = r["N_COMPARTIDAS"] >= 1 or r["COMPACTO_IGUAL"]
        empresa = comparte and r["SIN_SOBRAS"] and r["MARCAS_OK"] and sc >= cfg.umbral_piso_empresa
        grupo = r["N_COMPARTIDAS"] >= 1 and r["SOBRAS_GENERICAS"] and not r["SIN_SOBRAS"] and not r["SOLO_NOMBRE_PILA"]
        marca = r.get("COMPARTIDAS", "")
        sobras = " / ".join(s for s in (r["SOBRA_A"], r["SOBRA_B"]) if s)
        nivel, banda, motivo = "", "", ""
        if row.TIPO_SUP_B == "MATRIZ":
            if empresa or grupo:
                nivel, banda = "RELACION_MATRIZ", "C"
                motivo = f"La matriz reportada del registro ({grupo_b[k]}) coincide con el exportador; el registro es otra empresa del grupo"
        elif empresa:
            if mp is False:
                # Otro país: solo un nombre IDÉNTICO y no común sugiere el mismo grupo (filial/matriz);
                # una errata entre países es otra empresa con frecuencia (FLORIETTE NLD ≠ Florette FRA).
                nivel = "GRUPO"
                if r["COMPACTO_IGUAL"] and not r["ERRATA"] and not comp.palabra_comun(row.SUP_A) and r["UTILES_A"] >= 1 \
                        and len(r["COMPARTIDAS"].replace(" ", "")) >= 5:
                    banda, motivo = "B", "Mismo nombre pero en otro país: probable filial o matriz del mismo grupo"
                else:
                    banda, motivo = "C", "Nombre casi igual pero en otro país: ¿mismo grupo o empresa homónima?"
            else:
                detalles = []
                if r["ERRATA"]:
                    detalles.append("errata de una letra")
                if r["TRUNCADA"]:
                    detalles.append("nombre truncado en el origen")
                if row.TIPO_SUP_A == "ALIAS" or row.TIPO_SUP_B == "ALIAS":
                    detalles.append("coincide por el nombre entre paréntesis")
                if comp.palabra_comun(row.SUP_A):
                    detalles.append("nombre de una sola palabra, frecuente en las bases: posible homónimo")
                if mp is None:
                    detalles.append("sin país para contrastar")
                if inter_a[k]:
                    detalles.append("el exportador es un agente de carga")
                nivel = "EMPRESA"
                if not detalles and r["COMPACTO_IGUAL"]:
                    banda = "A"
                    motivo = "Mismo nombre tras normalizar (solo cambian forma legal, puntuación o tildes) y mismo país"
                else:
                    banda = "B"
                    motivo = "Mismas palabras y mismo país; " + "; ".join(detalles or [f"score {sc:.2f}"])
        elif grupo and not inter_a[k]:
            nivel = "GRUPO"
            if mp is False:
                banda = "C"
                motivo = f"Comparte la marca {marca} pero está en otro país; lo que sobra es genérico ({sobras})"
            else:
                banda = "B"
                motivo = f"Comparte la marca {marca}; lo que sobra es genérico ({sobras}): otra sociedad del mismo grupo"
        elif (r["N_FUERTES"] >= 1 and sc >= 0.50) or (r["N_COMPARTIDAS"] >= 1 and not r["SOLO_NOMBRE_PILA"] and sc >= 0.70):
            nivel, banda = "REVISAR", "C"
            motivo = f"Comparte {marca or 'palabra distintiva'}, pero difieren en: {sobras}"
        elif r["ABREVIATURA"]:
            nivel, banda, motivo = "REVISAR", "C", "Posible sigla o abreviatura del otro nombre"
        elif sc >= cfg.umbral_parecido:
            nivel, banda, motivo = "REVISAR", "C", f"Nombres muy parecidos como cadena (score {sc:.2f})"
        niveles.append(nivel)
        bandas.append(banda)
        motivos.append(motivo)
    out = x.copy()
    out["NIVEL_AUTO"], out["BANDA_AUTO"], out["MOTIVO"], out["MISMO_PAIS"] = niveles, bandas, motivos, mismo
    rs = pd.DataFrame(rasgos_l)
    out["COMPARTIDAS"] = rs["COMPARTIDAS"].to_numpy()
    out["SOBRA_EXPORTADOR"], out["SOBRA_DESTINO"] = rs["SOBRA_A"].to_numpy(), rs["SOBRA_B"].to_numpy()
    out = out[out["NIVEL_AUTO"] != ""].copy()
    out["_r"] = out["NIVEL_AUTO"].map(ORDEN_NIVEL) * 10 + out["BANDA_AUTO"].map(ORDEN_BANDA)
    out = out.sort_values(["g", "pos_b", "_r", "SCORE_RUES_LINKER"], ascending=[True, True, True, False], kind="stable")
    return out.drop_duplicates(["g", "pos_b"]).drop(columns="_r").reset_index(drop=True)


def evidencia_extra(pares: pd.DataFrame, graf_prep: pd.DataFrame, prep_b: pd.DataFrame) -> pd.DataFrame:
    """Variables adicionales presentes en AMBOS lados: ID comparable, web, email, teléfono, ciudad.

    Un ID comparable igual decide EMPRESA (A). Web/email/teléfono igual suben un candidato
    que comparte palabra a B. Distintos no vetan (una empresa tiene varias webs); se informan.
    """
    if pares.empty:
        return pares
    out = pares.copy()
    notas = [[] for _ in range(len(out))]
    for col, nombre in (("ID_COMPARABLE", "identificador"), ("DOMINIO_WEB", "web"), ("DOMINIO_EMAIL", "correo"),
                        ("TEL_DIGITOS", "teléfono"), ("CIUDAD_NORM", "ciudad")):
        if col not in graf_prep.columns or not (graf_prep[col] != "").any() or not (prep_b[col] != "").any():
            continue
        va = graf_prep[col].to_numpy()[out["g"].to_numpy()]
        vb = prep_b[col].to_numpy()[out["pos_b"].to_numpy()]
        igual = (va != "") & (va == vb)
        for k in np.flatnonzero(igual):
            notas[k].append(f"mismo {nombre}")
            if col == "ID_COMPARABLE":
                out.iat[k, out.columns.get_loc("NIVEL_AUTO")] = "EMPRESA"
                out.iat[k, out.columns.get_loc("BANDA_AUTO")] = "A"
            elif col in ("DOMINIO_WEB", "DOMINIO_EMAIL", "TEL_DIGITOS") and out["BANDA_AUTO"].iat[k] == "C" \
                    and out["COMPARTIDAS"].iat[k]:
                out.iat[k, out.columns.get_loc("NIVEL_AUTO")] = "GRUPO"
                out.iat[k, out.columns.get_loc("BANDA_AUTO")] = "B"
    out["EVIDENCIA_EXTRA"] = ["; ".join(n) for n in notas]
    return out


# ════════════════════════════════════════════════════════════════════════════
# 10 · VÍNCULOS POR EMPRESA, DECISIONES Y PROPAGACIÓN POR WEB
# ════════════════════════════════════════════════════════════════════════════
def _agregar_por_entidad(pares: pd.DataFrame, ent_de_g: np.ndarray, graf: pd.DataFrame) -> pd.DataFrame:
    """Mejor par por (empresa ancla, fuente, registro destino) entre todas sus grafías."""
    if pares.empty:
        return pares.assign(ENT=pd.Series(dtype=int))
    x = pares.copy()
    x["ENT"] = ent_de_g[x["g"].to_numpy()]
    x["GRAFIA_ANCLA"] = graf["NOMBRE_ORIGINAL"].to_numpy()[x["g"].to_numpy()]
    x["_r"] = x["NIVEL_AUTO"].map(ORDEN_NIVEL).fillna(9) * 10 + x["BANDA_AUTO"].map(ORDEN_BANDA).fillna(9)
    x = x.sort_values(["ENT", "FUENTE_DESTINO", "pos_b", "_r", "SCORE_RUES_LINKER"],
                      ascending=[True, True, True, True, False], kind="stable")
    return x.drop_duplicates(["ENT", "FUENTE_DESTINO", "pos_b"]).drop(columns="_r").reset_index(drop=True)


def _mapear_decisiones(dec: pd.DataFrame, graf: pd.DataFrame, ent_de_g: np.ndarray, prep: dict,
                       destinos: list[str]) -> tuple[pd.DataFrame, list[str]]:
    """Traduce las decisiones VINCULO a (ENT, FUENTE, pos_b). Verifica que el destino sea el mismo registro."""
    if dec.empty:
        return pd.DataFrame(columns=["ENT", "FUENTE_DESTINO", "pos_b", "DECISION", "AUTOR", "FECHA", "RAZON",
                                     "ARCHIVO", "ORDEN", "CLAVE_A_DEC"]), []
    d = dec[dec["TIPO"] == "VINCULO"].copy()
    pos_g = {k: i for i, k in enumerate(graf["CLAVE"])}
    avisos, filas, errores = [], [], []
    nombre_ok = {f: dict(zip(prep[f]["ID_REGISTRO"], _clave(prep[f]["NOMBRE_ORIGINAL"]).str.casefold())) for f in destinos}
    pos_b = {f: {k: i for i, k in enumerate(prep[f]["ID_REGISTRO"])} for f in destinos}
    for k, r in enumerate(d.itertuples(index=False)):
        if r.FUENTE not in pos_b:
            avisos.append(f"decisión con FUENTE desconocida {r.FUENTE!r}: {r.CLAVE_A} ↔ {r.CLAVE_B}")
            continue
        if r.CLAVE_A not in pos_g:
            avisos.append(f"{r.FUENTE}: la grafía del exportador '{r.CLAVE_A}' no existe en esta base (se omite)")
            continue
        if r.CLAVE_B not in pos_b[r.FUENTE]:
            avisos.append(f"{r.FUENTE}: el registro '{r.CLAVE_B}' no existe en esta base (se omite)")
            continue
        actual = nombre_ok[r.FUENTE][r.CLAVE_B]
        visto = _clave(pd.Series([r.NOMBRE_B])).str.casefold().iat[0]
        if visto and visto != actual:
            errores.append(f"{r.FUENTE} · {r.CLAVE_B}: la decisión dice {visto!r}, el archivo dice {actual!r}")
            continue
        filas.append((int(ent_de_g[pos_g[r.CLAVE_A]]), r.FUENTE, pos_b[r.FUENTE][r.CLAVE_B], r.DECISION, r.AUTOR,
                      r.FECHA, r.RAZON, r.ARCHIVO, int(r.ORDEN), r.CLAVE_A, k))
    if errores:
        raise ValueError(
            f"Qué pasó: {len(errores)} decisiones apuntan a un registro destino con OTRO nombre (primeras):\n  "
            + "\n  ".join(errores[:10]) + "\nPor qué importa: sin identificador nativo, la clave es la fila del Excel; "
            "si la hoja cambió de orden, la decisión aplicaría a otra empresa.\n"
            "Qué hacer: use decisiones hechas sobre ESTOS archivos o corrija CLAVE_B.")
    m = pd.DataFrame(filas, columns=["ENT", "FUENTE_DESTINO", "pos_b", "DECISION", "AUTOR", "FECHA", "RAZON",
                                     "ARCHIVO", "ORDEN", "CLAVE_A_DEC", "_fila"])
    # La última decisión manda (orden de archivo y de fila); los conflictos se informan.
    m = m.sort_values(["ORDEN", "_fila"], kind="stable")
    conflictos = m.groupby(["ENT", "FUENTE_DESTINO", "pos_b"])
    for (e, f, p), grp in conflictos:
        if grp["DECISION"].nunique() > 1:
            ult = grp.iloc[-1]
            avisos.append(f"{f}: «{ult['CLAVE_A_DEC'].split('|', 1)[-1]}» ↔ {prep[f]['ID_REGISTRO'].iat[p]} "
                          f"«{prep[f]['NOMBRE_ORIGINAL'].iat[p]}» tiene decisiones distintas "
                          f"({' → '.join(f'{a}: {d}' for a, d in zip(grp['ARCHIVO'], grp['DECISION']))}); se aplica la última")
    m = m.drop_duplicates(["ENT", "FUENTE_DESTINO", "pos_b"], keep="last").drop(columns="_fila")
    return m.reset_index(drop=True), avisos


def aplicar_decisiones(L: pd.DataFrame, dmap: pd.DataFrame, marca: str = "(verificar)") -> pd.DataFrame:
    """Nivel final, estado, confianza y origen de cada par (idempotente: parte siempre del automático)."""
    base = ["DECISION", "AUTOR", "FECHA", "RAZON", "ARCHIVO", "ORDEN", "CLAVE_A_DEC", "DECISION_VALIDADA"]
    L = L.drop(columns=[c for c in base if c in L.columns])
    x = L.merge(dmap, on=["ENT", "FUENTE_DESTINO", "pos_b"], how="outer", indicator=True)
    x["NIVEL_AUTO"] = x["NIVEL_AUTO"].fillna("")
    x["BANDA_AUTO"] = x["BANDA_AUTO"].fillna("")
    x["DECISION"] = x["DECISION"].fillna("")
    auto_vinc = x["BANDA_AUTO"].isin(["A", "B"]) & x["NIVEL_AUTO"].isin(["EMPRESA", "GRUPO"])
    tiene = x["DECISION"] != ""
    x["AUTOR"] = x["AUTOR"].fillna("") if "AUTOR" in x.columns else ""
    asistida = tiene & x["AUTOR"].astype(str).str.contains(re.escape(marca), regex=True) if marca else tiene & False
    x["DECISION_VALIDADA"] = np.where(tiene, np.where(asistida, "NO (revisión asistida)", "SÍ"), "")
    nivel = np.where(tiene, x["DECISION"], np.where(auto_vinc, x["NIVEL_AUTO"], ""))
    x["NIVEL_FINAL"] = np.where(pd.Series(nivel).isin(["NO_VINCULAR"]), "", nivel)
    vinc = pd.Series(x["NIVEL_FINAL"]).isin(["EMPRESA", "GRUPO"]).to_numpy()
    auto_a_igual = (x["BANDA_AUTO"] == "A") & (x["NIVEL_AUTO"] == pd.Series(x["NIVEL_FINAL"]))
    estado = np.select(
        [tiene & vinc & ~asistida, tiene & vinc & asistida & auto_a_igual, tiene & vinc & asistida,
         tiene & (x["DECISION"] == "NO_VINCULAR"),
         tiene & (x["DECISION"] == "RELACION_PROPIEDAD"),
         ~tiene & (x["BANDA_AUTO"] == "A") & auto_vinc, ~tiene & (x["BANDA_AUTO"] == "B") & auto_vinc],
        ["VINCULADO (revisión)", "VINCULADO (automático A)", "VINCULADO (revisión asistida · validar)", "DESCARTADO (revisión)",
         "RELACIÓN DE PROPIEDAD (revisión)", "VINCULADO (automático A)", "VINCULADO (automático B · revisar)"],
        "CANDIDATO (pendiente de revisión)")
    x["ESTADO"] = estado
    # ALTA: automático A (con o sin confirmación) o decisión validada por una persona.
    # MEDIA: automático B sin revisar, o decisión de revisión asistida aún sin validar.
    x["CONFIANZA"] = np.select([vinc & (auto_a_igual.to_numpy() | (tiene & ~asistida).to_numpy()), vinc],
                               ["ALTA", "MEDIA"], "")
    origen = np.select(
        [tiene & auto_vinc & (x["DECISION"] == x["NIVEL_AUTO"]),
         tiene & auto_vinc & (x["DECISION"] == "NO_VINCULAR"),
         tiene & auto_vinc,
         tiene],
        ["automático, confirmado por revisión", "automático, descartado por revisión",
         "automático, nivel corregido por revisión", "agregado por revisión"],
        np.where(auto_vinc, "automático", "candidato"))
    x["ORIGEN"] = origen
    return x.drop(columns="_merge")


def fusionar_por_empresa_comun(L: pd.DataFrame, ent_de_g: np.ndarray, graf: pd.DataFrame,
                               prep: dict) -> tuple[np.ndarray, list[tuple[int, int, str]]]:
    """Dos grafías que son la MISMA EMPRESA que un mismo registro destino son la misma empresa.

    Solo con vínculos EMPRESA de confianza ALTA (automático A o decidido) y dentro del mismo
    país; nunca une grafías que una decisión DISTINTA separó.
    """
    vetos = graf.attrs.get("vetos", set())
    ok = L[(L["NIVEL_FINAL"] == "EMPRESA") & (L["CONFIANZA"] == "ALTA")]
    iso_ent = pd.Series(graf["PAIS_ISO3"].to_numpy()).groupby(ent_de_g).first().to_dict()
    n = int(ent_de_g.max()) + 1 if len(ent_de_g) else 0
    padre = list(range(n))

    def raiz(v: int) -> int:
        while padre[v] != v:
            padre[v] = padre[padre[v]]
            v = padre[v]
        return v
    miembros = defaultdict(set)
    for gi, e in enumerate(ent_de_g):
        miembros[int(e)].add(gi)
    uniones = []
    for (f, p), grp in ok.groupby(["FUENTE_DESTINO", "pos_b"], sort=True):
        ents = sorted(set(int(e) for e in grp["ENT"]))
        for e2 in ents[1:]:
            a, b = raiz(ents[0]), raiz(e2)
            if a == b or iso_ent.get(ents[0]) != iso_ent.get(e2):
                continue
            cruza = any((min(x, y), max(x, y)) in vetos for x in miembros[a] for y in miembros[b])
            if cruza:
                continue
            padre[max(a, b)] = min(a, b)
            miembros[min(a, b)] |= miembros[max(a, b)]
            uniones.append((ents[0], e2, f"misma empresa que {f} {prep[f]['ID_REGISTRO'].iat[p]}"))
    nuevo = pd.factorize(np.array([raiz(int(e)) for e in ent_de_g]))[0]
    return nuevo, uniones


def propagar_por_web(L: pd.DataFrame, graf: pd.DataFrame, ent_de_g: np.ndarray, prep: dict, destinos: list[str],
                     nz: Normalizador, comp: Comparador) -> pd.DataFrame:
    """Registros destino con el MISMO sitio web que un vínculo EMPRESA se proponen (o suben) como vínculo.

    Evidencia combinada (nombre + web + país + ciudad), del registro nuevo contra la empresa ancla:
      * mismo dominio + comparte palabra distintiva + mismo país + misma ciudad → EMPRESA (B);
      * mismo dominio + palabras alineadas + mismo país → EMPRESA (B);
      * mismo dominio + comparte palabra distintiva → GRUPO (B);
      * mismo dominio sin palabra compartida → GRUPO (C, a revisión).
    Si el par ya existía como candidato de menor nivel, se SUBE; nunca se baja.
    Caso real: ALFRED RITTER GMBH & CO. KG (Orbis, ritter-sport.de, Waldenbuch) → FDI «Ritter Sport»
    (ritter-sport.de, Waldenbuch) = EMPRESA.
    """
    base = L[(L["NIVEL_FINAL"] == "EMPRESA")]
    if base.empty:
        return L
    L = L.copy()
    if "EVIDENCIA_EXTRA" not in L.columns:
        L["EVIDENCIA_EXTRA"] = ""
    L["EVIDENCIA_EXTRA"] = L["EVIDENCIA_EXTRA"].fillna("")
    dom = {f: prep[f]["DOMINIO_WEB"].to_numpy() for f in destinos}
    idx_dom = {f: defaultdict(list) for f in destinos}
    for f in destinos:
        for p, d in enumerate(dom[f]):
            if d:
                idx_dom[f][d].append(p)
    fila_de = {(int(e), f, int(p)): k for k, (e, f, p) in enumerate(zip(L["ENT"], L["FUENTE_DESTINO"], L["pos_b"]))}
    sup_ent = defaultdict(list)
    for gi, e in enumerate(ent_de_g):
        sup_ent[int(e)].append((graf["NOMBRE_NORM"].iat[gi], bool(graf["TRUNCADO"].iat[gi]), graf["PAIS_ISO3"].iat[gi],
                                graf["NOMBRE_ORIGINAL"].iat[gi], graf["CLAVE"].iat[gi]))
    nuevas: dict[tuple, dict] = {}
    for r in base.itertuples(index=False):
        fb, pb = r.FUENTE_DESTINO, int(r.pos_b)
        d = dom[fb][pb]
        if not d:
            continue
        origen_nombre = prep[fb]["NOMBRE_ORIGINAL"].iat[pb]
        ciudad_base = prep[fb]["CIUDAD_NORM"].iat[pb]
        for f in destinos:
            for p in idx_dom[f].get(d, []):
                if f == fb and p == pb:
                    continue
                nb = prep[f]["NOMBRE_NORM"].iat[p]
                evid = set(prep[f]["PAISES_EVIDENCIA"].iat[p].split("|")) - {""}
                misma_ciudad = bool(ciudad_base) and ciudad_base == prep[f]["CIUDAD_NORM"].iat[p]
                mejor = None
                for na, tr, iso, graf_nom, clave in sup_ent[int(r.ENT)]:
                    rs = comp.rasgos(na, nb, tr)
                    mp = (iso in evid) if (iso and evid) else None
                    # Un sobrante genérico en el registro nuevo (USA, DEUTSCHLAND, HOLDING…) delata una filial:
                    # GRUPO aunque comparta web y ciudad (en fDi la ciudad es la de la MATRIZ). Medido: Touton USA.
                    sobra_gen = any(t in nz.genericos for t in rs["SOBRA_B"].split())
                    if rs["N_COMPARTIDAS"] and mp and not sobra_gen and (rs["SIN_SOBRAS"] and rs["MARCAS_OK"] or misma_ciudad):
                        cand = ("EMPRESA", "B")
                    elif rs["N_COMPARTIDAS"]:
                        cand = ("GRUPO", "B")
                    else:
                        cand = ("GRUPO", "C")
                    rango = (ORDEN_NIVEL[cand[0]], ORDEN_BANDA[cand[1]])
                    if mejor is None or rango < mejor[0]:
                        mejor = (rango, cand, rs, graf_nom, na, mp, clave)
                rango, (nivel, banda), rs, graf_nom, na, mp, clave = mejor
                nota = f"mismo sitio web ({d}) que {fb} «{origen_nombre}»" + (" y misma ciudad" if misma_ciudad else "")
                motivo = (f"Mismo sitio web ({d}){' y ciudad' if misma_ciudad else ''} que {fb} «{origen_nombre}», "
                          f"vinculado como EMPRESA" + (f"; comparte {rs['COMPARTIDAS']}" if rs["COMPARTIDAS"]
                                                        else "; no comparte palabras del nombre"))
                k = fila_de.get((int(r.ENT), f, p))
                if k is not None:
                    actual = (ORDEN_NIVEL.get(L["NIVEL_AUTO"].iat[k], 9), ORDEN_BANDA.get(L["BANDA_AUTO"].iat[k], 9))
                    ev = L["EVIDENCIA_EXTRA"].iat[k]
                    if nota not in ev:
                        L.iat[k, L.columns.get_loc("EVIDENCIA_EXTRA")] = "; ".join(x for x in (ev, nota) if x)
                    if rango < actual:
                        L.iat[k, L.columns.get_loc("NIVEL_AUTO")] = nivel
                        L.iat[k, L.columns.get_loc("BANDA_AUTO")] = banda
                        L.iat[k, L.columns.get_loc("MOTIVO")] = motivo + f" (antes: {L['MOTIVO'].iat[k]})"
                    continue
                clave_n = (int(r.ENT), f, p)
                if clave_n in nuevas and (ORDEN_NIVEL[nuevas[clave_n]["NIVEL_AUTO"]], ORDEN_BANDA[nuevas[clave_n]["BANDA_AUTO"]]) <= rango:
                    continue
                punt = comp.puntajes(np.array([na], dtype=object), np.array([nb], dtype=object)).iloc[0]
                nuevas[clave_n] = {
                    "ENT": int(r.ENT), "FUENTE_DESTINO": f, "pos_b": p, "g": -1, "GRAFIA_ANCLA": graf_nom,
                    "CLAVE_GRAFIA": clave, "SUP_A": na, "SUP_B": nb, "TIPO_SUP_A": "NOMBRE", "TIPO_SUP_B": "WEB",
                    "NIVEL_AUTO": nivel, "BANDA_AUTO": banda, "MOTIVO": motivo,
                    "MISMO_PAIS": "SÍ" if mp else ("NO" if mp is False else "SIN DATO"),
                    "COMPARTIDAS": rs["COMPARTIDAS"], "SOBRA_EXPORTADOR": rs["SOBRA_A"], "SOBRA_DESTINO": rs["SOBRA_B"],
                    "EVIDENCIA_EXTRA": nota, **punt.to_dict()}
    if nuevas:
        L = pd.concat([L, pd.DataFrame(list(nuevas.values()))], ignore_index=True)
    return L


# ════════════════════════════════════════════════════════════════════════════
# 11 · MÉTRICAS DE VALIDACIÓN
# ════════════════════════════════════════════════════════════════════════════
def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (round(max(0.0, c - h), 3), round(min(1.0, c + h), 3))


def validar_contra_referencia(L_auto: pd.DataFrame, ref_map: pd.DataFrame, prep: dict, destinos: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Precisión y recall del motor AUTOMÁTICO (sin decisiones) contra una referencia de decisiones.

    Universo: pares del mismo país (la referencia se construyó con esa regla). Un vínculo
    automático que la referencia no confirma cuenta como falso positivo; un vínculo de la
    referencia que el motor no propone (ni como candidato C) cuenta como falso negativo.
    """
    filas, disc = [], []
    for f in destinos:
        a = L_auto[L_auto["FUENTE_DESTINO"] == f]
        pos_auto = set(zip(a.loc[a["BANDA_AUTO"].isin(["A", "B"]) & a["NIVEL_AUTO"].isin(["EMPRESA", "GRUPO"]), "ENT"],
                           a.loc[a["BANDA_AUTO"].isin(["A", "B"]) & a["NIVEL_AUTO"].isin(["EMPRESA", "GRUPO"]), "pos_b"]))
        cand = set(zip(a["ENT"], a["pos_b"]))
        mismo = dict(zip(zip(a["ENT"], a["pos_b"]), a["MISMO_PAIS"]))
        nivel_auto = dict(zip(zip(a["ENT"], a["pos_b"]), a["NIVEL_AUTO"]))
        r = ref_map[ref_map["FUENTE_DESTINO"] == f]
        ref_pos = {(e, p): d for e, p, d in zip(r["ENT"], r["pos_b"], r["DECISION"]) if d in ("EMPRESA", "GRUPO")}
        ref_neg = {(e, p) for e, p, d in zip(r["ENT"], r["pos_b"], r["DECISION"]) if d not in ("EMPRESA", "GRUPO")}
        auto_eval = set(pos_auto)  # la referencia final cubre también los pares entre países
        tp = auto_eval & set(ref_pos)
        fp = auto_eval - set(ref_pos)
        fn = set(ref_pos) - pos_auto
        fn_c = set(ref_pos) - cand
        clase_ok = sum(1 for k in tp if nivel_auto.get(k) == ref_pos[k])
        prec = len(tp) / len(auto_eval) if auto_eval else float("nan")
        rec = len(tp) / len(ref_pos) if ref_pos else float("nan")
        rec_c = (len(ref_pos) - len(fn_c)) / len(ref_pos) if ref_pos else float("nan")
        f1 = 2 * prec * rec / (prec + rec) if prec and rec and not math.isnan(prec + rec) else float("nan")
        ic_p, ic_r = wilson(len(tp), len(auto_eval)), wilson(len(tp), len(ref_pos))
        filas.append({"FUENTE_DESTINO": f, "VINCULOS_AUTOMATICOS_A_B": len(auto_eval),
                      "VINCULOS_REFERENCIA": len(ref_pos), "ACIERTOS": len(tp),
                      "PRECISION": round(prec, 4), "PRECISION_IC95": f"{ic_p[0]:.2f}–{ic_p[1]:.2f}",
                      "RECALL": round(rec, 4), "RECALL_IC95": f"{ic_r[0]:.2f}–{ic_r[1]:.2f}", "F1": round(f1, 4),
                      "RECALL_CONTANDO_CANDIDATOS_C": round(rec_c, 4),
                      "NIVEL_CORRECTO_EN_ACIERTOS": f"{clase_ok}/{len(tp)}",
                      "AUTOMATICOS_ENTRE_PAISES": len({k for k in pos_auto if mismo.get(k) == "NO"}),
                      "DESCARTES_EN_REFERENCIA": len(ref_neg)})
        ids = prep[f]["ID_REGISTRO"].to_numpy()
        noms = prep[f]["NOMBRE_ORIGINAL"].to_numpy()
        for k in sorted(fp):
            disc.append({"FUENTE_DESTINO": f, "TIPO": "Falso positivo (el motor vincula, la referencia no)", "ENT": k[0],
                         "ID_DESTINO": ids[k[1]], "NOMBRE_DESTINO": noms[k[1]], "NIVEL_AUTO": nivel_auto.get(k, ""),
                         "REFERENCIA": "descartado" if k in ref_neg else "no incluido"})
        for k in sorted(fn):
            disc.append({"FUENTE_DESTINO": f, "TIPO": "Falso negativo (la referencia vincula, el motor no)"
                         + (" · el motor lo propuso como candidato C" if k in cand else " · ni siquiera candidato"),
                         "ENT": k[0], "ID_DESTINO": ids[k[1]], "NOMBRE_DESTINO": noms[k[1]],
                         "NIVEL_AUTO": nivel_auto.get(k, ""), "REFERENCIA": ref_pos[k]})
    return pd.DataFrame(filas), pd.DataFrame(disc)


# ════════════════════════════════════════════════════════════════════════════
# 12 · ORQUESTADOR
# ════════════════════════════════════════════════════════════════════════════
@dataclass
class Resultado:
    """Salida completa: tablas del Excel, resumen, validación, invariantes y avisos."""

    cfg: ConfigVinculacion
    tablas: dict
    resumen: pd.DataFrame
    por_pais: pd.DataFrame
    validacion: pd.DataFrame
    invariantes: pd.DataFrame
    calidad: pd.DataFrame
    avisos: list
    parametros: pd.DataFrame
    segundos: float
    internos: dict = field(default_factory=dict)

    @property
    def todo_ok(self) -> bool:
        """True si ninguna invariante CRÍTICA falla (las advertencias no bloquean)."""
        return bool((self.invariantes["ESTADO"] != "FALLA").all()) if len(self.invariantes) else True

    def imprimir(self) -> None:
        print(self.resumen[["SECCION", "INDICADOR", "VALOR"]].to_string(index=False))
        if len(self.validacion):
            print("\nValidación del motor automático contra la referencia:")
            print(self.validacion.to_string(index=False))
        malos = self.invariantes[~self.invariantes["CUMPLE"]]
        print(f"\nInvariantes: {int(self.invariantes['CUMPLE'].sum())}/{len(self.invariantes)} cumplen")
        if len(malos):
            print(malos.to_string(index=False))
        for a in self.avisos[:20]:
            print("  ⚠️ ", a)


def _unir(valores, maximo: int = 12, sep: str = " | ") -> str:
    vistos, out = set(), []
    for v in valores:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            continue
        s = str(v).strip()
        if s and s.lower() not in {"nan", "none"} and s not in vistos:
            vistos.add(s)
            out.append(s)
    if len(out) > maximo:
        return sep.join(out[:maximo]) + f"{sep}… (+{len(out) - maximo})"
    return sep.join(out)


def _representante(graf_e: pd.DataFrame, nz: Normalizador, comp: "Comparador", con_empresa: set) -> int:
    """Grafía que da nombre a la empresa. Orden de preferencia:

    1. la que coincide literalmente con un registro destino (vínculo automático EMPRESA);
    2. la que NO es abreviatura de otra grafía de la misma empresa (CTRA ⊂ CUETARA, SEU ⊂ SEURODIS);
    3. la más central: la más parecida, en promedio, a las demás grafías (comparador de rues-linker);
    4. con forma legal y sin truncar; 5. la más frecuente; 6. orden alfabético.
    """
    idx = list(graf_e.index)
    if len(idx) == 1:
        return int(idx[0])
    noms = graf_e["NOMBRE_NORM"].to_dict()
    fichas = {k: fichas_abreviatura(noms[k], nz) for k in idx}
    abrev = {k: any(es_abreviatura(fa, fb, nz.genericos, True) and not es_abreviatura(fb, fa, nz.genericos, True)
                    for q in idx if q != k for fa in fichas[k] for fb in fichas[q]) or not nz.utiles(noms[k]) for k in idx}
    a = np.array([noms[k] for k in idx for q in idx if q != k], dtype=object)
    b = np.array([noms[q] for k in idx for q in idx if q != k], dtype=object)
    sc = comp.puntajes(a, b)["SCORE_RUES_LINKER"].to_numpy().reshape(len(idx), len(idx) - 1).mean(axis=1)
    orden = sorted(range(len(idx)), key=lambda t: (
        idx[t] not in con_empresa, abrev[idx[t]], -round(float(sc[t]), 2),
        not (graf_e["FORMA_LEGAL"].at[idx[t]] and not graf_e["TRUNCADO"].at[idx[t]]),
        -int(graf_e["N_FILAS"].at[idx[t]]), graf_e["NOMBRE_ORIGINAL"].at[idx[t]]))
    return int(idx[orden[0]])


def ejecutar(cfg: ConfigVinculacion, verboso: bool = True) -> Resultado:
    """Corre el cruce completo y devuelve todas las tablas. No escribe archivos (ver ``exportar_excel``)."""
    t0 = time.time()
    log = print if verboso else (lambda *a, **k: None)
    nz = Normalizador(cfg)
    A, D = cfg.ancla, list(cfg.destinos)
    dcl = [d.clave for d in D]
    fuentes = {f.clave: f for f in [A, *D]}
    crudos = {k: leer_fuente(f) for k, f in fuentes.items()}
    for k, v in crudos.items():
        log(f"📥 {k:<14} {len(v):>6,} filas · {fuentes[k].archivo.split('/')[-1]} [{fuentes[k].hoja}]")
    prep = {k: preparar_fuente(crudos[k], fuentes[k], nz) for k in fuentes}
    dec = leer_decisiones(cfg.archivos_decisiones)
    refs = [cfg.archivo_referencia] if isinstance(cfg.archivo_referencia, str) else list(cfg.archivo_referencia or [])
    ref = leer_decisiones([r for r in refs if r])
    log(f"🧾 decisiones: {len(dec):,} filas de {len(cfg.archivos_decisiones)} archivo(s) · referencia: {len(ref):,} filas")

    corpus = pd.concat([prep[k][c] for k in prep for c in ("NOMBRE_NORM", "ALIAS_NORM", "GRUPO_NORM")]).tolist()
    comp = Comparador(nz, corpus, cfg)

    # ── Deduplicación
    graf, un_a, pend_a = deduplicar_ancla(prep[A.clave], nz, comp, cfg, dec)
    pa = prep[A.clave].assign(CLAVE=clave_ancla(prep[A.clave]["PAIS_ORIGINAL"], prep[A.clave]["NOMBRE_ORIGINAL"]).to_numpy())
    for c in ("ID_COMPARABLE", "DOMINIO_WEB", "DOMINIO_EMAIL", "TEL_DIGITOS", "CIUDAD_NORM"):
        m = pa[pa[c] != ""].groupby("CLAVE")[c].first()
        graf[c] = graf["CLAVE"].map(m).fillna("").to_numpy()
    graf_auto, _, _ = deduplicar_ancla(prep[A.clave], nz, comp, cfg, leer_decisiones([]))
    assert (graf_auto["CLAVE"].to_numpy() == graf["CLAVE"].to_numpy()).all()
    ent_dest, un_dest = {}, {}
    for d in D:
        ent_dest[d.clave], un_dest[d.clave] = deduplicar_destino(prep[d.clave], d, nz, dec)
    log(f"🧩 {A.clave}: {len(prep[A.clave]):,} filas → {len(graf):,} grafías → {graf['_COMP'].nunique():,} empresas (antes de vínculos)")

    # ── Candidatos y clasificación automática
    piezas = []
    for d in D:
        x = generar_candidatos(graf, prep[d.clave], nz, comp, cfg)
        if x.empty:
            continue
        x = clasificar(x, graf, prep[d.clave], nz, comp, cfg)
        x = evidencia_extra(x, graf, prep[d.clave])
        x["FUENTE_DESTINO"] = d.clave
        piezas.append(x)
        log(f"🔗 {d.clave:<14} candidatos clasificados: {len(x):,} "
            f"(A={int((x.BANDA_AUTO == 'A').sum())}, B={int((x.BANDA_AUTO == 'B').sum())}, C={int((x.BANDA_AUTO == 'C').sum())})")
    P = pd.concat(piezas, ignore_index=True) if piezas else pd.DataFrame(
        columns=["g", "pos_b", "FUENTE_DESTINO", "NIVEL_AUTO", "BANDA_AUTO", "SCORE_RUES_LINKER"])
    P["CLAVE_GRAFIA"] = graf["CLAVE"].to_numpy()[P["g"].to_numpy().astype(int)] if len(P) else []

    # ── Motor automático puro (para medir) y resultado con decisiones
    ent_auto = graf_auto["_COMP"].to_numpy()
    sin_dec = _mapear_decisiones(leer_decisiones([]), graf, ent_auto, prep, dcl)[0]
    L_auto = aplicar_decisiones(_agregar_por_entidad(P, ent_auto, graf), sin_dec)
    if cfg.propagar_por_web and len(L_auto):
        L_auto = aplicar_decisiones(propagar_por_web(L_auto, graf, ent_auto, prep, dcl, nz, comp), sin_dec)
    ent0 = graf["_COMP"].to_numpy()
    L0 = _agregar_por_entidad(P, ent0, graf)
    dmap0, _ = _mapear_decisiones(dec, graf, ent0, prep, dcl)
    L0 = aplicar_decisiones(L0, dmap0, cfg.marca_sin_validar)
    ent1, uniones_emp = fusionar_por_empresa_comun(L0, ent0, graf, prep)
    L = _agregar_por_entidad(P, ent1, graf)
    dmap, avisos = _mapear_decisiones(dec, graf, ent1, prep, dcl)
    L = aplicar_decisiones(L, dmap, cfg.marca_sin_validar)
    if cfg.propagar_por_web and len(L):
        n0, ev0 = len(L), int(L.get("EVIDENCIA_EXTRA", pd.Series(dtype=str)).fillna("").str.contains("sitio web").sum())
        L = propagar_por_web(L, graf, ent1, prep, dcl, nz, comp)
        L = aplicar_decisiones(L, dmap, cfg.marca_sin_validar)
        ev1 = int(L["EVIDENCIA_EXTRA"].fillna("").str.contains("sitio web").sum())
        log(f"🌐 propagación por sitio web: {ev1 - ev0} pares con evidencia web ({len(L) - n0} nuevos)")
    avisos += [f"{A.clave}: decisión DUPLICADO con grafía inexistente {a} ↔ {b}" for a, b in graf.attrs.get("desconocidas", [])]

    # ── Validación contra referencia
    val, disc = pd.DataFrame(), pd.DataFrame()
    if len(ref):
        ref_map, av_ref = _mapear_decisiones(ref, graf_auto, ent_auto, prep, dcl)
        avisos += [f"referencia: {a}" for a in av_ref]
        val, disc = validar_contra_referencia(L_auto, ref_map, prep, dcl)

    internos = dict(nz=nz, comp=comp, prep=prep, crudos=crudos, graf=graf, ent=ent1, ent_auto=ent_auto, L=L,
                    L_auto=L_auto, P=P, un_a=un_a, pend_a=pend_a, uniones_emp=uniones_emp, ent_dest=ent_dest,
                    un_dest=un_dest, dec=dec, ref=ref, disc=disc)
    res = construir_tablas(cfg, internos, val, avisos, time.time() - t0)
    log(f"✅ listo en {res.segundos:.1f} s · invariantes {int(res.invariantes['CUMPLE'].sum())}/{len(res.invariantes)}")
    return res


# ════════════════════════════════════════════════════════════════════════════
# 13 · TABLAS DE SALIDA
# ════════════════════════════════════════════════════════════════════════════
def _info_destino(f: Fuente, prep: pd.DataFrame, crudo: pd.DataFrame, ent: pd.DataFrame) -> pd.DataFrame:
    """Una fila por registro destino con lo que se muestra al usuario (en el orden de ``prep``)."""
    out = pd.DataFrame({
        "ID_DESTINO": prep["ID_REGISTRO"].to_numpy(), "FILA_EXCEL": prep["FILA_EXCEL"].to_numpy(),
        "NOMBRE_DESTINO": prep["NOMBRE_ORIGINAL"].to_numpy(), "PAIS_DESTINO": prep["PAIS_ORIGINAL"].to_numpy(),
        "CIUDAD": prep["CIUDAD"].to_numpy(), "WEB": prep["WEB"].to_numpy(), "EMAIL": prep["EMAIL"].to_numpy(),
        "TELEFONO": prep["TELEFONO"].to_numpy(), "DIRECCION": prep["DIRECCION"].to_numpy(),
        "MATRIZ": prep["GRUPO_ORIGINAL"].to_numpy(), "ID_ENTIDAD": ent["ID_ENTIDAD"].to_numpy(),
    })
    for c in f.cols_contexto:
        nombre = f.nombres_contexto.get(c, c)
        if nombre in out.columns:
            nombre = f"{nombre}_{f.clave}"
        v = crudo[c].reset_index(drop=True)
        num = _num(v)
        utiles = v.astype(object).where(v.notna(), None).replace({"n.a.": None, "n.s.": None, "-": None, "": None}).notna()
        out[nombre] = num.round(4).to_numpy() if utiles.any() and num.notna().sum() >= 0.8 * utiles.sum() else \
            v.astype(object).where(v.notna(), "").astype(str).to_numpy()
    return out


def construir_tablas(cfg: ConfigVinculacion, I: dict, val: pd.DataFrame, avisos: list, segundos: float) -> Resultado:
    A, D = cfg.ancla, list(cfg.destinos)
    dcl = [d.clave for d in D]
    etq = {d.clave: d.etiqueta for d in D}
    nz, comp, prep, crudos, graf, ent = I["nz"], I["comp"], I["prep"], I["crudos"], I["graf"], I["ent"]
    L = I["L"].copy()

    # ── Empresas ancla: ID, nombre representativo
    g = graf.copy()
    g["ENT"] = ent
    P = I["P"]
    con_empresa = set(P.loc[(P["NIVEL_AUTO"] == "EMPRESA") & P["BANDA_AUTO"].isin(["A", "B"]), "g"].astype(int)) if len(P) else set()
    rep = {int(e): _representante(sub, nz, comp, con_empresa) for e, sub in g.groupby("ENT", sort=True)}
    orden = sorted(rep, key=lambda e: (g["PAIS_ISO3"].iat[rep[e]], g["NOMBRE_NORM"].iat[rep[e]], g["NOMBRE_ORIGINAL"].iat[rep[e]]))
    id_ent = {e: f"{A.prefijo_id}-{k + 1:04d}" for k, e in enumerate(orden)}
    nombre_ent = {e: g["NOMBRE_ORIGINAL"].iat[rep[e]] for e in rep}
    norm_ent = {e: g["NOMBRE_NORM"].iat[rep[e]] for e in rep}
    pais_ent = {e: g["PAIS_ORIGINAL"].iat[rep[e]] for e in rep}
    iso_ent = {e: g["PAIS_ISO3"].iat[rep[e]] for e in rep}
    ent_de_clave = dict(zip(g["CLAVE"], g["ENT"]))

    pa = prep[A.clave].copy()
    ca = crudos[A.clave].reset_index(drop=True)
    pa["CLAVE"] = clave_ancla(pa["PAIS_ORIGINAL"], pa["NOMBRE_ORIGINAL"]).to_numpy()
    pa["ENT"] = pa["CLAVE"].map(ent_de_clave).to_numpy()
    for c in A.cols_metricas:
        pa[c] = _num(ca[c]).fillna(0).to_numpy()
    for c in A.cols_contexto:
        pa[c] = ca[c].to_numpy()

    # ── Información de destinos y enriquecimiento de los pares
    info = {d.clave: _info_destino(d, prep[d.clave], crudos[d.clave].reset_index(drop=True), I["ent_dest"][d.clave]) for d in D}
    if len(L):
        L["pos_b"] = L["pos_b"].astype(int)
        L["ENT"] = L["ENT"].astype(int)
        for c in ("ID_DESTINO", "NOMBRE_DESTINO", "PAIS_DESTINO", "CIUDAD", "WEB", "MATRIZ", "ID_ENTIDAD"):
            L[c] = [info[f][c].iat[p] for f, p in zip(L["FUENTE_DESTINO"], L["pos_b"])]
        L["ID_EXPORTADOR"] = L["ENT"].map(id_ent)
        L["NOMBRE_EXPORTADOR"] = L["ENT"].map(nombre_ent)
        L["PAIS_EXPORTADOR"] = L["ENT"].map(pais_ent)
        # Pares que solo vienen de una decisión: se puntúan para dejar constancia
        falta = L["SCORE_RUES_LINKER"].isna() if "SCORE_RUES_LINKER" in L.columns else pd.Series(True, index=L.index)
        if falta.any():
            na = L.loc[falta, "ENT"].map(norm_ent).to_numpy()
            nb = np.array([prep[f]["NOMBRE_NORM"].iat[p] for f, p in zip(L.loc[falta, "FUENTE_DESTINO"], L.loc[falta, "pos_b"])], dtype=object)
            pu = comp.puntajes(na, nb)
            for c in pu.columns:
                L.loc[falta, c] = pu[c].to_numpy()
            rs = [comp.rasgos(a, b, False) for a, b in zip(na, nb)]
            L.loc[falta, "COMPARTIDAS"] = [r["COMPARTIDAS"] for r in rs]
            L.loc[falta, "SOBRA_EXPORTADOR"] = [r["SOBRA_A"] for r in rs]
            L.loc[falta, "SOBRA_DESTINO"] = [r["SOBRA_B"] for r in rs]
            L.loc[falta, "SUP_A"], L.loc[falta, "SUP_B"] = na, nb
            L.loc[falta, "MOTIVO"] = "Sin evidencia automática suficiente por nombre; vínculo aportado por revisión"
            evid = [set(prep[f]["PAISES_EVIDENCIA"].iat[p].split("|")) for f, p in zip(L.loc[falta, "FUENTE_DESTINO"], L.loc[falta, "pos_b"])]
            L.loc[falta, "MISMO_PAIS"] = ["SÍ" if iso_ent[e] in ev else "NO" for e, ev in zip(L.loc[falta, "ENT"], evid)]
        L["CLAVE_GRAFIA"] = L["CLAVE_GRAFIA"].where(L["CLAVE_GRAFIA"].notna() & (L["CLAVE_GRAFIA"] != ""),
                                                    L.get("CLAVE_A_DEC")) if "CLAVE_GRAFIA" in L.columns else L.get("CLAVE_A_DEC")
        rep_clave = {e: g["CLAVE"].iat[rep[e]] for e in rep}
        L["CLAVE_GRAFIA"] = [c if isinstance(c, str) and c else rep_clave[e] for c, e in zip(L["CLAVE_GRAFIA"], L["ENT"])]
        L["GRAFIA_ANCLA"] = [x.split("|", 1)[1] for x in L["CLAVE_GRAFIA"]]
        for c in ("MOTIVO", "EVIDENCIA_EXTRA", "COMPARTIDAS", "SOBRA_EXPORTADOR", "SOBRA_DESTINO", "AUTOR", "RAZON",
                  "FECHA", "ARCHIVO", "TIPO_SUP_A", "TIPO_SUP_B"):
            if c not in L.columns:
                L[c] = ""
            L[c] = L[c].fillna("").astype(str)
    vinc = L[L["NIVEL_FINAL"].isin(["EMPRESA", "GRUPO"])].copy() if len(L) else L.copy()
    # Vínculo principal por (empresa, destino)
    if len(vinc):
        vinc["_k1"] = vinc["NIVEL_FINAL"].map({"EMPRESA": 0, "GRUPO": 1})
        vinc["_k2"] = vinc["CONFIANZA"].map({"ALTA": 0, "MEDIA": 1}).fillna(2)
        vinc["_k3"] = vinc["MISMO_PAIS"].map({"SÍ": 0, "SIN DATO": 1, "NO": 2}).fillna(1)
        vinc["_k4"] = vinc["SOBRA_DESTINO"].str.split().str.len().fillna(0)
        vinc["_k5"] = [float(prep[f]["TAMANO"][p]) if "TAMANO" in prep[f] else -1.0 for f, p in zip(vinc["FUENTE_DESTINO"], vinc["pos_b"])]
        vinc = vinc.sort_values(["ENT", "FUENTE_DESTINO", "_k1", "_k2", "_k3", "_k5", "_k4", "SCORE_RUES_LINKER", "ID_DESTINO"],
                                ascending=[True, True, True, True, True, False, True, False, True], kind="stable")
        vinc["ES_PRINCIPAL"] = ~vinc.duplicated(["ENT", "FUENTE_DESTINO"])
        vinc = vinc.drop(columns=["_k1", "_k2", "_k3", "_k4", "_k5"])
    else:
        vinc["ES_PRINCIPAL"] = []
    L["ES_PRINCIPAL"] = False
    if len(vinc):
        principal = set(zip(vinc.loc[vinc["ES_PRINCIPAL"], "ENT"], vinc.loc[vinc["ES_PRINCIPAL"], "FUENTE_DESTINO"],
                            vinc.loc[vinc["ES_PRINCIPAL"], "pos_b"]))
        L["ES_PRINCIPAL"] = [k in principal for k in zip(L["ENT"], L["FUENTE_DESTINO"], L["pos_b"])]

    # Duplicados internos aún sin decidir: se señalan en la fila de cada empresa
    dup_pend = defaultdict(list)
    decididos = set()
    if len(I["dec"]):
        dd = I["dec"][(I["dec"].TIPO == "DUPLICADO") & (I["dec"].FUENTE == A.clave)]
        decididos = {frozenset((a, b)) for a, b in zip(dd.CLAVE_A, dd.CLAVE_B)}
    for r in I["pend_a"].itertuples(index=False):
        ea, eb = int(ent[r.pos_a]), int(ent[r.pos_b])
        if ea != eb and frozenset((g["CLAVE"].iat[r.pos_a], g["CLAVE"].iat[r.pos_b])) not in decididos:
            dup_pend[ea].append(g["NOMBRE_ORIGINAL"].iat[r.pos_b])
            dup_pend[eb].append(g["NOMBRE_ORIGINAL"].iat[r.pos_a])

    marca = cfg.marca_sin_validar
    reglas_union = defaultdict(list)
    for r in I["un_a"].itertuples(index=False):
        reglas_union[int(r.pos_a)].append(r.REGLA)
        reglas_union[int(r.pos_b)].append(r.REGLA)

    # ── EXPORTADORES (hoja principal)
    filas = []
    iso_destinos = {d.clave: set("|".join(prep[d.clave]["PAISES_EVIDENCIA"]).split("|")) - {""} for d in D}
    grp_rows = pa.groupby("ENT", sort=False)
    for e in orden:
        sub = grp_rows.get_group(e)
        gsub = g[g["ENT"] == e]
        fila = {"ID_EXPORTADOR": id_ent[e], "NOMBRE_EXPORTADOR": nombre_ent[e], "PAIS": pais_ent[e]}
        lv = vinc[vinc["ENT"] == e] if len(vinc) else vinc
        le = L[L["ENT"] == e] if len(L) else L
        pend_c = le[le["ESTADO"].str.startswith("CANDIDATO")] if len(le) else le
        pend_b = lv[lv["CONFIANZA"] == "MEDIA"] if len(lv) else lv
        fuentes_v = [etq[f] for f in dcl if len(lv) and (lv["FUENTE_DESTINO"] == f).any()]
        if len(lv):
            resultado = "VINCULADO"
        elif len(pend_c):
            resultado = "CANDIDATO POR REVISAR"
        else:
            resultado = "SIN COINCIDENCIA"
        nivel = "EMPRESA" if len(lv) and (lv["NIVEL_FINAL"] == "EMPRESA").any() else ("GRUPO" if len(lv) else "")
        princ = lv[lv["ES_PRINCIPAL"]] if len(lv) else lv
        confianza = ("ALTA" if len(princ) and (princ["CONFIANZA"] == "ALTA").all() else ("MEDIA" if len(princ) else ""))
        que = []
        if len(pend_b):
            n_asis = int(pend_b["ESTADO"].str.contains("asistida").sum())
            if len(pend_b) - n_asis:
                que.append(f"{len(pend_b) - n_asis} vínculo(s) automático(s) B por confirmar")
            if n_asis:
                que.append(f"{n_asis} vínculo(s) de revisión asistida por validar")
        if len(pend_c):
            que.append(f"{len(pend_c)} candidato(s) C por decidir")
        if any(marca and marca in rg for gi in gsub.index for rg in reglas_union.get(int(gi), [])):
            que.append("validar la unión de sus grafías (revisión asistida)")
        if dup_pend.get(e):
            que.append("¿duplicado de " + " / ".join(f"«{x}»" for x in sorted(set(dup_pend[e]))[:3]) + "?")
        sin_cob = [etq[d] for d in dcl if iso_ent[e] and iso_ent[e] not in iso_destinos[d]]
        obs = []
        if len(sin_cob) == len(dcl) and dcl:
            obs.append("país sin registros en " + " ni ".join(sin_cob))
        if gsub["INTERMEDIARIO"].any():
            obs.append("agente de carga: la empresa exportadora real no está identificada")
        if gsub["TRUNCADO"].any():
            obs.append(f"nombre truncado a {A.largo_truncamiento} caracteres en el origen")
        if (gsub["NOMBRE_NORM"].map(lambda x: not nz.utiles(x))).all():
            obs.append("nombre sin palabras identificables")
        fila.update({"RESULTADO": resultado, "NIVEL_VINCULO": nivel, "CONFIANZA": confianza,
                     "FUENTES_VINCULADAS": " + ".join(fuentes_v),
                     "REQUIERE_REVISION": "SÍ" if que else "NO", "QUE_REVISAR": "; ".join(que),
                     "OBSERVACIONES": "; ".join(obs)})
        for c in A.cols_metricas:
            nombre = A.nombres_contexto.get(c, c).upper().replace(" ", "_")
            if c in A.metricas_repetidas:  # la cifra recomendada va primero
                claves = ["ENT", *[k for k in A.claves_repeticion if k in sub.columns], c]
                fila[f"{nombre}_SIN_REPETIDOS"] = float(sub.drop_duplicates(claves)[c].sum())
            fila[nombre] = float(sub[c].sum())
        fila["N_FILAS"] = len(sub)
        fila["N_GRAFIAS"] = len(gsub)
        fila["GRAFIAS"] = _unir(gsub.sort_values("N_FILAS", ascending=False)["NOMBRE_ORIGINAL"])
        for c in A.cols_contexto:
            fila[A.nombres_contexto.get(c, c).upper().replace(" ", "_")] = _unir(sub[c], maximo=8)
        for d in D:
            pre = d.etiqueta.split()[0].upper()
            ld = lv[lv["FUENTE_DESTINO"] == d.clave] if len(lv) else lv
            p0 = ld[ld["ES_PRINCIPAL"]].iloc[0] if len(ld) else None
            fila[f"{pre}_NIVEL"] = p0["NIVEL_FINAL"] if p0 is not None else ""
            fila[f"{pre}_CONFIANZA"] = p0["CONFIANZA"] if p0 is not None else ""
            fila[f"{pre}_NOMBRE"] = p0["NOMBRE_DESTINO"] if p0 is not None else ""
            fila[f"{pre}_ID"] = p0["ID_DESTINO"] if p0 is not None else ""
            fila[f"{pre}_PAIS"] = p0["PAIS_DESTINO"] if p0 is not None else ""
            fila[f"{pre}_CIUDAD"] = p0["CIUDAD"] if p0 is not None else ""
            fila[f"{pre}_WEB"] = p0["WEB"] if p0 is not None else ""
            fila[f"{pre}_MATRIZ"] = p0["MATRIZ"] if p0 is not None else ""
            inf = info[d.clave]
            for c in d.cols_contexto:
                nom = d.nombres_contexto.get(c, c)
                col = nom if nom in inf.columns else f"{nom}_{d.clave}"
                vacio = np.nan if pd.api.types.is_numeric_dtype(inf[col]) else ""
                fila[f"{pre}_{nom}"] = inf[col].iat[int(p0["pos_b"])] if p0 is not None else vacio
            fila[f"{pre}_N_VINCULOS"] = len(ld)
            otros = ld[~ld["ES_PRINCIPAL"]] if len(ld) else ld
            fila[f"{pre}_OTROS_VINCULOS"] = _unir(otros["NOMBRE_DESTINO"] + " [" + otros["NIVEL_FINAL"] + "]", maximo=10) if len(otros) else ""
        # contacto consolidado (prioriza EMPRESA y el primer destino)
        webs, tels, mails = [], [], []
        if len(lv):
            lv2 = lv[lv["NIVEL_FINAL"] == "EMPRESA"].assign(_o=lambda x: x["FUENTE_DESTINO"].map({f: i for i, f in enumerate(dcl)}))
            for f, p in zip(lv2.sort_values(["_o", "ES_PRINCIPAL"], ascending=[True, False])["FUENTE_DESTINO"],
                            lv2.sort_values(["_o", "ES_PRINCIPAL"], ascending=[True, False])["pos_b"]):
                webs.append(prep[f]["WEB"].iat[p])
                tels.append(prep[f]["TELEFONO"].iat[p])
                mails.append(prep[f]["EMAIL"].iat[p])
        fila["CONTACTO_WEB"] = _unir(webs, 3)
        fila["CONTACTO_TELEFONO"] = _unir(tels, 3)
        fila["CONTACTO_EMAIL"] = _unir(mails, 3)
        fila["NOMBRE_ESTANDARIZADO"] = norm_ent[e]
        fila["_ENT"] = e
        filas.append(fila)
    EXP = pd.DataFrame(filas)
    ops_col = next((c for c in EXP.columns if c.upper().startswith("OPERACIONES")), None)
    orden_res = {"VINCULADO": 0, "CANDIDATO POR REVISAR": 1, "SIN COINCIDENCIA": 2}
    EXP = EXP.sort_values(["RESULTADO", ops_col] if ops_col else ["RESULTADO"],
                          key=lambda s: s.map(orden_res) if s.name == "RESULTADO" else -s, kind="stable").reset_index(drop=True)
    res_de_ent = dict(zip(EXP["_ENT"], EXP["RESULTADO"]))

    # ── ENLACES (vínculos finales y relaciones de propiedad)
    cols_enl = ["ID_EXPORTADOR", "NOMBRE_EXPORTADOR", "PAIS_EXPORTADOR", "FUENTE_DESTINO", "ID_DESTINO", "NOMBRE_DESTINO",
                "PAIS_DESTINO", "NIVEL_FINAL", "CONFIANZA", "ESTADO", "ORIGEN", "ES_PRINCIPAL", "MOTIVO", "EVIDENCIA_EXTRA",
                "MISMO_PAIS", "GRAFIA_ANCLA", "SUP_A", "SUP_B", "COMPARTIDAS", "SOBRA_EXPORTADOR", "SOBRA_DESTINO",
                "SCORE_RUES_LINKER", "SIM_TOKENS_IDF", "JARO_WINKLER", "NIVEL_AUTO", "BANDA_AUTO", "DECISION", "AUTOR",
                "RAZON", "CIUDAD", "WEB", "MATRIZ", "ID_ENTIDAD"]
    ENL = L[L["NIVEL_FINAL"].isin(["EMPRESA", "GRUPO", "RELACION_PROPIEDAD"])].copy() if len(L) else pd.DataFrame(columns=cols_enl)
    if len(ENL):
        ENL["_o"] = ENL["NIVEL_FINAL"].map({"EMPRESA": 0, "GRUPO": 1, "RELACION_PROPIEDAD": 2})
        ENL = ENL.sort_values(["ID_EXPORTADOR", "FUENTE_DESTINO", "ES_PRINCIPAL", "_o", "SCORE_RUES_LINKER"],
                              ascending=[True, True, False, True, False], kind="stable")
    ENL = ENL.reindex(columns=cols_enl).rename(columns={
        "FUENTE_DESTINO": "FUENTE", "NIVEL_FINAL": "NIVEL", "SUP_A": "NOMBRE_COMPARADO_EXPORTADOR",
        "SUP_B": "NOMBRE_COMPARADO_DESTINO", "GRAFIA_ANCLA": "GRAFIA_EXPORTADOR", "AUTOR": "REVISADO_POR",
        "RAZON": "RAZON_REVISION", "DECISION": "DECISION_REVISION", "CIUDAD": "CIUDAD_DESTINO", "WEB": "WEB_DESTINO",
        "MATRIZ": "MATRIZ_DESTINO", "ID_ENTIDAD": "ENTIDAD_DESTINO"})
    ENL["FUENTE"] = ENL["FUENTE"].map(etq).fillna(ENL["FUENTE"])
    ENL["ES_PRINCIPAL"] = np.where(ENL["ES_PRINCIPAL"].astype(bool), "SÍ", "NO")

    # ── REVISION (vínculos B, candidatos C, decididos y duplicados pendientes)
    rv = L[(L["BANDA_AUTO"].isin(["B", "C"])) | (L["DECISION"] != "")].copy() if len(L) else L.copy()
    ops_ent = EXP.set_index("_ENT")[ops_col].to_dict() if ops_col else {}
    filas_rv = []
    marca = cfg.marca_sin_validar
    for r in rv.itertuples(index=False):
        decidido = bool(r.DECISION)
        asistida = decidido and bool(marca) and marca in str(r.AUTOR)
        # Solo pide firma lo que la decisión asistida CAMBIA respecto del automático de alta confianza (A):
        # confirmar un A no agrega riesgo.
        igual_a_auto_a = r.BANDA_AUTO == "A" and r.NIVEL_AUTO == r.DECISION
        cambia = (r.DECISION in ("EMPRESA", "GRUPO", "RELACION_PROPIEDAD") and not igual_a_auto_a) or (
            r.DECISION == "NO_VINCULAR" and r.BANDA_AUTO in ("A", "B"))
        if decidido and asistida and cambia:
            prio, estado_rv = 2, "VALIDAR"
        elif decidido:
            prio, estado_rv = 6, "DECIDIDO"
        elif r.BANDA_AUTO == "B":
            prio, estado_rv = 1, "PENDIENTE"
        elif res_de_ent.get(r.ENT) != "VINCULADO":
            prio, estado_rv = 3, "PENDIENTE"
        else:
            prio, estado_rv = 4, "PENDIENTE"
        sug = f"{r.NIVEL_AUTO} ({r.BANDA_AUTO})" if r.NIVEL_AUTO else "—"
        filas_rv.append({
            "PRIORIDAD": prio, "ESTADO_REVISION": estado_rv,
            "PREGUNTA": (f"¿«{r.NOMBRE_EXPORTADOR}» ({r.PAIS_EXPORTADOR}) es la misma empresa (EMPRESA), del mismo grupo (GRUPO) "
                         f"o no tiene relación (NO_VINCULAR) con {etq[r.FUENTE_DESTINO]} «{r.NOMBRE_DESTINO}» ({r.PAIS_DESTINO})?"),
            "SUGERENCIA_MOTOR": sug, "MOTIVO": r.MOTIVO, "ESTADO_ACTUAL": r.ESTADO,
            "TIPO": "VINCULO", "FUENTE": r.FUENTE_DESTINO, "CLAVE_A": r.CLAVE_GRAFIA, "NOMBRE_A": r.GRAFIA_ANCLA,
            "CLAVE_B": r.ID_DESTINO, "NOMBRE_B": r.NOMBRE_DESTINO, "DECISION": r.DECISION, "AUTOR": r.AUTOR,
            "FECHA": r.FECHA, "RAZON": r.RAZON,
            "ID_EXPORTADOR": r.ID_EXPORTADOR, "OPERACIONES_EXPORTADOR": ops_ent.get(r.ENT, np.nan),
            "PAIS_EXPORTADOR": r.PAIS_EXPORTADOR, "PAIS_DESTINO": r.PAIS_DESTINO, "MISMO_PAIS": r.MISMO_PAIS,
            "CIUDAD_DESTINO": r.CIUDAD, "WEB_DESTINO": r.WEB, "MATRIZ_DESTINO": r.MATRIZ,
            "COMPARTIDAS": r.COMPARTIDAS, "SOBRA_EXPORTADOR": r.SOBRA_EXPORTADOR, "SOBRA_DESTINO": r.SOBRA_DESTINO,
            "SCORE_RUES_LINKER": r.SCORE_RUES_LINKER, "EVIDENCIA_EXTRA": r.EVIDENCIA_EXTRA})
    # duplicados de la ancla pendientes (+ los decididos)
    pe = I["pend_a"]
    dec = I["dec"]
    dup_dec = {}
    if len(dec):
        for r in dec[(dec.TIPO == "DUPLICADO") & (dec.FUENTE == A.clave)].itertuples():
            dup_dec[frozenset((r.CLAVE_A, r.CLAVE_B))] = r
    pos_clave = {k: i for i, k in enumerate(g["CLAVE"])}
    pares_dup = {(int(r.pos_a), int(r.pos_b)): r.MOTIVO for r in pe.itertuples(index=False)}
    for k_, d0 in dup_dec.items():
        ks = sorted(k_)
        if len(ks) == 2 and all(k in pos_clave for k in ks):
            a_, b_ = sorted((pos_clave[ks[0]], pos_clave[ks[1]]))
            pares_dup.setdefault((a_, b_), "decisión de revisión")
    for (pa_, pb_), motivo in sorted(pares_dup.items()):
        r = type("P", (), {"pos_a": pa_, "pos_b": pb_, "MOTIVO": motivo})
        ka, kb = g["CLAVE"].iat[r.pos_a], g["CLAVE"].iat[r.pos_b]
        d0 = dup_dec.get(frozenset((ka, kb)))
        ea, eb = int(ent[r.pos_a]), int(ent[r.pos_b])
        if ea == eb and d0 is None:
            continue  # quedaron unidas por otra vía
        if d0 is None:
            prio, estado_rv = 5, "PENDIENTE"
        elif marca and marca in str(d0.AUTOR) and d0.DECISION == "MISMA_ENTIDAD":
            prio, estado_rv = 2, "VALIDAR"
        else:
            prio, estado_rv = 6, "DECIDIDO"
        filas_rv.append({
            "PRIORIDAD": prio, "ESTADO_REVISION": estado_rv,
            "PREGUNTA": f"¿«{g['NOMBRE_ORIGINAL'].iat[r.pos_a]}» y «{g['NOMBRE_ORIGINAL'].iat[r.pos_b]}» son el MISMO exportador "
                        f"(MISMA_ENTIDAD) o distintos (DISTINTA)?",
            "SUGERENCIA_MOTOR": "¿MISMA_ENTIDAD?", "MOTIVO": r.MOTIVO,
            "ESTADO_ACTUAL": "unidas" if ea == eb else "separadas",
            "TIPO": "DUPLICADO", "FUENTE": A.clave, "CLAVE_A": ka, "NOMBRE_A": g["NOMBRE_ORIGINAL"].iat[r.pos_a],
            "CLAVE_B": kb, "NOMBRE_B": g["NOMBRE_ORIGINAL"].iat[r.pos_b],
            "DECISION": d0.DECISION if d0 is not None else "", "AUTOR": d0.AUTOR if d0 is not None else "",
            "FECHA": d0.FECHA if d0 is not None else "", "RAZON": d0.RAZON if d0 is not None else "",
            "ID_EXPORTADOR": f"{id_ent[ea]} / {id_ent[eb]}" if ea != eb else id_ent[ea],
            "OPERACIONES_EXPORTADOR": np.nan, "PAIS_EXPORTADOR": g["PAIS_ORIGINAL"].iat[r.pos_a]})
    REV = pd.DataFrame(filas_rv)
    if len(REV):
        REV = REV.sort_values(["PRIORIDAD", "OPERACIONES_EXPORTADOR", "ID_EXPORTADOR"], ascending=[True, False, True],
                              kind="stable").reset_index(drop=True)
        REV["PRIORIDAD"] = REV["PRIORIDAD"].map({1: "1 · confirmar vínculo automático B",
                                                 2: "2 · validar decisión de revisión asistida",
                                                 3: "3 · candidato para empresa sin vínculo", 4: "4 · candidato adicional",
                                                 5: "5 · ¿duplicado interno?", 6: "6 · decidido"})

    # ── CONTACTOS
    filas_c = []
    # Solo contactos de la MISMA empresa y del registro principal de su grupo: los de otras filiales
    # (Lidl Polska, Lidl Austria…) no sirven para contactar al exportador.
    vc = vinc[(vinc["NIVEL_FINAL"] == "EMPRESA") | vinc["ES_PRINCIPAL"]] if len(vinc) else vinc
    for r in vc.itertuples(index=False) if len(vc) else []:
        pr = prep[r.FUENTE_DESTINO]
        for tipo, col in (("SITIO_WEB", "WEB"), ("TELEFONO", "TELEFONO"), ("CORREO", "EMAIL"), ("DIRECCION", "DIRECCION")):
            v = str(pr[col].iat[r.pos_b]).strip()
            if v and v.lower() != "nan":
                filas_c.append({"ID_EXPORTADOR": r.ID_EXPORTADOR, "NOMBRE_EXPORTADOR": r.NOMBRE_EXPORTADOR,
                                "FUENTE": etq[r.FUENTE_DESTINO], "ID_DESTINO": r.ID_DESTINO, "NOMBRE_DESTINO": r.NOMBRE_DESTINO,
                                "NIVEL": r.NIVEL_FINAL, "PAIS": r.PAIS_DESTINO, "CIUDAD": r.CIUDAD,
                                "TIPO_CONTACTO": tipo, "VALOR_CONTACTO": v})
    CON = pd.DataFrame(filas_c, columns=["ID_EXPORTADOR", "NOMBRE_EXPORTADOR", "FUENTE", "ID_DESTINO", "NOMBRE_DESTINO",
                                         "NIVEL", "PAIS", "CIUDAD", "TIPO_CONTACTO", "VALOR_CONTACTO"])

    # ── DETALLE de la ancla (todas las filas)
    exp_idx = EXP.set_index("_ENT")
    det_a = ca.copy()
    det_a.insert(0, "ID_EXPORTADOR", pa["ENT"].map(id_ent).to_numpy())
    det_a.insert(1, "NOMBRE_EXPORTADOR", pa["ENT"].map(nombre_ent).to_numpy())
    det_a["NOMBRE_NORMALIZADO"] = pa["NOMBRE_NORM"].to_numpy()
    det_a["TRUNCADO"] = np.where(pa["TRUNCADO"], "SÍ", "NO")
    det_a["AGENTE_DE_CARGA"] = np.where(pa["INTERMEDIARIO"], "SÍ", "NO")
    for c in ("RESULTADO", "NIVEL_VINCULO", "CONFIANZA", "FUENTES_VINCULADAS"):
        det_a[c] = pa["ENT"].map(exp_idx[c]).to_numpy()
    for d in D:
        pre = d.etiqueta.split()[0].upper()
        det_a[f"{pre}_NOMBRE"] = pa["ENT"].map(exp_idx[f"{pre}_NOMBRE"]).to_numpy()
    if A.metricas_repetidas:
        claves = ["ENT", *[k for k in A.claves_repeticion if k in pa.columns]]
        for c in A.metricas_repetidas:
            det_a[f"{c.upper()}_REPETIDO"] = np.where(pa.duplicated(claves + [c], keep="first"), "SÍ", "NO")
    det_a = det_a.drop(columns=["FUENTE"], errors="ignore")

    # ── DETALLE de cada destino
    dets = {}
    for d in D:
        inf = info[d.clave].copy()
        pr = prep[d.clave]
        inf.insert(3, "NOMBRE_NORMALIZADO", pr["NOMBRE_NORM"].to_numpy())
        inf.insert(5, "PAIS_ISO3", pr["PAIS_ISO3"].to_numpy())
        lv = vinc[vinc["FUENTE_DESTINO"] == d.clave] if len(vinc) else vinc
        agg = lv.groupby("pos_b").agg(EXPORTADORES_VINCULADOS=("NOMBRE_EXPORTADOR", lambda s: _unir(s, 6)),
                                      IDS_EXPORTADOR=("ID_EXPORTADOR", lambda s: _unir(s, 6)),
                                      NIVEL=("NIVEL_FINAL", lambda s: "EMPRESA" if (s == "EMPRESA").any() else "GRUPO")) if len(lv) else None
        for c in ("EXPORTADORES_VINCULADOS", "IDS_EXPORTADOR", "NIVEL"):
            inf[c] = inf.index.map(agg[c]).fillna("") if agg is not None else ""
        inf = inf.rename(columns={"NIVEL": "NIVEL_MAS_ALTO"})
        cols = ["ID_DESTINO", "FILA_EXCEL", "NOMBRE_DESTINO", "NOMBRE_NORMALIZADO", "PAIS_DESTINO", "PAIS_ISO3",
                "IDS_EXPORTADOR", "EXPORTADORES_VINCULADOS", "NIVEL_MAS_ALTO", "ID_ENTIDAD"]
        inf = inf[cols + [c for c in inf.columns if c not in cols]]
        dets[d.clave] = inf.rename(columns={"ID_DESTINO": "ID_REGISTRO", "NOMBRE_DESTINO": "NOMBRE", "PAIS_DESTINO": "PAIS"})

    # ── DEDUPLICACIÓN
    reglas_g = defaultdict(list)
    for r in I["un_a"].itertuples(index=False):
        reglas_g[r.pos_a].append(r.REGLA)
        reglas_g[r.pos_b].append(r.REGLA)
    ent_uniones = defaultdict(list)
    for a, b, regla in I["uniones_emp"]:
        ent_uniones[a].append(regla)
        ent_uniones[b].append(regla)
    ent0 = graf["_COMP"].to_numpy()
    filas_d = []
    for e, sub in g.groupby("ENT", sort=False):
        if len(sub) < 2:
            continue
        for gi, rr in sub.iterrows():
            reglas = sorted(set(reglas_g.get(gi, []) + ent_uniones.get(int(ent0[gi]), [])))
            filas_d.append({"FUENTE": A.etiqueta, "ENTIDAD": id_ent[int(e)], "NOMBRE_ENTIDAD": nombre_ent[int(e)],
                            "REGISTRO": rr["CLAVE"], "NOMBRE_REGISTRO": rr["NOMBRE_ORIGINAL"], "N_FILAS": int(rr["N_FILAS"]),
                            "FORMA_LEGAL": rr["FORMA_LEGAL"], "COMO_SE_UNIO": "; ".join(reglas)})
    for d in D:
        ed = I["ent_dest"][d.clave]
        multi = ed[ed["N_EN_ENTIDAD"] > 1]
        reg = defaultdict(list)
        for r in I["un_dest"][d.clave].itertuples(index=False):
            reg[r.pos_a].append(r.REGLA)
            reg[r.pos_b].append(r.REGLA)
        for pos, rr in multi.iterrows():
            filas_d.append({"FUENTE": d.etiqueta, "ENTIDAD": rr["ID_ENTIDAD"], "NOMBRE_ENTIDAD": "",
                            "REGISTRO": rr["ID_REGISTRO"], "NOMBRE_REGISTRO": rr["NOMBRE_ORIGINAL"], "N_FILAS": 1,
                            "FORMA_LEGAL": rr["FORMA_LEGAL"], "COMO_SE_UNIO": "; ".join(sorted(set(reg.get(pos, []))))})
    DED = pd.DataFrame(filas_d, columns=["FUENTE", "ENTIDAD", "NOMBRE_ENTIDAD", "REGISTRO", "NOMBRE_REGISTRO", "N_FILAS",
                                         "FORMA_LEGAL", "COMO_SE_UNIO"])
    if len(DED):
        DED["NOMBRE_ENTIDAD"] = DED.groupby("ENTIDAD")["NOMBRE_REGISTRO"].transform("first").where(
            DED["NOMBRE_ENTIDAD"] == "", DED["NOMBRE_ENTIDAD"])

    # ── Calidad de datos, invariantes, resumen, parámetros
    CAL = calidad_datos(cfg, I, pa, ca, EXP)
    INV = invariantes(cfg, I, pa, EXP, L, vinc, ENL, g, dets)
    RES, PP = resumen(cfg, I, EXP, L, vinc, val, CAL, INV, pa)
    PAR = parametros(cfg, I, segundos, ENL)

    # ── VALIDACIÓN: métricas + discrepancias + comparación entre fuentes de decisión
    disc = I["disc"].copy() if len(I["disc"]) else pd.DataFrame()
    if len(disc):
        ea = I["ent_auto"]
        nom_auto = {}
        for gi, e in enumerate(ea):
            nom_auto.setdefault(int(e), []).append(graf["NOMBRE_ORIGINAL"].iat[gi])
        disc.insert(1, "EXPORTADOR", disc["ENT"].map(lambda e: " | ".join(nom_auto[int(e)])))
        disc = disc.drop(columns=["ENT"])
        disc["FUENTE_DESTINO"] = disc["FUENTE_DESTINO"].map(etq).fillna(disc["FUENTE_DESTINO"])
    comparacion = comparar_fuentes_decision(I["dec"])
    val2 = val.copy()
    if len(val2):
        val2["FUENTE_DESTINO"] = val2["FUENTE_DESTINO"].map(etq).fillna(val2["FUENTE_DESTINO"])

    EXP = EXP.drop(columns=["_ENT"])
    DEC = I["dec"][COLUMNAS_DECISION + ["ARCHIVO"]].rename(columns={"ARCHIVO": "ARCHIVO_ORIGEN"}) if len(I["dec"]) else \
        pd.DataFrame(columns=COLUMNAS_DECISION + ["ARCHIVO_ORIGEN"])
    tablas = {"EXPORTADORES": EXP, "ENLACES": ENL, "REVISION": REV, "DECISIONES": DEC, "CONTACTOS": CON,
              "DETALLE_EXPORTADORES": det_a}
    for d in D:
        tablas[f"DETALLE_{d.clave}"] = dets[d.clave]
    tablas.update({"DEDUPLICACION": DED, "VALIDACION": val2, "VALIDACION_DETALLE": disc, "COMPARACION_REVISIONES": comparacion,
                   "CALIDAD_DATOS": CAL, "CONTROL_QA": INV, "PARAMETROS": PAR})
    I["id_ent"], I["vinc"] = id_ent, vinc
    return Resultado(cfg=cfg, tablas=tablas, resumen=RES, por_pais=PP, validacion=val2, invariantes=INV, calidad=CAL,
                     avisos=avisos, parametros=PAR, segundos=segundos, internos=I)


def comparar_fuentes_decision(dec: pd.DataFrame) -> pd.DataFrame:
    """Pares decididos en más de un archivo: ¿coinciden las revisiones?"""
    if dec.empty or dec["ARCHIVO"].nunique() < 2:
        return pd.DataFrame(columns=["TIPO", "FUENTE", "CLAVE_A", "CLAVE_B", "NOMBRE_B", "DECISIONES", "COINCIDEN"])
    d = dec.copy()
    d["PAR"] = [" ↔ ".join(sorted((a, b))) if t == "DUPLICADO" else f"{a} → {b}" for t, a, b in zip(d.TIPO, d.CLAVE_A, d.CLAVE_B)]
    filas = []
    for (t, f, par), grp in d.groupby(["TIPO", "FUENTE", "PAR"], sort=True):
        if grp["ARCHIVO"].nunique() < 2:
            continue
        decs = [f"{a}: {x}" for a, x in zip(grp["ARCHIVO"], grp["DECISION"])]
        filas.append({"TIPO": t, "FUENTE": f, "CLAVE_A": grp["CLAVE_A"].iat[0], "CLAVE_B": grp["CLAVE_B"].iat[0],
                      "NOMBRE_B": grp["NOMBRE_B"].iat[-1], "DECISIONES": " | ".join(decs),
                      "COINCIDEN": "SÍ" if grp["DECISION"].nunique() == 1 else "NO",
                      "RAZON_ULTIMA": grp["RAZON"].iat[-1]})
    return pd.DataFrame(filas)


def calidad_datos(cfg: ConfigVinculacion, I: dict, pa: pd.DataFrame, ca: pd.DataFrame, EXP: pd.DataFrame) -> pd.DataFrame:
    """Hallazgos sobre los insumos que cambian la lectura del resultado."""
    A, D = cfg.ancla, list(cfg.destinos)
    prep, crudos = I["prep"], I["crudos"]
    out = []

    def add(fuente, tema, hallazgo, filas, impacto, qhacer, ejemplos=""):
        out.append({"FUENTE": fuente, "TEMA": tema, "HALLAZGO": hallazgo, "FILAS_AFECTADAS": filas, "IMPACTO": impacto,
                    "QUE_HACER": qhacer, "EJEMPLOS": ejemplos})
    for c in A.cols_metricas:
        v = pa[c]
        if len(v) and float(v.abs().sum()) == 0:
            add(A.etiqueta, "Métrica vacía", f"'{c}' vale 0 en las {len(v):,} filas.", len(v), "No sirve para priorizar ni ponderar.",
                "Pedir la base con la métrica diligenciada; mientras tanto no usarla.")
    for c in A.metricas_repetidas:
        claves = ["ENT", *[k for k in A.claves_repeticion if k in pa.columns], c]
        rep = pa.duplicated(claves, keep=False) & (pa[c] != 0)
        extra = pa.loc[pa.duplicated(claves, keep="first"), c].sum()
        total = pa[c].sum()
        if rep.any():
            ej = pa.loc[rep].groupby(claves)["NOMBRE_ORIGINAL"].first().head(4).tolist()
            otras = [m for m in A.cols_metricas if m != c]
            con_dif = int(pa.loc[rep].groupby(claves)[otras].transform("nunique").gt(1).any(axis=1).sum()) if otras else 0
            n_grupos = int(pa.loc[rep].groupby(claves).ngroups)
            sin_rep = total - extra
            add(A.etiqueta, "Métrica repetida", f"'{c}' se repite idéntico en {int(rep.sum())} filas ({n_grupos} grupos de misma "
                f"empresa y {'/'.join(A.claves_repeticion)}); en {con_dif} de ellas las demás métricas cambian (p. ej. distintas "
                f"operaciones con los mismos kilos): parece un total de un nivel superior copiado en cada fila.", int(rep.sum()),
                (f"Sumar todas las filas da {total:,.0f}; sin repetidos {sin_rep:,.0f}. Lo repetido es el {extra / total:.1%} del "
                 f"total, es decir, la suma simple infla la cifra en {extra / sin_rep:.1%}.") if total and sin_rep else "",
                f"Usar {c.upper()}_SIN_REPETIDOS para comparar empresas y confirmar con quien generó la base cómo se agregó.",
                "; ".join(map(str, ej)))
    for c in [*A.cols_contexto]:
        vals = ca[c].astype(str).str.strip()
        if len(vals) > 1 and vals.nunique() == 1:
            add(A.etiqueta, "Columna constante", f"'{c}' tiene el mismo valor en todas las filas ({vals.iat[0]!r}).", len(vals),
                "No distingue a ninguna empresa (en EXPORTADORES se oculta).", "Nada; confirmar que el filtro de la base es el esperado.")
    cols_a = [c for c in ca.columns if c not in ("FILA_EXCEL", "ID_REGISTRO", "FUENTE")]
    dup = ca.duplicated(cols_a, keep=False)
    if dup.any():
        extra_ops = {m: float(_num(ca.loc[ca.duplicated(cols_a), m]).sum()) for m in A.cols_metricas}
        add(A.etiqueta, "Filas idénticas", f"{int(dup.sum())} filas son idénticas en todas sus columnas "
            f"({int(ca.duplicated(cols_a).sum())} repetidas).", int(dup.sum()),
            "Pueden ser registros legítimos (sin la columna que los distingue) o duplicados; si son duplicados, inflan "
            + ", ".join(f"{m} en {v:,.0f}" for m, v in extra_ops.items() if v) + ".",
            "Confirmar con la fuente; el cruce no se ve afectado.",
            "; ".join(ca.loc[dup, A.col_nombre].astype(str).drop_duplicates().head(4)))
    if A.largo_truncamiento:
        t = pa["TRUNCADO"]
        add(A.etiqueta, "Nombres truncados", f"{int(t.sum())} filas tienen el nombre cortado a {A.largo_truncamiento} caracteres.",
            int(t.sum()), "El final del nombre (a veces la forma legal o una palabra) se pierde; el motor lo tolera.",
            "Nada: el motor admite que la última palabra sea un prefijo.",
            "; ".join(pa.loc[t, "NOMBRE_ORIGINAL"].drop_duplicates().head(4)))
    inter = pa["INTERMEDIARIO"]
    if inter.any():
        add(A.etiqueta, "Agentes de carga", f"{int(inter.sum())} filas corresponden a agentes de carga o navieras, no al exportador real.",
            int(inter.sum()), "No representan a una empresa exportadora: solo se vinculan como EMPRESA exacta.",
            "Excluirlas de análisis de oferta exportable.", "; ".join(pa.loc[inter, "NOMBRE_ORIGINAL"].drop_duplicates().head(5)))
    sin = pa["SIN_NOMBRE_UTIL"]
    if sin.any():
        add(A.etiqueta, "Nombres sin palabras útiles", f"{int(sin.sum())} filas tienen un nombre que tras quitar la forma legal "
            "queda en letras sueltas (abreviaturas de aduana).", int(sin.sum()),
            "El motor no puede vincularlas por nombre; solo una decisión de revisión puede unirlas a otra grafía (ver DEDUPLICACION).",
            "Validar esas uniones en REVISION o completar el nombre en el origen.",
            "; ".join(pa.loc[sin, "NOMBRE_ORIGINAL"].drop_duplicates().head(6)))
    iso_dest = set()
    for d in D:
        iso_dest |= set("|".join(prep[d.clave]["PAISES_EVIDENCIA"]).split("|")) - {""}
    sin_cob = pa[(pa["PAIS_ISO3"] != "") & ~pa["PAIS_ISO3"].isin(iso_dest)]
    if len(sin_cob) and D:
        paises = sin_cob["PAIS_ORIGINAL"].value_counts()
        add(A.etiqueta, "País sin cobertura", f"{len(sin_cob)} filas son de países que no aparecen en ninguna base destino: "
            + ", ".join(f"{k} ({v})" for k, v in paises.items()) + ".", len(sin_cob),
            "Solo pueden vincularse por nombre a filiales/matrices en otros países (nivel GRUPO).",
            "Descargar Orbis/FDI también para esos países si se quieren cubrir.")
    html = pa["NOMBRE_ORIGINAL"].str.contains(r"&[A-Za-z]+;|&#\d+;", regex=True)
    if html.any():
        add(A.etiqueta, "Entidades HTML", f"{int(html.sum())} nombres traen entidades HTML (&AMP;).", int(html.sum()),
            "La librería las decodifica antes de comparar.", "Nada.", "; ".join(pa.loc[html, "NOMBRE_ORIGINAL"].head(3)))
    for d in D:
        pr, cr = prep[d.clave], crudos[d.clave]
        if d.col_id:
            suc = pr["ID_REGISTRO"].str.contains("-")
            if suc.any():
                add(d.etiqueta, "Identificadores de sucursal", f"{int(suc.sum())} IDs traen sufijo de sucursal (…-1003).", int(suc.sum()),
                    "La sucursal y la casa matriz se tratan como la misma entidad (DEDUPLICACION).", "Nada.",
                    "; ".join(pr.loc[suc, "ID_REGISTRO"].head(4)))
        bloqueadas = [c for c in cr.columns if cr[c].astype(str).str.contains("Your access does not allow", na=False).mean() > 0.9]
        if bloqueadas:
            add(d.etiqueta, "Columnas sin acceso", f"{len(bloqueadas)} columnas vienen con el texto 'Your access does not allow…'.",
                len(cr), "No aportan información.", "Pedir la licencia o excluirlas de la descarga.", "; ".join(bloqueadas[:5]))
        if d.cols_pais_extra and d.col_pais:
            difiere = pr["PAISES_EVIDENCIA"].str.contains(r"\|")
            if difiere.any():
                add(d.etiqueta, "País de consulta ≠ sede", f"{int(difiere.sum())} registros tienen '{d.col_pais}' distinto de "
                    f"{', '.join(d.cols_pais_extra)}.", int(difiere.sum()),
                    f"En fDi Markets '{d.col_pais}' es el país del inversionista consultado, no la sede de cada filial: ambos cuentan como 'mismo país'.",
                    "Nada; leer MISMO_PAIS con esto en mente.", "; ".join(pr.loc[difiere, "NOMBRE_ORIGINAL"].head(4)))
        dn = pr[pr["NOMBRE_NORM"] != ""].duplicated(["NOMBRE_NORM"], keep=False)
        if dn.any():
            add(d.etiqueta, "Nombres repetidos", f"{int(dn.sum())} registros comparten el nombre normalizado con otro.", int(dn.sum()),
                "Pueden ser la misma entidad o filiales homónimas en países distintos (ver DEDUPLICACION).", "Nada.",
                "; ".join(pr.loc[pr['NOMBRE_NORM'] != ''].loc[dn, "NOMBRE_ORIGINAL"].drop_duplicates().head(4)))
    return pd.DataFrame(out)


def invariantes(cfg, I, pa, EXP, L, vinc, ENL, g, dets) -> pd.DataFrame:
    A, D = cfg.ancla, list(cfg.destinos)
    prep = I["prep"]
    out = []

    def add(control, cumple, valor, detalle, critico=True):
        out.append({"CONTROL": control, "CUMPLE": bool(cumple), "VALOR": valor, "DETALLE": detalle,
                    "ESTADO": "OK" if cumple else ("FALLA" if critico else "ADVERTENCIA")})
    n = len(prep[A.clave])
    add("Una fila de DETALLE por fila de entrada de la ancla", pa["ENT"].notna().all() and len(pa) == n, len(pa), f"entrada {n}")
    add("Cada fila de la ancla tiene empresa asignada", pa["ENT"].notna().all(), int(pa["ENT"].notna().sum()), "")
    for c in A.cols_metricas:
        nombre = A.nombres_contexto.get(c, c).upper().replace(" ", "_")
        suma_e, suma_f = float(EXP[nombre].sum()), float(pa[c].sum())
        add(f"'{c}' se conserva al consolidar por empresa", abs(suma_e - suma_f) < 1e-6 * max(1, abs(suma_f)),
            f"{suma_e:,.0f}", f"filas {suma_f:,.0f}")
    paises_por_ent = g.groupby("ENT")["PAIS_ISO3"].nunique()
    add("Cada empresa de la ancla tiene un solo país", (paises_por_ent <= 1).all(), int((paises_por_ent > 1).sum()), "empresas con >1 país")
    ok_ids = all(set(vinc.loc[vinc["FUENTE_DESTINO"] == d.clave, "ID_DESTINO"]) <= set(prep[d.clave]["ID_REGISTRO"]) for d in D) if len(vinc) else True
    add("Todo ID vinculado existe en su base destino", ok_ids, "", "")
    dup = ENL.duplicated(["ID_EXPORTADOR", "FUENTE", "ID_DESTINO"]).sum() if len(ENL) else 0
    add("Sin pares repetidos en ENLACES", dup == 0, int(dup), "")
    if len(L):
        emp_auto = L[(L["NIVEL_AUTO"] == "EMPRESA") & (L["BANDA_AUTO"].isin(["A", "B"])) & (L["DECISION"] == "")]
        malos = emp_auto[(emp_auto["MISMO_PAIS"] == "NO")]
        add("Ningún vínculo EMPRESA automático cruza países", len(malos) == 0, len(malos), "")
        emp_a = L[(L["BANDA_AUTO"] == "A") & (L["TIPO_SUP_B"] != "WEB")]
        sin_alinear = emp_a[(emp_a["SOBRA_EXPORTADOR"] != "") | (emp_a["SOBRA_DESTINO"] != "")]
        add("Todo vínculo automático A tiene todas las palabras alineadas", len(sin_alinear) == 0, len(sin_alinear), "")
        rech = L[(L["DECISION"] == "NO_VINCULAR") & L["NIVEL_FINAL"].isin(["EMPRESA", "GRUPO"])]
        add("Ningún par descartado por revisión aparece vinculado", len(rech) == 0, len(rech), "")
        niveles_ok = set(L["NIVEL_FINAL"].fillna("")) <= {"", "EMPRESA", "GRUPO", "RELACION_PROPIEDAD"}
        bandas_ok = set(L["BANDA_AUTO"].fillna("")) <= {"", "A", "B", "C"}
        add("Niveles y bandas válidos", niveles_ok and bandas_ok, "", "")
    if len(vinc):
        np_ = vinc.groupby(["ENT", "FUENTE_DESTINO"])["ES_PRINCIPAL"].sum()
        add("Exactamente un vínculo principal por empresa y destino", (np_ == 1).all(), int((np_ != 1).sum()), "")
    vetos = g.attrs.get("vetos", set())
    ent = I["ent"]
    viol = [(a, b) for a, b in vetos if ent[a] == ent[b]]
    add("Los pares separados por revisión (DISTINTA) no quedan en la misma empresa", not viol, len(viol),
        "; ".join(f"{g['NOMBRE_ORIGINAL'].iat[a]} ~ {g['NOMBRE_ORIGINAL'].iat[b]}" for a, b in viol[:5]))
    ver = tuple(int(x) for x in re.findall(r"\d+", record_linkage.__version__)[:3])
    add("rues-linker ≥ 0.22.4", ver >= (0, 22, 4), record_linkage.__version__, "")
    for d in D:
        add(f"DETALLE_{d.clave} conserva todas las filas", len(dets[d.clave]) == len(prep[d.clave]), len(dets[d.clave]),
            f"entrada {len(prep[d.clave])}")
    if len(I["ref"]):
        disc = I["disc"]
        ni = int(disc["TIPO"].str.contains("ni siquiera candidato").sum()) if len(disc) else 0
        add("Recall del bloqueo: todo vínculo de la referencia es al menos candidato", ni == 0, ni,
            "los que faltan solo se encuentran con conocimiento externo (ver VALIDACION_DETALLE)", critico=False)
    return pd.DataFrame(out)


def resumen(cfg, I, EXP, L, vinc, val, CAL, INV, pa) -> tuple[pd.DataFrame, pd.DataFrame]:
    A, D = cfg.ancla, list(cfg.destinos)
    prep = I["prep"]
    filas = []

    def add(sec, ind, val_, uni="", lectura=""):
        filas.append({"SECCION": sec, "INDICADOR": ind, "VALOR": val_, "UNIDAD": uni, "LECTURA": lectura})
    n_emp = len(EXP)
    add("Entradas", f"Filas {A.etiqueta}", len(pa), "filas", "DETALLE_EXPORTADORES conserva todas.")
    add("Entradas", f"Empresas {A.etiqueta} (tras deduplicar)", n_emp, "empresas",
        f"{I['graf'].shape[0]} grafías distintas del nombre; ver DEDUPLICACION.")
    for d in D:
        ne = I["ent_dest"][d.clave]["ID_ENTIDAD"].nunique()
        add("Entradas", f"Registros {d.etiqueta}", len(prep[d.clave]), "registros", f"{ne:,} entidades tras deduplicar.")
    vincul = EXP[EXP["RESULTADO"] == "VINCULADO"]
    add("Resultado", "Empresas vinculadas a al menos una base", len(vincul), "empresas",
        f"{len(vincul) / n_emp:.1%} de las empresas." if n_emp else "")
    add("Resultado", "  · con la MISMA EMPRESA en alguna base", int((vincul["NIVEL_VINCULO"] == "EMPRESA").sum()), "empresas",
        "La razón social existe tal cual (salvo forma legal, erratas o truncamiento).")
    add("Resultado", "  · solo con su GRUPO empresarial", int((vincul["NIVEL_VINCULO"] == "GRUPO").sum()), "empresas",
        "Otra sociedad de la misma marca/grupo (filial, matriz o hermana). No atribuir sus cifras a la empresa.")
    for d in D:
        k = int(EXP["FUENTES_VINCULADAS"].str.contains(d.etiqueta, regex=False).sum())
        add("Resultado", f"  · vinculadas a {d.etiqueta}", k, "empresas", "")
    if len(D) > 1:
        k = int(EXP["FUENTES_VINCULADAS"].str.count(r"\+").ge(len(D) - 1).sum())
        add("Resultado", "  · vinculadas a todas las bases", k, "empresas", "")
    ops_col = next((c for c in EXP.columns if c.upper().startswith("OPERACIONES")), None)
    for col in [c for c in EXP.columns if c in {A.nombres_contexto.get(m, m).upper().replace(" ", "_") for m in A.cols_metricas}
                or c.endswith("_SIN_REPETIDOS")]:
        tot = float(EXP[col].sum())
        if tot > 0:
            cub = float(vincul[col].sum())
            add("Resultado", f"{col} de las empresas vinculadas", round(cub / tot, 4), "%",
                f"{cub:,.0f} de {tot:,.0f}.")
    add("Resultado", "Vínculos (pares empresa–registro) EMPRESA", int((vinc["NIVEL_FINAL"] == "EMPRESA").sum()) if len(vinc) else 0, "pares", "ENLACES")
    add("Resultado", "Vínculos (pares empresa–registro) GRUPO", int((vinc["NIVEL_FINAL"] == "GRUPO").sum()) if len(vinc) else 0, "pares", "ENLACES")
    rel = int((L["NIVEL_FINAL"] == "RELACION_PROPIEDAD").sum()) if len(L) else 0
    if rel:
        add("Resultado", "Relaciones de propiedad sin nombre compartido", rel, "pares",
            "Aportadas por revisión (p. ej. Kaufland frente a Lidl). No cuentan como vínculo.")
    add("Confianza", "Vínculos de confianza ALTA", int((vinc["CONFIANZA"] == "ALTA").sum()) if len(vinc) else 0, "pares",
        "Automático A, o decidido por una persona.")
    n_asis = int(vinc["ESTADO"].str.contains("asistida").sum()) if len(vinc) else 0
    add("Confianza", "Vínculos de confianza MEDIA", int((vinc["CONFIANZA"] == "MEDIA").sum()) if len(vinc) else 0,
        "pares", f"{n_asis} decididos por revisión asistida sin validar (REVISION, prioridad 2) y el resto automáticos B (prioridad 1).")
    pend = L[L["ESTADO"].str.startswith("CANDIDATO")] if len(L) else L
    add("Confianza", "Candidatos C pendientes de decisión", len(pend), "pares",
        f"{pend['ENT'].nunique() if len(pend) else 0} empresas; no se vinculan hasta que alguien decida.")
    add("Confianza", "Empresas que requieren revisión", int((EXP["REQUIERE_REVISION"] == "SÍ").sum()), "empresas",
        "Vínculos B, candidatos C o posibles duplicados internos sin decidir (hoja REVISION).")
    dec = I["dec"]
    if len(dec):
        for arch, grp in dec.groupby("ARCHIVO", sort=False):
            add("Revisión", f"Decisiones leídas de {arch}", len(grp), "decisiones", "; ".join(f"{k}: {v}" for k, v in grp["DECISION"].value_counts().items()))
    if len(val):
        for r in val.itertuples(index=False):
            add("Calidad del motor", f"{r.FUENTE_DESTINO}: precisión automática (A+B)", r.PRECISION, "%",
                f"IC95 {r.PRECISION_IC95}; {r.ACIERTOS} de {r.VINCULOS_AUTOMATICOS_A_B} contra la referencia.")
            add("Calidad del motor", f"{r.FUENTE_DESTINO}: recall automático (A+B)", r.RECALL, "%",
                f"IC95 {r.RECALL_IC95}; con candidatos C: {r.RECALL_CONTANDO_CANDIDATOS_C:.0%}.")
    fallas = INV[INV["ESTADO"] == "FALLA"]
    advert = INV[INV["ESTADO"] == "ADVERTENCIA"]
    add("Controles", "Invariantes que cumplen", f"{int(INV['CUMPLE'].sum())} de {len(INV)}", "",
        ("Todo OK" if fallas.empty else "FALLAN: " + "; ".join(fallas["CONTROL"].head(3)))
        + (f" · {len(advert)} advertencia(s) no crítica(s): " + "; ".join(advert["CONTROL"].head(2)) if len(advert) else ""))
    add("Controles", "Hallazgos de calidad de datos", len(CAL), "hallazgos", "Ver CALIDAD_DATOS.")
    RES = pd.DataFrame(filas)
    # Por país
    pp = []
    for pais, sub in EXP.groupby("PAIS", sort=False):
        v = sub[sub["RESULTADO"] == "VINCULADO"]
        fila = {"PAIS": pais, "EMPRESAS": len(sub), "VINCULADAS": len(v),
                "PCT_VINCULADAS": round(len(v) / len(sub), 4) if len(sub) else 0,
                "NIVEL_EMPRESA": int((v["NIVEL_VINCULO"] == "EMPRESA").sum()),
                "SOLO_GRUPO": int((v["NIVEL_VINCULO"] == "GRUPO").sum()),
                "CANDIDATOS_POR_REVISAR": int((sub["RESULTADO"] == "CANDIDATO POR REVISAR").sum())}
        for d in D:
            fila[f"VINCULADAS_{d.clave}"] = int(sub["FUENTES_VINCULADAS"].str.contains(d.etiqueta, regex=False).sum())
        if ops_col:
            tot = float(sub[ops_col].sum())
            fila[f"PCT_{ops_col}_CUBIERTO"] = round(float(v[ops_col].sum()) / tot, 4) if tot else 0
        pp.append(fila)
    PP = pd.DataFrame(pp).sort_values("EMPRESAS", ascending=False).reset_index(drop=True)
    return RES, PP


def _huella(df: pd.DataFrame) -> str:
    if df is None or not len(df):
        return "vacía"
    return hashlib.sha256(pd.util.hash_pandas_object(df.astype(str), index=False).values.tobytes()).hexdigest()[:16]


def parametros(cfg: ConfigVinculacion, I: dict, segundos: float, ENL: pd.DataFrame) -> pd.DataFrame:
    import rapidfuzz
    filas = [("fecha_ejecucion_utc", datetime.now(timezone.utc).isoformat(timespec="seconds")),
             ("motor", f"vinculador_empresas {VERSION_MOTOR}"),
             ("rues_linker", record_linkage.__version__), ("python", platform.python_version()),
             ("pandas", pd.__version__), ("numpy", np.__version__), ("rapidfuzz", rapidfuzz.__version__),
             ("segundos", round(segundos, 1))]
    vistos = set()
    for f in [cfg.ancla, *cfg.destinos]:
        if f.archivo not in vistos:
            vistos.add(f.archivo)
            filas.append((f"sha256 · {Path(f.archivo).name}", sha256_archivo(f.archivo)))
        cols = {k: v for k, v in asdict(f).items() if k.startswith("col") and v}
        filas.append((f"fuente {f.clave}", f"hoja={f.hoja!r} " + json.dumps(cols, ensure_ascii=False)))
    for a in cfg.archivos_decisiones:
        if Path(a).exists():
            filas.append((f"decisiones · {Path(a).name}", sha256_archivo(a)))
    refs = [cfg.archivo_referencia] if isinstance(cfg.archivo_referencia, str) else list(cfg.archivo_referencia or [])
    for a in refs:
        if a and Path(a).exists():
            filas.append((f"referencia · {Path(a).name}", sha256_archivo(a)))
    filas += [("umbral_piso_empresa", cfg.umbral_piso_empresa), ("umbral_dedup", cfg.umbral_dedup),
              ("umbral_parecido", cfg.umbral_parecido), ("max_ediciones_palabra", cfg.max_ediciones_palabra),
              ("propagar_por_web", cfg.propagar_por_web),
              ("comparador rues-linker", "SimilitudNombre(alfa=0,25, prefix_weight=0,10, max_dif_informativa=0, "
                                         "min_informativos=1, fraccion_max_distintivo=0,01)"),
              ("alias_paises", json.dumps(cfg.alias_paises, ensure_ascii=False)),
              ("formas_legales_extra", ", ".join(sorted(cfg.formas_legales_extra))),
              ("genericos_extra", f"{len(cfg.genericos_extra)} términos: " + ", ".join(sorted(cfg.genericos_extra))),
              ("patrones_intermediario", " | ".join(cfg.patrones_intermediario)),
              ("dominios_genericos", ", ".join(sorted(cfg.dominios_genericos))),
              ("nombres_pila", f"{len(cfg.nombres_pila)} nombres"),
              ("huella ENLACES (sha256/16)", _huella(ENL))]
    return pd.DataFrame(filas, columns=["PARAMETRO", "VALOR"]).astype({"VALOR": str})


# ════════════════════════════════════════════════════════════════════════════
# 14 · EXCEL DE ENTREGA
# ════════════════════════════════════════════════════════════════════════════
def _textos_leeme(cfg: ConfigVinculacion) -> tuple[list, list, list]:
    A, D = cfg.ancla, list(cfg.destinos)
    dest = " y ".join(d.etiqueta for d in D) or "(sin destinos)"
    hojas = [
        ("RESUMEN", "Cifras principales, cobertura por país, calidad medida del motor y controles.", "Empiece aquí."),
        ("EXPORTADORES", f"UNA FILA POR EMPRESA de {A.etiqueta} (todas, vinculadas o no): su resultado, nivel y confianza, "
                         f"métricas consolidadas y el mejor registro de {dest} con sus datos.",
         "Filtre RESULTADO = VINCULADO. NIVEL_VINCULO dice si es la misma empresa o su grupo."),
        ("ENLACES", "Todos los vínculos finales, uno por par empresa–registro, con el porqué (MOTIVO) y la evidencia.",
         "ES_PRINCIPAL = SÍ es el que aparece en EXPORTADORES."),
        ("REVISION", "Lo que el motor no decide solo: vínculos B por confirmar, candidatos C y posibles duplicados. "
                     "Las columnas amarillas (DECISION, AUTOR, FECHA, RAZON) son para llenar.",
         "Llene DECISION y vuelva a correr el notebook apuntando a este Excel: la decisión se aplica y queda trazada."),
        ("DECISIONES", "Historial de todas las decisiones humanas aplicadas (archivo, autor, fecha y razón).",
         "No se edita: se conserva para que la próxima corrida reproduzca este resultado."),
        ("CONTACTOS", "Web, teléfono, correo y dirección de cada registro vinculado, sin mezclar fuentes.", "Uso comercial."),
        (f"DETALLE_{A.clave}", f"Cada fila original de {A.etiqueta} con su ID_EXPORTADOR y resultado.", "Para cruzar con otras tablas."),
        *[(f"DETALLE_{d.clave}", f"Cada registro de {d.etiqueta} con los exportadores a los que quedó vinculado.", "")
          for d in D],
        ("DEDUPLICACION", "Registros que son la misma entidad dentro de cada base y cómo se unieron.", ""),
        ("VALIDACION*", "Precisión y recall del motor automático contra una referencia revisada, y cada discrepancia.",
         "Mide qué tan confiable es el automático sin revisión."),
        ("CALIDAD_DATOS", "Problemas de los insumos que cambian la lectura (métricas en cero o repetidas, truncamientos…).", "Léala antes de usar las cifras."),
        ("CONTROL_QA", "Invariantes verificadas en cada corrida (conservación de filas y métricas, coherencia de vínculos).", ""),
        ("PARAMETROS · METODOLOGIA", "Versiones, huellas SHA-256 de los insumos, umbrales, listas y el método paso a paso.", "Trazabilidad."),
    ]
    glosario = [
        ("NIVEL EMPRESA", "El registro destino es la MISMA persona jurídica o empresa: coinciden todas las palabras del nombre "
                          "(salvo forma legal, puntuación, tildes, una errata o el truncamiento) y el país. Una decisión de "
                          "revisión con evidencia (web, aviso legal, descripción) puede afirmarlo aunque el país del registro "
                          "sea el de la matriz (fDi Markets)."),
        ("NIVEL GRUPO", "El registro es OTRA sociedad del mismo grupo o marca (filial, matriz, hermana): comparten la marca y lo que "
                        "sobra es genérico (país, sector, 'holding'), o tienen el mismo nombre en otro país. Úselo para contexto de "
                        "grupo; no atribuya a la empresa las cifras financieras de otra entidad."),
        ("RELACIÓN DE PROPIEDAD", "Vínculo por dueño o matriz SIN nombre compartido (Kaufland frente a Lidl). Solo lo aporta la "
                                  "revisión y no cuenta como vínculo."),
        ("BANDA A", "Automático, alta confianza: mismo nombre normalizado y mismo país."),
        ("BANDA B", "Automático, confianza media: errata, truncamiento, alias entre paréntesis, nombre de una palabra, marca "
                    "compartida con sobrantes genéricos o mismo nombre en otro país. Se vincula, pero conviene confirmarlo."),
        ("BANDA C", "Candidato: comparte una palabra distintiva pero difiere en otras, es una posible sigla o la matriz reportada "
                    "coincide. NO se vincula hasta que alguien decida."),
        ("CONFIANZA ALTA / MEDIA", "ALTA = automático banda A, o decidido por una PERSONA. MEDIA = automático banda B sin "
                                   "revisar, o decidido por una revisión ASISTIDA (AUTOR con la marca '(verificar)') que nadie ha "
                                   "validado todavía. Para validar: ponga su nombre en AUTOR (sin la marca) en la hoja REVISION."),
        ("ESTADO_REVISION", "PENDIENTE = nadie ha decidido. VALIDAR = lo decidió una revisión asistida y falta la firma de una "
                            "persona. DECIDIDO = decisión validada, o descarte de un candidato que no cambia el resultado automático."),
        ("SCORE_RUES_LINKER", "Similitud 0–1 del comparador SimilitudNombre de rues-linker (Jaccard ponderado por IDF + "
                              "Jaro-Winkler). Es evidencia, no la regla: la decisión la toma la alineación palabra a palabra."),
    ]
    metodo = [
        ("1", "Ingesta", "Lee cada hoja declarada, valida que existan las columnas, conserva la fila de Excel y asigna un ID por registro "
                         "(el nativo si existe, p. ej. BvD ID)."),
        ("2", "Normalización (rues-linker)", "sanear_texto → ascii_mayusculas → unir_iniciales → normalizar_nombre: tildes, "
                                             "mayúsculas, entidades HTML, iniciales sueltas y formas legales (GMBH, S.L.U., S.P.A., STIFTUNG & CO. KG…). "
                                             "El nombre entre paréntesis se guarda como alias y la matriz (Parent/GUO) como superficie aparte."),
        ("3", "País (rues-linker)", "canonizar_pais contra el catálogo ISO 3166 de la librería + alias declarados (FRANCE, ITALY…). "
                                    "Una grafía desconocida DETIENE la corrida con la sugerencia. El país es evidencia, no bloqueo."),
        ("4", "Deduplicación interna", f"{A.etiqueta}: mismo nombre normalizado, mismas letras con otros espacios o errata/truncamiento "
                                       "con forma legal compatible; lo dudoso (siglas, contención) va a REVISION. Con ID nativo (Orbis) "
                                       "solo por raíz del ID. Dos grafías que son la MISMA EMPRESA que un mismo registro destino se unen."),
        ("5", "Candidatos", "Comparación exhaustiva ancla × destino (recall del bloqueo = 1): pares que comparten una palabra "
                            "distintiva (con tolerancia de una letra), cadenas con Jaro-Winkler ≥ 0,85 o siglas."),
        ("6", "Clasificación", "Alineación palabra a palabra + SimilitudNombre de rues-linker. EMPRESA: todas las palabras se "
                               "corresponden, mismas iniciales/números, score ≥ piso y mismo país. GRUPO: marca compartida y sobrantes "
                               "genéricos. C: lo demás con evidencia. Nombres de pila solos no cuentan como evidencia."),
        ("7", "Variables adicionales", "Si ancla y destino comparten identificador, web, correo, teléfono o ciudad, se usan como "
                                       "evidencia. Además, un registro con el MISMO SITIO WEB que un vínculo EMPRESA se propone o se sube "
                                       "de nivel (web + ciudad + palabra compartida = EMPRESA)."),
        ("8", "Decisiones de revisión", "Se leen de CSV/Excel (TIPO, FUENTE, CLAVE_A, CLAVE_B, DECISION…) en orden; la última manda. "
                                        "Confirman, cambian el nivel, agregan o descartan, y cada vínculo dice su ORIGEN. Una decisión "
                                        "cuyo registro destino cambió de nombre DETIENE la corrida."),
        ("9", "Consolidación", "Una fila por empresa ancla: métricas sumadas (y sin repetidos cuando la base repite totales), mejor "
                               "vínculo por destino (EMPRESA > GRUPO, confianza, mismo país, menos sobrantes, score) y contactos."),
        ("10", "Validación", "Precisión/recall del motor AUTOMÁTICO (sin decisiones) contra una referencia revisada, con IC95 de Wilson."),
        ("11", "Controles", "Invariantes de conservación y coherencia; huellas SHA-256 de insumos y resultado para saber si algo cambió."),
    ]
    return hojas, glosario, metodo


def _escribir_tabla(ws, df: pd.DataFrame, fila0: int = 1, col0: int = 1):
    for j, c in enumerate(df.columns):
        ws.cell(row=fila0, column=col0 + j, value=str(c))
    vals = df.astype(object).where(df.notna(), None).to_numpy()
    for i in range(len(df)):
        for j in range(len(df.columns)):
            v = vals[i, j]
            if isinstance(v, (np.integer,)):
                v = int(v)
            elif isinstance(v, (np.floating,)):
                v = None if np.isnan(v) else float(v)
            elif isinstance(v, (np.bool_, bool)):
                v = "SÍ" if bool(v) else "NO"
            elif isinstance(v, str):
                v = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", v)
            cel = ws.cell(row=fila0 + 1 + i, column=col0 + j, value=v)
            if isinstance(v, str) and v.startswith("="):
                cel.data_type = "s"  # texto que empieza con '=' no es una fórmula


def exportar_excel(res: Resultado, ruta: str | Path | None = None) -> Path:
    """Escribe el Excel de entrega con formato profesional. Devuelve la ruta."""
    from openpyxl import Workbook
    from openpyxl.comments import Comment
    from openpyxl.formatting.rule import FormulaRule
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter
    from openpyxl.worksheet.datavalidation import DataValidation

    cfg = res.cfg
    if ruta is None:
        Path(cfg.carpeta_salida).mkdir(parents=True, exist_ok=True)
        sello = datetime.now().strftime("_%Y-%m-%d_%H%M") if cfg.versionar_salida else ""
        ruta = Path(cfg.carpeta_salida) / f"{cfg.prefijo_salida}{sello}.xlsx"
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    F = "Arial"
    AZUL, BLANCO = "1F3864", "FFFFFF"
    f_base, f_bold = Font(name=F, size=10), Font(name=F, size=10, bold=True)
    f_head = Font(name=F, size=10, bold=True, color=BLANCO)
    fill_head = PatternFill("solid", fgColor=AZUL)
    fill_input = PatternFill("solid", fgColor="FFF2CC")
    fill_ok, fill_warn, fill_bad = PatternFill("solid", fgColor="E2EFDA"), PatternFill("solid", fgColor="FFF2CC"), PatternFill("solid", fgColor="F8CBAD")
    wb = Workbook()
    wb.remove(wb.active)
    tabs = {"negocio": "548235", "revision": "C65911", "detalle": "2F5597", "auditoria": "7F7F7F"}

    def formatear(ws, df: pd.DataFrame, fila0: int = 1, anchos_max: int = 55, congelar: str | None = "A2", filtro: bool = True,
                  envolver: bool = True):
        ncol = len(df.columns)
        for j in range(1, ncol + 1):
            c = ws.cell(row=fila0, column=j)
            c.font, c.fill = f_head, fill_head
            c.alignment = Alignment(wrap_text=True, vertical="center")
        ws.row_dimensions[fila0].height = 30
        largos = [len(str(c)) for c in df.columns]
        muestra = df.head(300).astype(str)
        for j, c in enumerate(df.columns):
            if len(muestra):
                largos[j] = max(largos[j], int(muestra[c].str.len().quantile(0.9)) if len(muestra) else 0)
        for j, c in enumerate(df.columns, 1):
            letra = get_column_letter(j)
            ancho = min(max(8, largos[j - 1] + 2), anchos_max)
            ws.column_dimensions[letra].width = ancho
            serie = df[c]
            es_num = pd.api.types.is_numeric_dtype(serie) and not pd.api.types.is_bool_dtype(serie)
            fmt = None
            nombre = str(c).upper()
            if es_num:
                if nombre.startswith("PCT") or nombre in ("PRECISION", "RECALL", "F1") or nombre.startswith("RECALL_"):
                    fmt = "0.0%"
                elif any(k in nombre for k in ("SCORE", "SIM_", "JARO")):
                    fmt = "0.00"
                elif "ANIO" in nombre or nombre.startswith("FILA") or nombre.endswith("_ID"):
                    fmt = "0"
                elif serie.dropna().apply(lambda v: float(v).is_integer()).all():
                    fmt = "#,##0"
                else:
                    fmt = "#,##0.00"
            largo = envolver and ancho >= anchos_max
            alin = Alignment(wrap_text=largo, vertical="top")
            for i in range(fila0 + 1, fila0 + 1 + len(df)):
                cel = ws.cell(row=i, column=j)
                cel.font = f_base
                if fmt:
                    cel.number_format = fmt
                cel.alignment = alin
        if congelar:
            ws.freeze_panes = congelar
        if filtro and len(df.columns):
            ws.auto_filter.ref = f"A{fila0}:{get_column_letter(ncol)}{fila0 + max(1, len(df))}"

    def colorear(ws, df, columna, reglas, fila0=1):
        if columna not in df.columns or not len(df):
            return
        j = list(df.columns).index(columna) + 1
        letra = get_column_letter(j)
        rango = f"{letra}{fila0 + 1}:{letra}{fila0 + len(df)}"
        for texto, fill in reglas:
            ws.conditional_formatting.add(rango, FormulaRule(formula=[f'LEFT(${letra}{fila0 + 1},{len(texto)})="{texto}"'], fill=fill))

    def hoja_df(nombre, df, tab, congelar="A2", anchos_max=55, envolver=True):
        ws = wb.create_sheet(nombre[:31])
        ws.sheet_properties.tabColor = tabs[tab]
        if df is None or not len(df.columns):
            ws["A1"] = "(sin datos)"
            ws["A1"].font = f_base
            return ws
        _escribir_tabla(ws, df)
        formatear(ws, df, congelar=congelar, anchos_max=anchos_max, envolver=envolver)
        return ws

    hojas, glosario, metodo = _textos_leeme(cfg)
    T = res.tablas
    A = cfg.ancla

    # ── LEEME
    ws = wb.create_sheet("LEEME")
    ws.sheet_properties.tabColor = tabs["negocio"]
    ws["A1"] = cfg.titulo
    ws["A1"].font = Font(name=F, size=14, bold=True, color=AZUL)
    ws["A2"] = (f"Vinculación de {A.etiqueta} con " + " y ".join(d.etiqueta for d in cfg.destinos)
                + f" · motor vinculador_empresas {VERSION_MOTOR} sobre rues-linker {record_linkage.__version__} · "
                + datetime.now().strftime("%Y-%m-%d %H:%M"))
    ws["A2"].font = Font(name=F, size=10, italic=True)
    fila = 4
    ws.cell(row=fila, column=1, value="Cómo leer este archivo en tres pasos").font = Font(name=F, size=12, bold=True, color=AZUL)
    pasos = ["1. RESUMEN: cuántas empresas quedaron vinculadas, con qué confianza y qué tan bueno es el motor.",
             "2. EXPORTADORES: una fila por empresa; filtre RESULTADO = VINCULADO. Los datos de cada base vienen en columnas con su prefijo.",
             "3. REVISION: llene DECISION (lista desplegable) en las filas PENDIENTE y vuelva a correr el notebook con este archivo "
             "como ARCHIVO_DECISIONES. Nada de lo que está en C se vincula sin esa decisión."]
    for p in pasos:
        fila += 1
        ws.cell(row=fila, column=1, value=p).font = f_base
    fila += 2
    ws.cell(row=fila, column=1, value="Hojas").font = Font(name=F, size=12, bold=True, color=AZUL)
    fila += 1
    df_h = pd.DataFrame(hojas, columns=["HOJA", "QUE_CONTIENE", "COMO_USARLA"])
    _escribir_tabla(ws, df_h, fila0=fila)
    for j in range(1, 4):
        c = ws.cell(row=fila, column=j)
        c.font, c.fill = f_head, fill_head
    for i in range(fila + 1, fila + 1 + len(df_h)):
        for j in range(1, 4):
            ws.cell(row=i, column=j).font = f_bold if j == 1 else f_base
            ws.cell(row=i, column=j).alignment = Alignment(wrap_text=True, vertical="top")
        largo = max(len(str(ws.cell(row=i, column=2).value or "")) / 95, len(str(ws.cell(row=i, column=3).value or "")) / 60)
        ws.row_dimensions[i].height = 14 * (int(largo) + 1) + 2
    fila += len(df_h) + 2
    ws.cell(row=fila, column=1, value="Glosario").font = Font(name=F, size=12, bold=True, color=AZUL)
    fila += 1
    for t, d_ in glosario:
        ws.cell(row=fila, column=1, value=t).font = f_bold
        ws.cell(row=fila, column=1).alignment = Alignment(vertical="top")
        ws.cell(row=fila, column=2, value=d_).font = f_base
        ws.cell(row=fila, column=2).alignment = Alignment(wrap_text=True, vertical="top")
        ws.merge_cells(start_row=fila, start_column=2, end_row=fila, end_column=3)
        ws.row_dimensions[fila].height = 14 * (len(d_) // 150 + 1) + 2
        fila += 1
    fila += 1
    ws.cell(row=fila, column=1, value="Columnas de EXPORTADORES").font = Font(name=F, size=12, bold=True, color=AZUL)
    dic = [("ID_EXPORTADOR · NOMBRE_EXPORTADOR", "La empresa tras unir sus grafías. El nombre es la grafía más representativa "
            "(la que coincide con un registro oficial, o la más completa y frecuente); todas están en GRAFIAS."),
           ("RESULTADO", "VINCULADO · CANDIDATO POR REVISAR (solo candidatos C) · SIN COINCIDENCIA."),
           ("NIVEL_VINCULO · CONFIANZA", "El mejor nivel alcanzado (EMPRESA > GRUPO) y la confianza de sus vínculos principales."),
           ("REQUIERE_REVISION · QUE_REVISAR", "SÍ cuando hay vínculos de confianza MEDIA, candidatos C, duplicados internos por "
            "decidir o una unión de grafías hecha por revisión asistida. QUE_REVISAR dice qué."),
           ("Métricas", "Sumadas por empresa. Si una métrica viene repetida en el origen, la columna *_SIN_REPETIDOS es la "
            "recomendada y la suma simple queda al lado (ver CALIDAD_DATOS). Columnas constantes (p. ej. un valor en 0 para "
            "todos) se ocultan."),
           ]
    for d in cfg.destinos:
        pre = d.etiqueta.split()[0].upper()
        txt = (f"Registro principal de {d.etiqueta} y sus datos. Si {pre}_NIVEL = GRUPO, los datos (ingresos, empleados, "
               f"proyectos, contactos) son de OTRA sociedad del grupo: sirven como contexto, no para atribuirlos al exportador. "
               f"{pre}_N_VINCULOS cuenta todos sus vínculos en esa base; {pre}_OTROS_VINCULOS los lista.")
        if d.cols_pais_extra:
            txt += (f" {pre}_PAIS es '{d.col_pais}' tal como viene; también cuenta como mismo país "
                    f"{', '.join(d.cols_pais_extra)}.")
        dic.append((f"{pre}_…", txt))
    dic.append(("CONTACTO_*", "Web, teléfono y correo de los registros vinculados a nivel EMPRESA (nunca de otras filiales)."))
    for t, d_ in dic:
        fila += 1
        ws.cell(row=fila, column=1, value=t).font = f_bold
        ws.cell(row=fila, column=1).alignment = Alignment(vertical="top", wrap_text=True)
        ws.cell(row=fila, column=2, value=d_).font = f_base
        ws.cell(row=fila, column=2).alignment = Alignment(wrap_text=True, vertical="top")
        ws.merge_cells(start_row=fila, start_column=2, end_row=fila, end_column=3)
        ws.row_dimensions[fila].height = 14 * (len(d_) // 150 + 1) + 2
    fila += 2
    ws.cell(row=fila, column=1, value="Cómo llenar la revisión").font = Font(name=F, size=12, bold=True, color=AZUL)
    ayuda = [("DECISION", "VINCULO: EMPRESA · GRUPO · NO_VINCULAR · RELACION_PROPIEDAD. DUPLICADO: MISMA_ENTIDAD · DISTINTA."),
             ("AUTOR · FECHA", "Quién decidió y cuándo (texto libre; p. ej. 'M. Pérez', '2026-10-02'). Un AUTOR con la marca "
                               "'(verificar)' es una revisión asistida: para VALIDARLA basta con reemplazar el AUTOR por su nombre."),
             ("RAZON", "La evidencia: 'misma web gullon.es y misma ciudad', 'es la matriz según su web', 'homónimo en Málaga'."),
             ("Ejemplo", "TIPO=VINCULO · FUENTE=ORBIS · CLAVE_A=España|GALLETAS GULLON, S.A., · CLAVE_B=ESA34002501 · "
                         "DECISION=EMPRESA · AUTOR=M. Pérez · FECHA=2026-10-02 · RAZON=Mismo nombre y web gullon.es"),
             ("No edite", "TIPO, FUENTE, CLAVE_A, CLAVE_B ni NOMBRE_B: son la llave con la que la decisión se vuelve a aplicar.")]
    for t, d_ in ayuda:
        fila += 1
        ws.cell(row=fila, column=1, value=t).font = f_bold
        ws.cell(row=fila, column=1).alignment = Alignment(vertical="top")
        ws.cell(row=fila, column=2, value=d_).font = f_base
        ws.cell(row=fila, column=2).alignment = Alignment(wrap_text=True, vertical="top")
        ws.merge_cells(start_row=fila, start_column=2, end_row=fila, end_column=3)
        ws.row_dimensions[fila].height = 14 * (len(d_) // 150 + 1) + 2
    ws.column_dimensions["A"].width = 26
    ws.column_dimensions["B"].width = 95
    ws.column_dimensions["C"].width = 60

    # ── RESUMEN (con fórmulas vivas sobre EXPORTADORES donde aplica)
    ws = wb.create_sheet("RESUMEN")
    ws.sheet_properties.tabColor = tabs["negocio"]
    ws["A1"] = "Resumen del cruce"
    ws["A1"].font = Font(name=F, size=13, bold=True, color=AZUL)
    R = res.resumen.copy()
    EXP = T["EXPORTADORES"]
    nexp = len(EXP) + 1
    colL = {c: get_column_letter(i + 1) for i, c in enumerate(EXP.columns)}

    def rng(c):
        return f"EXPORTADORES!${colL[c]}$2:${colL[c]}${nexp}"
    formulas = {}
    for k, r in R.iterrows():
        ind = r["INDICADOR"]
        if ind == "Empresas vinculadas a al menos una base":
            formulas[k] = f'=COUNTIF({rng("RESULTADO")},"VINCULADO")'
        elif ind.startswith(f"Empresas {A.etiqueta}"):
            formulas[k] = f'=COUNTA({rng("ID_EXPORTADOR")})'
        elif ind.strip() == "· con la MISMA EMPRESA en alguna base":
            formulas[k] = f'=COUNTIFS({rng("RESULTADO")},"VINCULADO",{rng("NIVEL_VINCULO")},"EMPRESA")'
        elif ind.strip() == "· solo con su GRUPO empresarial":
            formulas[k] = f'=COUNTIFS({rng("RESULTADO")},"VINCULADO",{rng("NIVEL_VINCULO")},"GRUPO")'
        elif ind.strip().startswith("· vinculadas a ") and "todas" not in ind:
            etiqueta = ind.strip()[len("· vinculadas a "):]
            formulas[k] = f'=COUNTIF({rng("FUENTES_VINCULADAS")},"*{etiqueta}*")'
        elif ind.endswith("de las empresas vinculadas"):
            col = ind.split(" de las empresas vinculadas")[0]
            if col in colL:
                formulas[k] = f'=IFERROR(SUMIF({rng("RESULTADO")},"VINCULADO",{rng(col)})/SUM({rng(col)}),0)'
        elif ind == "Empresas que requieren revisión":
            formulas[k] = f'=COUNTIF({rng("REQUIERE_REVISION")},"SÍ")'
    _escribir_tabla(ws, R, fila0=3)
    for k, f_ in formulas.items():
        ws.cell(row=4 + k, column=3, value=f_)
    formatear(ws, R, fila0=3, congelar=None, filtro=False, anchos_max=90)
    for k, r in R.iterrows():
        c = ws.cell(row=4 + k, column=3)
        c.number_format = "0.0%" if r["UNIDAD"] == "%" else "#,##0"
        c.alignment = Alignment(horizontal="right")
        if not str(r["INDICADOR"]).startswith("  "):
            ws.cell(row=4 + k, column=2).font = f_bold
    ws.column_dimensions["A"].width = 18
    ws.column_dimensions["B"].width = 58
    ws.column_dimensions["C"].width = 14
    ws.column_dimensions["D"].width = 12
    ws.column_dimensions["E"].width = 90
    # Cobertura por país: a la derecha de los indicadores (columna G en adelante)
    c0 = 7
    ws.cell(row=3 - 1, column=c0, value="Cobertura por país").font = Font(name=F, size=12, bold=True, color=AZUL)
    PP = res.por_pais
    _escribir_tabla(ws, PP, fila0=3, col0=c0)
    for j, cname in enumerate(PP.columns):
        c = ws.cell(row=3, column=c0 + j)
        c.font, c.fill = f_head, fill_head
        c.alignment = Alignment(wrap_text=True, vertical="center")
        ws.column_dimensions[get_column_letter(c0 + j)].width = max(12, min(18, len(str(cname)) // 2 + 5))
        for i in range(4, 4 + len(PP)):
            cel = ws.cell(row=i, column=c0 + j)
            cel.font = f_base
            if str(cname).startswith("PCT"):
                cel.number_format = "0.0%"
    ws.column_dimensions[get_column_letter(c0)].width = 16
    ws.column_dimensions["F"].width = 3
    ws.row_dimensions[3].height = 42
    fila = 4 + len(R) + 2
    ws.cell(row=fila, column=1, value="Advertencias de los datos (detalle en CALIDAD_DATOS)").font = Font(name=F, size=12, bold=True, color=AZUL)
    for r in res.calidad.head(8).itertuples(index=False):
        fila += 1
        ws.cell(row=fila, column=1, value=r.FUENTE).font = f_bold
        ws.cell(row=fila, column=2, value=r.TEMA).font = f_bold
        texto = f"{r.HALLAZGO} {r.IMPACTO}"
        ws.cell(row=fila, column=3, value=texto).font = f_base
        ws.cell(row=fila, column=3).alignment = Alignment(wrap_text=True, vertical="top")
        for col in (1, 2):
            ws.cell(row=fila, column=col).alignment = Alignment(vertical="top", wrap_text=True)
        ws.merge_cells(start_row=fila, start_column=3, end_row=fila, end_column=5)
        ws.row_dimensions[fila].height = 14 * (len(texto) // 110 + 1) + 2

    # ── EXPORTADORES
    ws = hoja_df("EXPORTADORES", EXP, "negocio", congelar="C2", anchos_max=45, envolver=False)
    colorear(ws, EXP, "RESULTADO", [("VINCULADO", fill_ok), ("CANDIDATO", fill_warn)])
    colorear(ws, EXP, "CONFIANZA", [("MEDIA", fill_warn)])
    colorear(ws, EXP, "REQUIERE_REVISION", [("SÍ", fill_warn)])
    for c in [c for c in EXP.columns if c.endswith("_WEB") or c == "CONTACTO_WEB"]:
        j = list(EXP.columns).index(c) + 1
        for i in range(2, len(EXP) + 2):
            cel = ws.cell(row=i, column=j)
            v = str(cel.value or "").split(" | ")[0].strip()
            if v and "." in v and " " not in v:
                cel.hyperlink = v if v.startswith("http") else f"https://{v}"
                cel.font = Font(name=F, size=10, color="0563C1", underline="single")
    cols_exp = list(EXP.columns)
    for d in cfg.destinos:
        pre = d.etiqueta.split()[0].upper()
        if f"{pre}_NIVEL" in cols_exp:
            ws.cell(row=1, column=cols_exp.index(f"{pre}_NIVEL") + 1).comment = Comment(
                f"Bloque {pre}_*: registro principal de {d.etiqueta}. Si {pre}_NIVEL = GRUPO, sus datos son de OTRA sociedad "
                f"del grupo (contexto, no atribuibles al exportador).", "motor")
    metr = {A.nombres_contexto.get(m, m).upper().replace(" ", "_") for m in A.cols_metricas} | \
           {A.nombres_contexto.get(m, m).upper().replace(" ", "_") for m in A.cols_contexto}
    for c in cols_exp:
        if c in metr and EXP[c].astype(str).nunique() <= 1 and len(EXP) > 1:
            ws.column_dimensions[get_column_letter(cols_exp.index(c) + 1)].hidden = True
    ws.cell(row=1, column=list(EXP.columns).index("RESULTADO") + 1).comment = Comment(
        "VINCULADO: al menos un vínculo EMPRESA o GRUPO (automático A/B o decidido). CANDIDATO POR REVISAR: solo candidatos C. "
        "SIN COINCIDENCIA: nada con evidencia.", "motor")

    # ── ENLACES
    ws = hoja_df("ENLACES", T["ENLACES"], "negocio", congelar="C2")
    colorear(ws, T["ENLACES"], "CONFIANZA", [("MEDIA", fill_warn), ("ALTA", fill_ok)])

    # ── REVISION
    REV = T["REVISION"]
    ws = hoja_df("REVISION", REV, "revision", congelar="D2", anchos_max=60)
    if len(REV):
        colorear(ws, REV, "ESTADO_REVISION", [("PENDIENTE", fill_warn), ("DECIDIDO", fill_ok)])
        cols = list(REV.columns)
        dv = DataValidation(type="list", formula1='"EMPRESA,GRUPO,NO_VINCULAR,RELACION_PROPIEDAD,MISMA_ENTIDAD,DISTINTA"',
                            allow_blank=True, showErrorMessage=True, errorTitle="Decisión no válida",
                            error="Use EMPRESA, GRUPO, NO_VINCULAR o RELACION_PROPIEDAD (vínculos) o MISMA_ENTIDAD / DISTINTA (duplicados).")
        ws.add_data_validation(dv)
        jd = cols.index("DECISION") + 1
        dv.add(f"{get_column_letter(jd)}2:{get_column_letter(jd)}{len(REV) + 200}")
        for c in ("DECISION", "AUTOR", "FECHA", "RAZON"):
            j = cols.index(c) + 1
            ws.cell(row=1, column=j).fill = PatternFill("solid", fgColor="BF8F00")
            for i in range(2, len(REV) + 2):
                ws.cell(row=i, column=j).fill = fill_input
        ws.cell(row=1, column=jd).comment = Comment("Llene esta columna (lista desplegable). Luego vuelva a correr el notebook "
                                                    "con este Excel en ARCHIVOS_DECISIONES.", "motor")

    ws = hoja_df("DECISIONES", T["DECISIONES"], "revision", anchos_max=70, envolver=False)
    if len(T["DECISIONES"]):
        ws.cell(row=1, column=1).comment = Comment(
            "Historial de TODAS las decisiones aplicadas en esta corrida (de todos los archivos). Al usar este Excel como "
            "archivo de decisiones se leen esta hoja y, después, las decisiones nuevas de REVISION.", "motor")
    hoja_df("CONTACTOS", T["CONTACTOS"], "negocio", envolver=False)
    for k in [k for k in T if k.startswith("DETALLE_")]:
        hoja_df(k, T[k], "detalle", congelar="C2", anchos_max=45, envolver=False)
    hoja_df("DEDUPLICACION", T["DEDUPLICACION"], "detalle", envolver=False)
    # VALIDACION: métricas + detalle + comparación de revisiones en una hoja
    ws = wb.create_sheet("VALIDACION")
    ws.sheet_properties.tabColor = tabs["auditoria"]
    ws["A1"] = "Motor automático (sin decisiones) medido contra la referencia revisada"
    ws["A1"].font = Font(name=F, size=12, bold=True, color=AZUL)
    ws["A2"] = ("Precisión = vínculos automáticos A+B confirmados por la referencia / vínculos automáticos del mismo país. "
                "Recall = vínculos de la referencia que el motor propone solo. IC95 de Wilson. La referencia no es una verdad de "
                "campo: su propio error no está medido.")
    ws["A2"].font = Font(name=F, size=9, italic=True)
    fila = 4
    for titulo, df in (("Métricas", T["VALIDACION"]), ("Discrepancias motor vs referencia", T["VALIDACION_DETALLE"]),
                       ("Pares decididos en más de una revisión", T["COMPARACION_REVISIONES"])):
        ws.cell(row=fila, column=1, value=titulo).font = Font(name=F, size=11, bold=True, color=AZUL)
        fila += 1
        if df is None or not len(df):
            ws.cell(row=fila, column=1, value="(sin datos: no se declaró referencia o no hay pares)").font = f_base
            fila += 2
            continue
        _escribir_tabla(ws, df, fila0=fila)
        for j, c in enumerate(df.columns, 1):
            h = ws.cell(row=fila, column=j)
            h.font, h.fill = f_head, fill_head
            h.alignment = Alignment(wrap_text=True)
            for i in range(fila + 1, fila + 1 + len(df)):
                cel = ws.cell(row=i, column=j)
                cel.font = f_base
                if str(c) in ("PRECISION", "RECALL", "F1", "RECALL_CONTANDO_CANDIDATOS_C"):
                    cel.number_format = "0.0%"
        fila += len(df) + 2
    for j, w in enumerate([16, 60, 40, 16, 40, 16, 16, 16, 16, 16, 16, 16, 16, 16], 1):
        ws.column_dimensions[get_column_letter(j)].width = w
    ws = hoja_df("CALIDAD_DATOS", T["CALIDAD_DATOS"], "auditoria", anchos_max=70)
    ws = hoja_df("CONTROL_QA", T["CONTROL_QA"], "auditoria", anchos_max=70)
    colorear(ws, T["CONTROL_QA"], "ESTADO", [("OK", fill_ok), ("ADVERTENCIA", fill_warn), ("FALLA", fill_bad)])
    hoja_df("PARAMETROS", T["PARAMETROS"], "auditoria", anchos_max=120)
    ws = hoja_df("METODOLOGIA", pd.DataFrame(metodo, columns=["PASO", "TEMA", "DESCRIPCION"]), "auditoria", anchos_max=120)
    for ws in wb.worksheets:
        ws.sheet_view.zoomScale = 90
    wb.calculation.fullCalcOnLoad = True  # Excel recalcula las fórmulas del RESUMEN al abrir
    wb.save(ruta)
    return ruta


In [ ]:
import importlib
import sys

sys.path.insert(0, ".")
import vinculador_empresas as ve  # noqa: E402

importlib.reload(ve)
print(f"✅ motor vinculador_empresas {ve.VERSION_MOTOR}")

## 3 · Configuración — la única celda que se edita
Rutas, bases y el papel de cada columna, listas de limpieza, umbrales y archivos de decisiones. Para **otra base destino** agregue otra `Fuente(...)` a `DESTINOS`; para **otra variable** declare su columna (`col_web`, `col_email`, `col_telefono`, `col_ciudad`, `col_id_comparable`) en la ancla y en el destino.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  3 · CONFIGURACIÓN — LA ÚNICA CELDA QUE SE EDITA                        ║
# ╚══════════════════════════════════════════════════════════════════════╝
from vinculador_empresas import ConfigVinculacion, Fuente

# ── 3.1 · Archivos ───────────────────────────────────────────────────────
CARPETA = "/content/drive/MyDrive/ProColombia/Piloto_agroalimentos"  # @param {type:"string"}
ARCHIVO_SEGMENTACION = f"{CARPETA}/Piloto_segmentacion_Empresas_exportadoras.xlsx"
ARCHIVO_DESCARGAS = f"{CARPETA}/Base_Piloto3_Descargas_Agroalimentos.xlsx"
CARPETA_SALIDA = f"{CARPETA}/resultados"  # @param {type:"string"}

#  Decisiones de revisión, en orden (la última manda). Pueden ser CSV o un Excel de resultados
#  anterior (se leen sus hojas DECISIONES y REVISION). Vacío = solo el motor automático.
ARCHIVOS_DECISIONES = [
    f"{CARPETA}/referencia_revision_2026-09-24.csv",   # revisión asistida del notebook 09 (paneles de jueces)
    f"{CARPETA}/decisiones_revision_2026-09-29.csv",   # revisión asistida de candidatos nuevos y correcciones
]
#  Referencia SOLO para medir al motor automático (no se aplica). Lo usual: las mismas decisiones. [] = no medir.
ARCHIVOS_REFERENCIA = list(ARCHIVOS_DECISIONES)
#  Decisiones cuyo AUTOR trae esta marca son de revisión ASISTIDA (no firmada por una persona): se aplican, pero el
#  vínculo queda con confianza MEDIA y en REVISION como "VALIDAR". "" = confiar en todas las decisiones.
MARCA_SIN_VALIDAR = "(verificar)"  # @param {type:"string"}

# ── 3.2 · Bases: la ancla y los destinos (el contrato de entrada) ─────────
ANCLA = Fuente(
    clave="EXPORTADORES", etiqueta="EXPORTADORES", archivo=ARCHIVO_SEGMENTACION, hoja=0, prefijo_id="EXP",
    col_nombre="Empresa", col_pais="País de Empresa",
    cols_metricas=("Operaciones", "Valor U$S", "Kilos"),
    cols_contexto=("HS6", "Descripción", "Socios Comerciales"),
    nombres_contexto={"Descripción": "DESCRIPCION_PRODUCTO", "Socios Comerciales": "SOCIOS_COMERCIALES", "Valor U$S": "VALOR_USD"},
    transformaciones=(("HS6", "hs6"),),              # 1905.9 → '190590'
    metricas_repetidas=("Kilos",), claves_repeticion=("HS6",),   # Kilos se repite por empresa y partida
    largo_truncamiento=35, detectar_intermediarios=True,          # aduana: nombre de 35 caracteres
)
ORBIS = Fuente(
    clave="ORBIS", etiqueta="ORBIS", archivo=ARCHIVO_DESCARGAS, hoja="Orbis", prefijo_id="ORB",
    col_nombre="Company name Latin alphabet", col_pais="Country", col_id="BvD ID number",
    col_grupo="GUO - Name", col_web="Website address", col_email="E-mail address",
    col_telefono="Telephone number", col_ciudad="City Latin Alphabet", col_direccion="Address line 1 Latin Alphabet",
    col_tamano="Operating revenue (Turnover) m USD Last avail. yr",
    cols_contexto=("Operating revenue (Turnover) m USD Last avail. yr", "Number of employees Last avail. yr",
                   "Last avail. year", "NACE Rev. 2, core code - description", "Number of projects in your search",
                   "Total number of projects", "GUO - Country ISO code", "Quoted", "Inactive"),
    nombres_contexto={"Operating revenue (Turnover) m USD Last avail. yr": "INGRESOS_USD_M",
                      "Number of employees Last avail. yr": "EMPLEADOS", "Last avail. year": "ANIO_DATOS",
                      "NACE Rev. 2, core code - description": "ACTIVIDAD_NACE",
                      "Number of projects in your search": "PROYECTOS_BUSQUEDA", "Total number of projects": "PROYECTOS_TOTAL",
                      "GUO - Country ISO code": "PAIS_MATRIZ_ISO2", "Quoted": "COTIZADA", "Inactive": "INACTIVA"},
)
FDI = Fuente(
    clave="FDI", etiqueta="FDI MARKETS", archivo=ARCHIVO_DESCARGAS, hoja="FDI", prefijo_id="FDI",
    col_nombre="Company name", col_pais="País de consulta", cols_pais_extra=("HQ country",),
    col_grupo="Parent company", col_web="Website", col_email="Company email", col_telefono="Company telephone #",
    col_ciudad="HQ city (Parent)", col_direccion="HQ address", col_tamano="No. of projects (in current search)",
    cols_contexto=("Primary sector", "Primary activity", "No. of projects (in current search)",
                   "Total capex USDm (in current search)", "Total jobs (in current search)", "Date of Last Project",
                   "Revenue (USDm)", "No. Employees (band)", "HQ country", "No of contacts"),
    nombres_contexto={"Primary sector": "SECTOR", "Primary activity": "ACTIVIDAD",
                      "No. of projects (in current search)": "PROYECTOS_BUSQUEDA",
                      "Total capex USDm (in current search)": "CAPEX_USD_M", "Total jobs (in current search)": "EMPLEOS_PROYECTOS",
                      "Date of Last Project": "ULTIMO_PROYECTO", "Revenue (USDm)": "INGRESOS_USD_M",
                      "No. Employees (band)": "EMPLEADOS_RANGO", "HQ country": "PAIS_SEDE", "No of contacts": "CONTACTOS_FDI"},
)
DESTINOS = [ORBIS, FDI]

# ── 3.3 · Países ─────────────────────────────────────────────────────────
#  Grafías que el catálogo de rues-linker 0.22.4 no trae. Si aparece otra, la celda 4 se detiene y la lista.
ALIAS_PAISES = {
    "FRA": ("FRANCE",), "ITA": ("ITALY",), "CHE": ("SWITZERLAND",), "BEL": ("BELGIUM",), "THA": ("THAILAND",),
    "ZAF": ("SOUTH AFRICA",), "LUX": ("LUXEMBOURG",), "POL": ("POLAND",), "SWE": ("SWEDEN",), "NOR": ("NORWAY",),
    "DNK": ("DENMARK",), "FIN": ("FINLAND",), "IRL": ("IRELAND",), "GRC": ("GREECE",), "JPN": ("JAPAN",),
    "KOR": ("SOUTH KOREA",), "CZE": ("CZECH REPUBLIC", "CZECHIA"), "HUN": ("HUNGARY",), "RUS": ("RUSSIA", "RUSSIAN FEDERATION"),
    "ARE": ("UNITED ARAB EMIRATES",), "SAU": ("SAUDI ARABIA",), "EGY": ("EGYPT",), "MAR": ("MOROCCO",),
    "PHL": ("PHILIPPINES",), "NZL": ("NEW ZEALAND",), "HRV": ("CROATIA",), "SVK": ("SLOVAKIA",), "SVN": ("SLOVENIA",),
    "LTU": ("LITHUANIA",), "LVA": ("LATVIA",), "CYP": ("CYPRUS",), "ISL": ("ICELAND",), "ISR": ("ISRAEL",),
}

# ── 3.4 · Listas de limpieza (datos declarados, no inferidos) ─────────────
FORMAS_LEGALES_EXTRA = frozenset({
    "STIFTUNG", "SE", "EG", "KGAA", "MBH", "UG", "OHG", "EV", "AKTIENGESELLSCHAFT", "GESELLSCHAFT MBH",
    "GESELLSCHAFT MIT BESCHRANKTER HAFTUNG", "GESELLSCHAFT MIT BESCHRAENKTER HAFTUNG", "MIT BESCHRAENKTER HAFTUNG",
    "SOCIETA PER AZIONI", "SOCIETA A RESPONSABILITA LIMITATA", "SOCIETA COOPERATIVA", "SOCIETA COOPERATIVA AGRICOLA",
    "IN LIQUIDAZIONE", "SOCIETE PAR ACTIONS SIMPLIFIEE", "SOCIETE ANONYME", "SOCIEDAD LIMITADA UNIPERSONAL",
    "SOCIEDAD LIMITADA", "SOCIEDAD ANONIMA UNIPERSONAL", "UNIPERSONAL", "SAU", "SLU", "SASU", "EURL", "SNC", "SCA",
    "SCARL", "SRLS", "A SOCIO UNICO", "CON SOCIO UNICO", "CON UNICO SOCIO", "SOCIO UNICO", "UNICO SOCIO", "IN SIGLA",
    "SIGLA", "IN FORMA ABBREVIATA", "ABBREVIATA", "S COOP", "SCOOP", "S DE RL DE CV",
})
#  Palabras de sector, geografía y forma de grupo: ruido, nunca identidad. ES LA PALANCA PRINCIPAL DE CALIDAD:
#  una palabra genérica que falte puede unir dos empresas; una marca puesta aquí por error impide enlazar su grupo.
GENERICOS_EXTRA = frozenset("""
SCHOKOLADE SCHOKOLADEN SCHOKOLADENFABRIK SCHOKOLADENMANUFAKTUR SUESSWAREN SUSSWAREN CONFISERIE CHOCOLATERIE LEBENSMITTEL
FEINKOST BACKWAREN KAFFEE HANDEL HANDELS VERTRIEB VERTRIEBS GRUPPE BETEILIGUNGS INDUSTRIE FABRIK MANUFAKTUR DOLCIARIA
DOLCIARIO DOLCIARIE CIOCCOLATO BISCOTTI BISCOTTIFICIO PASTICCERIA ALIMENTARI ALIMENTARE CAFFE GRUPPO PRODOTTI ITALIANA
BISCUITERIE BISCUITS CHOCOLAT CHOCOLATS BOULANGERIE PATISSERIE ALIMENTAIRE ALIMENTAIRES GROUPE COMPAGNIE FRANCAISE
GALLETAS PANADERIA PASTELERIA BOLLERIA CHOCOLATES DULCES CAFES ALIMENTACION ALIMENTARIA IBERICA IBERIA ESPANOLA HARINERA
CONSERVAS EMPRESARIAL SUPERMARKET SUPERMARKETS SUPERMARKETI SUPERMERCADOS SUPERMERCATI SUPERMARCHE HYPERMARCHE MARKT
RETAIL DISCOUNT FOODSERVICE CATERING CHOCOLADEFABRIK CHOCOLADEFABRIKEN CHOCOLADE CHOCOLATE CHOCOLATIER KAKAO CACAO COCOA
BAKERY BAKERIES BACKEREI BAECKEREI PANIFICIO FORNO DOLCI DOLCERIA SWEETS CONFECTIONERY CANDY BISCUIT COOKIES PRODUITS
PRODUKTE FOODSTUFF NAHRUNGSMITTEL ALIMENTS ALIMENTI INGREDIENTS INGREDIENTES PROCESSING IMPEX DISTRIBUCION DISTRIBUZIONE
COMPANIA MANAGEMENT PRODUKTION FACTORY WERK WERKE SAN SANTA GIDA SANAYI TICARET VE DAIRY NUTRITION TECHNOLOGIES CASH
CARRY AGRO AGRAR BRAUEREI CASA NATURAL FRERES MAISON SEAFOOD GOURMET BODEGAS WORLD BIO PLANTA PLANTAS PLANT STABILIMENTO
DR DOTT SPECIALTY SPECIALITY OVERSEAS SOHN SOEHNE HIJOS FIGLI FILS HERMANOS HNOS FRATELLI BRUDER GEBRUDER GEBRUEDER
DEUTSCHLAND OSTERREICH OESTERREICH SCHWEIZ SUISSE POLSKA NEDERLAND BENELUX LIETUVA LATVIJA EESTI MAGYARORSZAG CESKA
CESKO REPUBLIKA SLOVENSKO SLOVENIJA HRVATSKA SRBIJA ROMANIA BULGARIA BELGIQUE BELGIE NEDERLANDEN DANMARK SVERIGE NORGE
SUOMI HELLAS PORTUGAL TURKIYE UKRAINE CYPRUS MALTA IRELAND UK ZENTRALE ZENTRAL CENTRALE INTERNAZIONALE INTERNATIONALE
AUSTRIA BELGIUM SWITZERLAND SVIZZERA CROATIA SLOVENIA SWEDEN CZECH REPUBLIC CZECHIA SLOVAKIA HUNGARY POLAND SERBIA
GREECE NETHERLANDS DENMARK NORWAY FINLAND ESTONIA LATVIA LITHUANIA TURKEY RUS NORDIC NORDICS SCANDINAVIA BALTIC BALTICS
CEE DACH EUROPEAN EURO EUROPEENNE PACIFIC MIDDLE AFRICA LATAM ITALIA ITALY ESPANA SPAIN FRANCE FRANCIA GERMANY ALEMANIA
HOLLAND
""".split())
#  Agentes de carga y navieras (en la ancla): solo se vinculan como EMPRESA exacta.
PATRONES_INTERMEDIARIO = (
    r"\bAS AGENTS? OF\b", r"\bON BEHALF OF\b", r"\bDUMMY\b", r"\bKUEHNE\b", r"\bLOGISTIC", r"\bSHIPPING\b",
    r"\bSPED\b", r"\bSPEDITION", r"\bFREIGHT\b", r"\bCARGO\b", r"\bINTERFRACHT\b", r"\bTRANSPORT", r"\bFORWARD",
    r"\bHILLEBRAND\b", r"\bSAVINO DEL BENE\b", r"\bCLASQUIN\b", r"\bINTERGLOBO\b", r"\bTRANSCOMA\b", r"\bFLETAMENTOS\b",
    r"\bTRANSPED\b", r"\bEGETRANS\b",
)
#  Dominios que no identifican a nadie (no se usan como evidencia).
DOMINIOS_GENERICOS = frozenset("""
gmail.com hotmail.com yahoo.com outlook.com live.com icloud.com aol.com gmx.de web.de t-online.de orange.fr wanadoo.fr
free.fr libero.it tiscali.it telefonica.net linkedin.com facebook.com instagram.com google.com wixsite.com wordpress.com
blogspot.com europages.com bing.com yahoo.es yahoo.fr hotmail.es hotmail.fr
""".split())

# ── 3.5 · Umbrales (medidos en el piloto; cámbielos solo midiendo en la hoja VALIDACION) ──
UMBRAL_PISO_EMPRESA = 0.70   # @param {type:"number"}  score mínimo de rues-linker con todas las palabras alineadas
UMBRAL_DEDUP = 0.84          # @param {type:"number"}  para unir dos grafías del exportador sin revisión
UMBRAL_PARECIDO = 0.88       # @param {type:"number"}  para proponer a revisión un par sin palabra compartida
PROPAGAR_POR_WEB = True      # @param {type:"boolean"}

cfg = ConfigVinculacion(
    ancla=ANCLA, destinos=DESTINOS, alias_paises=ALIAS_PAISES,
    formas_legales_extra=FORMAS_LEGALES_EXTRA, genericos_extra=GENERICOS_EXTRA,
    patrones_intermediario=PATRONES_INTERMEDIARIO, dominios_genericos=DOMINIOS_GENERICOS,
    umbral_piso_empresa=UMBRAL_PISO_EMPRESA, umbral_dedup=UMBRAL_DEDUP, umbral_parecido=UMBRAL_PARECIDO,
    propagar_por_web=PROPAGAR_POR_WEB,
    archivos_decisiones=tuple(ARCHIVOS_DECISIONES), archivo_referencia=tuple(ARCHIVOS_REFERENCIA),
    marca_sin_validar=MARCA_SIN_VALIDAR,
    carpeta_salida=CARPETA_SALIDA, prefijo_salida="VINCULACION_EXPORTADORES_ORBIS_FDI", versionar_salida=True,
    titulo="Piloto 3 agroalimentos · exportadores × Orbis × FDI Markets",
)
print(f"📁 {CARPETA}\n🔗 {ANCLA.clave} × {', '.join(d.clave for d in DESTINOS)}\n🧾 decisiones: {len(ARCHIVOS_DECISIONES)} archivo(s)")

## 4 · Preflight
Verifica que los archivos existan, que cada hoja tenga las columnas declaradas y que todos los países estén en el catálogo. Cuesta segundos y evita correr el cruce con un insumo roto.

In [ ]:
from pathlib import Path

import pandas as pd

faltan = [a for a in {f.archivo for f in [cfg.ancla, *cfg.destinos]} if not Path(a).exists()]
if faltan:
    raise FileNotFoundError("Qué pasó: no existen " + ", ".join(faltan) + "\nQué hacer: corrija CARPETA o los nombres en la celda 3.")
for a in cfg.archivos_decisiones:
    print(("✅ " if Path(a).exists() else "⚠️  NO EXISTE (se omitirá): ") + a)
nz = ve.Normalizador(cfg)
resumen = []
for f in [cfg.ancla, *cfg.destinos]:
    df = ve.leer_fuente(f)                      # verifica columnas e identificador único
    prep = ve.preparar_fuente(df, f, nz)        # verifica países contra el catálogo (se detiene si falta alguno)
    resumen.append({"fuente": f.clave, "hoja": f.hoja, "filas": len(df), "nombres distintos": prep["NOMBRE_NORM"].nunique(),
                    "países": ", ".join(prep["PAIS_ISO3"].value_counts().index[:8]),
                    "con web": int((prep["DOMINIO_WEB"] != "").sum())})
display(pd.DataFrame(resumen))
print("✅ preflight OK")

## 5 · Ejecutar el cruce
Deduplica la ancla, compara todos contra todos, clasifica, aplica las decisiones, propaga por web, consolida y verifica invariantes.

In [ ]:
res = ve.ejecutar(cfg)
res.imprimir()
if not res.todo_ok:
    raise RuntimeError("Qué pasó: falla una invariante crítica (ver arriba y la hoja CONTROL_QA).\n"
                       "Qué hacer: no use el resultado; revise el aviso y corrija la configuración o las decisiones.")

## 6 · Vista rápida

In [ ]:
E = res.tablas["EXPORTADORES"]
print(E["RESULTADO"].value_counts().to_string(), "\n")
cols = ["ID_EXPORTADOR", "NOMBRE_EXPORTADOR", "PAIS", "NIVEL_VINCULO", "CONFIANZA", "FUENTES_VINCULADAS", "OPERACIONES",
        "ORBIS_NOMBRE", "ORBIS_NIVEL", "FDI_NOMBRE", "FDI_NIVEL"]
display(E.loc[E["RESULTADO"] == "VINCULADO", [c for c in cols if c in E.columns]])
display(res.por_pais)
R = res.tablas["REVISION"]
pend = R[R["ESTADO_REVISION"] == "PENDIENTE"] if len(R) else R
print(f"🟡 pendientes de revisión: {len(pend)}")
if len(pend):
    display(pend[["PRIORIDAD", "TIPO", "FUENTE", "NOMBRE_A", "NOMBRE_B", "SUGERENCIA_MOTOR", "MOTIVO"]].head(30))

## 7 · Exportar el Excel
Se escribe primero en `/content` (Drive/FUSE no garantiza escrituras atómicas) y luego se copia a `CARPETA_SALIDA`.

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

nombre = f"{cfg.prefijo_salida}_{datetime.now():%Y-%m-%d_%H%M}.xlsx"
local = (Path("/content/salida") if Path("/content").exists() else Path("salida")) / nombre
ve.exportar_excel(res, local)
destino = Path(cfg.carpeta_salida)
destino.mkdir(parents=True, exist_ok=True)
final = destino / nombre
if local.resolve() != final.resolve():
    shutil.copyfile(local, final)
print(f"✅ {final}  ({final.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files  # type: ignore
    files.download(str(final))
except Exception:
    pass

## 8 · Revisar y volver a correr

1. Abra el Excel, hoja **REVISION**, filtre `ESTADO_REVISION = PENDIENTE`.
2. Llene **DECISION** (lista desplegable): `EMPRESA`, `GRUPO`, `NO_VINCULAR` o `RELACION_PROPIEDAD` en vínculos; `MISMA_ENTIDAD` o `DISTINTA` en duplicados. Complete **AUTOR**, **FECHA** y **RAZON** (la evidencia).
3. Para **validar** una decisión asistida (`ESTADO_REVISION = VALIDAR`), reemplace el AUTOR por su nombre (sin la marca `(verificar)`) o corrija la DECISION si no está de acuerdo.
4. Guarde el Excel en Drive y agréguelo **al final** de `ARCHIVOS_DECISIONES` (celda 3). El motor lee su historial (`DECISIONES`) y lo nuevo de `REVISION`.
5. Corra de nuevo desde la celda 3. `ORIGEN` en `ENLACES` dirá qué confirmó, agregó o descartó la revisión; `CONFIANZA` pasa a ALTA en lo validado.

> No edite `TIPO`, `FUENTE`, `CLAVE_A`, `CLAVE_B` ni `NOMBRE_B`: son la llave. Si una base destino sin identificador (FDI) se vuelve a descargar y cambia el orden de filas, el motor **se detiene** en lugar de aplicar una decisión a otra empresa.

**Para medir al motor** (hoja `VALIDACION`): `ARCHIVO_REFERENCIA` apunta a decisiones revisadas; el motor automático se compara contra ellas sin aplicarlas.

In [ ]:
# Auditoría: calidad del motor, controles y calidad de los datos
display(res.validacion)
display(res.invariantes)
display(res.calidad[["FUENTE", "TEMA", "HALLAZGO", "IMPACTO"]])